# 🔥 LLM Benchmarking Pipeline — Task 1 + Task 2 (Full Agent)

## Task 1: Component Extraction Agent
LLM extracts hardware component names from project descriptions.

## Task 2: Library Retrieval Agent
LLM infers correct Arduino libraries from component names + project context.

### Pipeline Flow
```
Description + Libraries
        ↓
  [Task 1 Agent] LLM → predicted component names
        ↓
  [Task 2 Agent] LLM → inferred libraries per component
        ↓
  Compare vs GT libs → Precision / Recall / F1
```

---
# ⚙️ Setup

In [ ]:
# ── CELL 1: IMPORTS ───────────────────────────────────────────────────────────
import os
import re
import json
import logging
import difflib
import time as _time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Set, Optional, Tuple
from collections import defaultdict, Counter
from dotenv import load_dotenv
from tqdm.auto import tqdm
from openai import OpenAI

load_dotenv(dotenv_path=Path('../.env'))

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
logger = logging.getLogger(__name__)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

np.random.seed(42)
print(' Imports complete')

In [ ]:
# ── CELL 2: PATHS ─────────────────────────────────────────────────────────────
os.chdir('/Users/anonymous_user/Desktop/text-to-arduino')

DATA_DIR  = Path('data')
META_PATH = DATA_DIR / 'projects_metadata.xlsx'
COMP_PATH = DATA_DIR / 'components_dictionary.xlsx'
GT_PATH   = DATA_DIR / 'ground_truth_final.json'

print(f'Working dir : {os.getcwd()}')
print(f'META exists : {META_PATH.exists()}')
print(f'COMP exists : {COMP_PATH.exists()}')
print(f'GT exists   : {GT_PATH.exists()}')

In [ ]:
# ── CELL 3: LOAD DATA ─────────────────────────────────────────────────────────
df_meta = pd.read_excel(META_PATH)
df_meta.columns = df_meta.columns.str.strip()
df_meta = df_meta.rename(columns={
    'Project_ID':  'project_id',
    'Description': 'description',
    'Board':       'board',
    'Complexity':  'complexity',
})

df_comp = pd.read_excel(COMP_PATH)
df_comp.columns = df_comp.columns.str.strip()

meta_lookup = df_meta.set_index('project_id').to_dict('index')
comp_lookup = {
    int(row['Component_Id']): row.to_dict()
    for _, row in df_comp.iterrows()
}

FQBN_MAP = {
    'Arduino Uno':                'arduino:avr:uno',
    'Arduino Uno R4 Wifi Module': 'arduino:renesas_uno:unor4wifi',
    'Arduino Mega':               'arduino:avr:mega',
}

with open(GT_PATH) as f:
    projects_raw = json.load(f)

projects = []
for proj in projects_raw:
    pid   = proj.get('project_id')
    meta  = meta_lookup.get(pid, {})
    board = meta.get('board', 'Arduino Uno')
    projects.append({
        **proj,
        'description': meta.get('description', proj.get('description', '')),
        'board':       board,
        'board_fqbn':  FQBN_MAP.get(board, 'arduino:avr:uno'),
        'complexity':  meta.get('complexity', 'Unknown'),
    })

print(f' Projects loaded  : {len(projects)}')
print(f' Components loaded: {len(comp_lookup)}')

---
# 🤖 LLM Client

In [ ]:
# ── CELL 4: LLM CLIENT — PATCHED ──────────────────────────────────────────────
MODELS = {
    'llama3_3':   ('nvidia', 'meta/llama-3.3-70b-instruct',                  'NVIDIA_API_KEY_LLAMA33'),
    'mistral':    ('nvidia', 'mistralai/mistral-large-3-675b-instruct-2512', 'NVIDIA_API_KEY_MISTRAL'),
    'phi_medium': ('nvidia', 'microsoft/phi-3-medium-128k-instruct',         'NVIDIA_API_KEY_PHI3MEDIUM'),
    'gemma_27b':  ('nvidia', 'google/gemma-2-27b-it',                        'NVIDIA_API_KEY_GEMMA'),
    'phi_mini':   ('nvidia', 'microsoft/phi-3-mini-128k-instruct',           'NVIDIA_API_KEY_PHI3MINI'),
}

NVIDIA_BASE_URL  = 'https://integrate.api.nvidia.com/v1'
API_CALL_DELAY   = 1.0
API_TIMEOUT      = 45          
MAX_RETRIES      = 5
MAX_PROMPT_CHARS = 32000


class LLMClient:
    def __init__(self, model_key: str, temperature: float = 0):
        if model_key not in MODELS:
            raise ValueError(f'Unknown model_key: {model_key!r}. Valid: {list(MODELS.keys())}')
        self.model_key   = model_key
        self.provider, self.model_name, api_key_name = MODELS[model_key]
        self.temperature = temperature
        self.call_log    = []
        api_key = os.getenv(api_key_name)
        if not api_key:
            raise ValueError(f'Missing API key — set {api_key_name} in your .env file')
        self.client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=api_key)
        print(f'LLMClient ready: {model_key} ({self.model_name})')

    # ── KEY CHANGE: max_tokens is now a parameter, not hard-coded 512 ────────
    def generate(self, prompt: str, system_prompt: str,
                 max_tokens: int = 512) -> Tuple[str, float]:
        last_exc = None
        for attempt in range(MAX_RETRIES):
            try:
                _time.sleep(API_CALL_DELAY)
                t0 = _time.time()
                messages = (
                    [{'role': 'user', 'content': system_prompt + '\n\n' + prompt}]
                    if self.model_key == 'gemma_27b' else
                    [
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user',   'content': prompt},
                    ]
                )
                response = self.client.chat.completions.create(
                    model=self.model_name, messages=messages,
                    temperature=self.temperature,
                    max_tokens=max_tokens,      # ← was hard-coded 512
                    timeout=API_TIMEOUT,
                )
                latency = _time.time() - t0
                text    = response.choices[0].message.content.strip()
                self.call_log.append({
                    'latency': round(latency, 3), 'attempt': attempt + 1, 'success': True
                })
                return text, latency
            except Exception as exc:
                last_exc = exc
                if attempt < MAX_RETRIES - 1:
                    wait = (15.0 * (2 ** attempt) + random.uniform(1, 3)
                            if '429' in str(exc) else 0.5 * (attempt + 1))
                    print(f'  Attempt {attempt+1}/{MAX_RETRIES} failed: {exc} — retrying in {wait:.1f}s')
                    _time.sleep(wait)
        self.call_log.append({'latency': 0.0, 'success': False, 'error': str(last_exc)[:100]})
        print(f'LLM call FAILED after {MAX_RETRIES} retries: {last_exc}')
        return '', 0.0


def call_llm_json(prompt: str, model, system_prompt: str,
                  max_tokens: int = 512) -> Tuple[any, float]:
    """
    Generic LLM call → parsed JSON + latency.
    max_tokens is now exposed — Task 5 passes 1500 to avoid truncation.
    Includes robust JSON recovery for truncated outputs.
    """
    if model == 'stub':
        return {}, 0.0
    if not isinstance(model, LLMClient):
        raise TypeError('model must be "stub" or LLMClient instance')
    if len(prompt) > MAX_PROMPT_CHARS:
        prompt = prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = model.generate(prompt, system_prompt=system_prompt, max_tokens=max_tokens)
    if not raw:
        return {}, latency

    # Strip markdown fences
    text = raw.strip()
    text = re.sub(r'^```(?:json)?\n?', '', text)
    text = re.sub(r'\n?```$',          '', text).strip()
    text = re.sub(r':\s*(A[0-9])\b', r': "\1"', text)
    # ── 1) Try full parse ─────────────────────────────────────────────────────
    try:
        return json.loads(text), latency
    except Exception:
        pass

    # ── 2) Depth-tracking outermost JSON extractor ────────────────────────────
    # Fixes the old regex r'\{.*?\}' which grabbed the FIRST tiny {} instead
    # of the outermost object, returning a partial/empty dict every time.
    def extract_outermost(s: str) -> Optional[str]:
        for start, ch in enumerate(s):
            if ch not in ('{', '['):
                continue
            close = '}' if ch == '{' else ']'
            depth, in_str, esc = 0, False, False
            for i in range(start, len(s)):
                c = s[i]
                if esc:
                    esc = False; continue
                if c == '\\' and in_str:
                    esc = True;  continue
                if c == '"':
                    in_str = not in_str; continue
                if in_str:
                    continue
                if c in ('{', '['):
                    depth += 1
                elif c in ('}', ']'):
                    depth -= 1
                    if depth == 0 and c == close:
                        return s[start:i + 1]
        return None

    fragment = extract_outermost(text)
    if fragment:
        try:
            return json.loads(fragment), latency
        except Exception:
            pass

    # ── 3) Truncation repair — append missing closing brackets ───────────────
    # Handles the exact failure mode from max_tokens=512: JSON cut mid-array.
    open_b  = text.count('{') - text.count('}')
    open_sq = text.count('[') - text.count(']')
    if 0 < open_b + open_sq <= 6:        # sanity guard — don't patch a mess
        patched = text + (']' * max(0, open_sq)) + ('}' * max(0, open_b))
        try:
            return json.loads(patched), latency
        except Exception:
            pass

    logger.warning('Could not parse LLM response:\n%s', text[:400])
    return {}, latency


# ── API Key check ─────────────────────────────────────────────────────────────
print('\n── API Key Check ──')
for key, (_, _, env_var) in MODELS.items():
    status = 'Yes' if os.getenv(env_var) else ' MISSING'
    print(f'  {key:<12} {env_var:<35} {status}')
print('\nReady — run: active_client = LLMClient(\'llama3_3\')')

In [ ]:
# ── CELL 5: INIT CLIENT ───────────────────────────────────────────────────────
active_client = LLMClient('llama3_3')

---
# 📦 Task 1: Component Extraction Agent
> **Measures:** Precision, Recall, F1 on predicted vs ground-truth component names

**Method:** LLM reads description + libraries → outputs canonical component names with confidence.
Evaluated via fuzzy name matching with synonym expansion.

**Status:** Done

| Approach | Precision | Recall | F1 |
|---|---|---|---|
| BM25 + Embedding baseline | 0.247 | 0.571 | 0.300 |
| LLM Agent (llama3_3) | TBD | TBD | TBD |

In [ ]:
# ── CELL 6: TASK 1 — PROMPTS ──────────────────────────────────────────────────
SYSTEM_PROMPT_MODULE1 = """You are a hardware component extraction specialist for Arduino and embedded systems projects.

Your task is to extract ALL hardware components from a natural language project description.
A component is any physical electronic part: sensors, actuators, modules, displays, motors, LEDs, relays, or communication devices.
Do NOT include boards (Arduino, ESP32, etc.), wires, resistors, capacitors, or breadboards.

Rules:
- For high-level project descriptions (e.g. "smart dustbin", "surveillance robot"),
  infer the MINIMUM set of components needed to implement the core functionality
- Prefer specificity: if a project needs motor control, pick the specific driver 
  (L298N, L293D) not just 'DC Motor' as a standalone component
- Do not list both a component AND its driver — pick the most specific one
  (e.g. 'L298N Motor Driver' covers DC Motor implicitly)
- Limit output to components directly required for the stated goal
- Extract components explicitly named AND components implied by function
  (e.g. "measures temperature" → temperature sensor)
- Normalise synonyms to canonical names
  (e.g. "HC-SR04" → "Ultrasonic Sensor", "SG90" → "Servo Motor")
- If a library name is provided (e.g. "Servo.h", "DHT.h"), infer the associated component
- Output ONLY a JSON array. No explanation, no markdown, no preamble.

Confidence levels:
- "high"   → explicitly named or strong library signal
- "medium" → inferred from function description
- "low"    → inferred from domain context only

Output format:
[
  {
    "canonical_name": "Ultrasonic Sensor",
    "raw_mention": "HC-SR04",
    "confidence": "high"
  }
]

--- EXAMPLES ---

Example 1:
Description: "Smart dustbin that uses an ultrasonic sensor to detect nearby objects and opens the lid using a servo motor."
Libraries: Servo.h
Output:
[
  {"canonical_name": "Ultrasonic Sensor", "raw_mention": "ultrasonic sensor", "confidence": "high"},
  {"canonical_name": "Servo Motor",       "raw_mention": "servo motor",       "confidence": "high"}
]

Example 2:
Description: "Plant watering system that monitors soil moisture and controls a pump via Blynk, with LED matrix feedback."
Libraries: BlynkSimpleWifi, Arduino_LED_Matrix, EEPROM
Output:
[
  {"canonical_name": "Soil Moisture Sensor", "raw_mention": "soil moisture",   "confidence": "high"},
  {"canonical_name": "Relay Module",         "raw_mention": "controls a pump", "confidence": "medium"},
  {"canonical_name": "LED Matrix",           "raw_mention": "LED matrix",      "confidence": "high"}
]

Example 3:
Description: "Line follower robot using infrared sensors and DC motors controlled by an L293D motor driver."
Libraries: None
Output:
[
  {"canonical_name": "IR Sensor", "raw_mention": "infrared sensors", "confidence": "high"},
  {"canonical_name": "DC Motor",  "raw_mention": "DC motors",        "confidence": "high"}
]

--- END EXAMPLES ---"""

USER_PROMPT_MODULE1_TEMPLATE = """Extract all hardware components from the following Arduino project.

Project description:
{description}

Libraries used (if known):
{libraries}

Return ONLY the JSON array."""

print(' Task 1 prompts defined')

In [ ]:
# ── CELL 7: TASK 1 — HELPER FUNCTIONS ────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

INFRASTRUCTURE_NAMES = {
    'Breadboard', 'Jumper Wires', 'Power Supply',
    'Resistor', 'USB Cable', 'LED'
}

COMPONENT_SYNONYMS = {
    'ir sensor':          ['line finder', 'infrared sensor', 'line follower sensor', 'grove line finder'],
    'line finder':        ['ir sensor', 'infrared sensor'],
    'ultrasonic sensor':  ['distance sensor', 'hc-sr04', 'urm09', 'waterproof ultrasonic'],
    'soil moisture':      ['moisture sensor', 'grove moisture'],
    'relay':              ['relay module'],
    'dc motor':           ['motor with h-bridge', 'gear motor', 'dc motor with h-bridge'],
    'servo':              ['servo motor'],
    'bluetooth':          ['hc-05', 'hc-06', 'bluetooth module'],
    'oled':               ['oled display', 'ssd1306'],
    'water pump':         ['submersible pump', 'mini pump', 'submersible water pump'],
    'led matrix':         ['arduino led matrix', '8x8 matrix', 'led strip'],
    'temperature sensor': ['dht11', 'dht22', 'dht sensor'],
    'neopixel':           ['ws2812', 'ws2812b', 'led strip', 'rgb led strip'],
    'gps module':         ['neo-6m', 'gps sensor', 'gps tracker'],
    'rfid':               ['mfrc522', 'rfid reader', 'rc522'],
    'tft display':        ['tft lcd', 'ili9341', 'tft screen'],
    'dht22':              ['dht22 sensor', 'temperature humidity sensor', 'dht sensor', 'dht11'],  
    'dht sensor':         ['dht22', 'dht11', 'temperature sensor', 'humidity sensor'],            
}


# ── Ground truth helpers ──────────────────────────────────────────────────────
def get_gt_names(project: dict) -> set:
    """Convert ground truth component IDs → canonical names, strip infrastructure."""
    gt_ids   = project.get('ground_truth_component_ids', [])
    gt_names = set()
    for cid in gt_ids:
        entry = comp_lookup.get(int(cid))
        if entry:
            name = str(entry.get('Component name', '')).strip()
            if name and name not in INFRASTRUCTURE_NAMES:
                gt_names.add(name.lower())
    return gt_names


# ── Name matching ─────────────────────────────────────────────────────────────
def names_match(pred_name: str, gt_name: str, threshold: float = 0.60) -> bool:
    """Returns True if two component names refer to the same component."""
    p = pred_name.lower().strip()
    g = gt_name.lower().strip()
    if p == g:            return True
    if p in g or g in p:  return True
    for key, synonyms in COMPONENT_SYNONYMS.items():
        if key in p or key in g:
            if any(s in p or s in g for s in [key] + synonyms):
                return True
    p_tok = set(p.split())
    g_tok = set(g.split())
    if p_tok and g_tok:
        if len(p_tok & g_tok) / len(p_tok | g_tok) >= threshold:
            return True
    if difflib.SequenceMatcher(None, p, g).ratio() >= threshold:
        return True
    return False


def match_predicted_to_gt(predicted_names: list, gt_names: set) -> dict:
    """Score predicted names against GT. Returns tp, fp, fn, matched_gt."""
    matched_gt = set()
    tp = 0
    for pred in predicted_names:
        for gt in gt_names:
            if gt not in matched_gt and names_match(pred, gt):
                tp += 1
                matched_gt.add(gt)
                break
    fp = len(predicted_names) - tp
    fn = len(gt_names) - tp
    return {'tp': tp, 'fp': fp, 'fn': fn, 'matched_gt': matched_gt}


# ── RAG helper ────────────────────────────────────────────────────────────────
def get_similar_projects(description: str, projects: list,
                          comp_lookup: dict, top_k: int = 3) -> list:
    """Find top-k similar projects using TF-IDF similarity."""
    all_descs  = [str(p.get('description', '')) for p in projects]
    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_mat  = vectorizer.fit_transform(all_descs + [description])

    custom_vec = tfidf_mat[-1]
    proj_mat   = tfidf_mat[:-1]
    sims       = cos_sim(custom_vec, proj_mat)[0]

    top_indices = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_indices:
        proj     = projects[idx]
        gt_ids   = proj.get('ground_truth_component_ids', [])
        gt_names = [
            comp_lookup.get(int(c), {}).get('Component name', '?')
            for c in gt_ids
        ]
        results.append({
            'description': str(proj.get('description', ''))[:150],
            'components':  gt_names,
            'libraries':   proj.get('ground_truth_libraries', []),
            'similarity':  float(sims[idx]),
        })
    return results


# ── Task 1 Agent (with RAG) ───────────────────────────────────────────────────
def run_task1_agent(project: dict, client,
                    use_rag: bool = True) -> dict:
    """
    Task 1 Agent: LLM extracts component names from description + libraries.
    use_rag=True  → injects similar projects as few-shot context (recommended)
    use_rag=False → description + libraries only (baseline)
    """
    desc    = str(project.get('description', ''))
    libs    = project.get('ground_truth_libraries', [])
    lib_str = ', '.join(libs) if libs else 'None'
    board   = project.get('board', 'Arduino Uno')

    # ── Build RAG context ─────────────────────────────────────────────────────
    rag_context = ''
    rag_examples = []
    if use_rag:
        rag_examples = get_similar_projects(desc, projects, comp_lookup, top_k=3)
        for i, s in enumerate(rag_examples, 1):
            if s['similarity'] < 0.10:   # skip if too dissimilar
                continue
            rag_context += f"\nExample {i} (similarity: {s['similarity']:.2f}):\n"
            rag_context += f"  Description : {s['description']}\n"
            rag_context += f"  Components  : {s['components']}\n"
            rag_context += f"  Libraries   : {s['libraries']}\n"

    # ── Build prompt ──────────────────────────────────────────────────────────
    user_prompt = f"""Extract all hardware components from the following Arduino project.

Project description:
{desc}

Board: {board}

Libraries used (if known):
{lib_str}
"""

    if rag_context:
        user_prompt += f"""
Similar projects for reference — use these to infer components 
if the description is vague or incomplete:
{rag_context}
"""

    user_prompt += "\nReturn ONLY the JSON array."

    # ── LLM call ──────────────────────────────────────────────────────────────
    extractions, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_MODULE1,
    )

    predicted_names = []
    for item in (extractions if isinstance(extractions, list) else []):
        if not isinstance(item, dict): continue
        name = item.get('canonical_name', '').strip()
        conf = item.get('confidence', 'low')
        if name and conf in ('high', 'medium'):
            predicted_names.append(name)

    # Deduplicate preserving order
    seen = set()
    predicted_names = [x for x in predicted_names
                       if not (x.lower() in seen or seen.add(x.lower()))]

    return {
        'project_id':      project.get('project_id'),
        'predicted_names': predicted_names,
        'extractions':     extractions if isinstance(extractions, list) else [],
        'rag_examples':    rag_examples,
        'latency':         latency,
    }


print(' Task 1 helper functions defined')
print('   run_task1_agent(project, client)              → with RAG (default)')
print('   run_task1_agent(project, client, use_rag=False) → without RAG (baseline)')

---
# 📚 Task 2: Library Retrieval Agent
> **Measures:** Precision, Recall, F1 on retrieved vs ground-truth libraries

**Method:** For each component predicted by Task 1, LLM infers the correct Arduino library
using component name + project description as context. No dictionary ID lookup involved.

**Key advantage over rule-based:** Can infer libraries not in dictionary
(e.g. TFT_eSPI, AnimatedGIF, PNGdec, ArduinoBLE).

**Status:** Done

In [ ]:
# # ── CELL 8: TASK 2 — PROMPTS ──────────────────────────────────────────────────
# SYSTEM_PROMPT_MODULE2 = """You are a hardware library specialist for Arduino and embedded systems.

# Given a hardware component name and project description, identify the exact Arduino library needed.

# Rules:
# - Return ONLY the canonical library name as it appears in #include statements (without .h)
# - If multiple libraries exist for this component, return the most commonly used one
# - If no library is needed (component uses only digitalWrite/analogRead), return null
# - Do NOT return platform libraries: Wire, SPI, EEPROM, WiFi, Ethernet, SD, SoftwareSerial
# - Output ONLY a JSON object. No explanation, no markdown, no preamble.
# - Never return generic names like 'tft', 'lcd', 'sensor' — always use the full library name
# - For I2C sensors, return the sensor-specific library not Wire
# - For accelerometers: use MPU6050, Adafruit_MPU6050, or LSM6DS3 depending on chip

# Confidence levels:
# - "high"   → standard well-known library with no ambiguity
# - "medium" → library inferred from component type, minor uncertainty
# - "low"    → uncertain, multiple valid options exist

# Output format:
# {
#   "library": "Servo",
#   "confidence": "high",
#   "reasoning": "Standard Arduino Servo library for servo motors"
# }

# --- EXAMPLES ---

# Example 1:
# Component: "Servo Motor"
# Description: "Smart dustbin that opens lid using a servo motor"
# Output:
# {"library": "Servo", "confidence": "high", "reasoning": "Standard Arduino Servo library"}

# Example 2:
# Component: "DHT11 Temperature Sensor"
# Description: "Weather station monitoring temperature and humidity"
# Output:
# {"library": "DHT", "confidence": "high", "reasoning": "Adafruit DHT library for DHT11/DHT22"}

# Example 3:
# Component: "LED Matrix"
# Description: "Plant watering system with LED matrix feedback using Arduino UNO R4 WiFi"
# Output:
# {"library": "Arduino_LED_Matrix", "confidence": "high", "reasoning": "Built-in library for Arduino UNO R4 WiFi LED matrix"}

# Example 4:
# Component: "Ultrasonic Sensor"
# Description: "Distance measurement using HC-SR04"
# Output:
# {"library": null, "confidence": "high", "reasoning": "HC-SR04 uses pulseIn() only — no library needed"}

# Example 5:
# Component: "NeoPixel LED Strip"
# Description: "RGB LED strip controlled via ESP32"
# Output:
# {"library": "Adafruit_NeoPixel", "confidence": "high", "reasoning": "Adafruit NeoPixel library for WS2812 LED strips"}

# Example 6:
# Component: "TFT Display"
# Description: "ESP32-S3 digital board with round TFT display and animated graphics"
# Output:
# {"library": "TFT_eSPI", "confidence": "high", "reasoning": "TFT_eSPI is the standard library for TFT displays on ESP32"}

# Example 7:
# Component: "IR Sensor"
# Description: "Line follower robot using infrared sensors"
# Output:
# {"library": null, "confidence": "high", "reasoning": "IR line sensors use digitalRead() only — no library needed"}

# --- END EXAMPLES ---"""

# USER_PROMPT_MODULE2_TEMPLATE = """Identify the Arduino library for this component.

# Component: {component_name}
# Project description: {description}
# Board: {board}

# Important notes:
# - If board is Arduino UNO R4 WiFi, use Arduino_LED_Matrix for LED matrix
# - For TFT displays on ESP32, use TFT_eSPI
# - For I2C sensors, return the sensor library not Wire
# - Never return generic names like 'tft' or 'lcd' — use the full library name

# Return ONLY the JSON object."""

In [ ]:
# ── CELL 8: TASK 2 — BATCHED PROMPT ──────────────────────────────────────────
SYSTEM_PROMPT_MODULE2 = """You are a hardware library specialist for Arduino and embedded systems.

Given a list of hardware components and a project description, identify the exact 
Arduino library needed for EACH component.

Rules:
- Return the canonical library name as it appears in #include statements (without .h)
- If multiple libraries exist, return the most commonly used one
- If no library is needed (component uses only digitalWrite/analogRead), return null
- Do NOT return platform libraries: Wire, SPI, EEPROM, WiFi, Ethernet, SD, SoftwareSerial
- Output ONLY a JSON array. No explanation, no markdown, no preamble.

Confidence levels:
- "high"   → standard well-known library, no ambiguity
- "medium" → inferred from component type, minor uncertainty
- "low"    → uncertain, multiple valid options exist

Output format:
[
  {"component": "Servo Motor",       "library": "Servo",              "confidence": "high"},
  {"component": "Ultrasonic Sensor", "library": null,                 "confidence": "high"},
  {"component": "LED Matrix",        "library": "Arduino_LED_Matrix", "confidence": "high"}
]

--- EXAMPLES ---

Example 1:
Components: ["Servo Motor", "Ultrasonic Sensor"]
Board: Arduino Uno
Description: "Smart dustbin that opens lid using a servo motor and detects objects with ultrasonic sensor"
Output:
[
  {"component": "Servo Motor",       "library": "Servo", "confidence": "high"},
  {"component": "Ultrasonic Sensor", "library": null,    "confidence": "high"}
]

Example 2:
Components: ["Soil Moisture Sensor", "Relay Module", "LED Matrix", "Water Pump"]
Board: Arduino Uno R4 WiFi
Description: "Plant watering system with soil moisture monitoring and LED matrix feedback"
Output:
[
  {"component": "Soil Moisture Sensor", "library": null,                 "confidence": "high"},
  {"component": "Relay Module",         "library": null,                 "confidence": "high"},
  {"component": "LED Matrix",           "library": "Arduino_LED_Matrix", "confidence": "high"},
  {"component": "Water Pump",           "library": null,                 "confidence": "high"}
]

Example 3:
Components: ["DHT11 Sensor", "OLED Display", "NeoPixel LED Strip"]
Board: Arduino Uno
Description: "Weather station displaying temperature on OLED with LED indicators"
Output:
[
  {"component": "DHT11 Sensor",       "library": "DHT",              "confidence": "high"},
  {"component": "OLED Display",       "library": "Adafruit_SSD1306", "confidence": "high"},
  {"component": "NeoPixel LED Strip", "library": "Adafruit_NeoPixel","confidence": "high"}
]

Example 4:
Components: ["TFT Display", "Accelerometer", "Bluetooth Module"]
Board: ESP32
Description: "ESP32 motion tracker with TFT display and BLE communication"
Output:
[
  {"component": "TFT Display",    "library": "TFT_eSPI",       "confidence": "high"},
  {"component": "Accelerometer",  "library": "MPU6050",        "confidence": "high"},
  {"component": "Bluetooth Module","library": "ArduinoBLE",    "confidence": "high"}
]

Example 5:
Components: ["IR Sensor", "DC Motor"]
Board: Arduino Uno
Description: "Line follower robot using IR sensors and DC motors"
Output:
[
  {"component": "IR Sensor", "library": null, "confidence": "high"},
  {"component": "DC Motor",  "library": null, "confidence": "high"}
]

--- END EXAMPLES ---"""


USER_PROMPT_MODULE2_TEMPLATE = """Identify the Arduino library for each component below.

Components: {component_list}
Board: {board}
Project description: {description}

Important notes:
- If board is Arduino UNO R4 WiFi, use Arduino_LED_Matrix for LED matrix
- For TFT displays on ESP32, use TFT_eSPI
- For I2C sensors return the sensor-specific library, not Wire
- Never return generic names like 'tft' or 'lcd' — use the full library name
- For Bluetooth on nRF52 boards, use ArduinoBLE

Return ONLY the JSON array."""


print(' Task 2 batched prompts defined')

In [ ]:
# ── CELL 9: TASK 2 — HELPER FUNCTIONS (BATCHED) ──────────────────────────────

def normalize_lib(lib: str) -> str:
    if not lib: return ''
    lib = lib.lower().strip()
    lib = re.sub(r'#include\s*[<"]', '', lib)
    lib = re.sub(r'[>"]', '', lib)
    lib = lib.replace('.h', '')
    return lib.strip()

def parse_libraries(lib_str: str) -> set:
    if not lib_str or str(lib_str) == 'nan': return set()
    return {l.strip() for l in str(lib_str).split(',') if l.strip()}

def split_libs(lib: str) -> list:
    return [l.strip() for l in re.split(r'[/|,]', str(lib)) if l.strip()]

ALIASES = {
    'fastled library': 'fastled',       'neopixel': 'adafruit_neopixel',
    'adafruit_neopixel': 'adafruit_neopixel', 'adafruit_gfx': 'adafruit_gfx',
    'adafruit_sh1106': 'sh1106',        'ssd1306_minimal': 'ssd1306',
    'liquidcrystal_i2c': 'liquidcrystal', 'liquidcrystal': 'liquidcrystal',
    'adafruit_sensor': 'adafruit_unified_sensor',
    'dht_u': 'dht',                     'dht': 'dht',
    'adafruit_mpu6050': 'mpu6050',      'blynksimpleesp8266': 'blynk',
    'blynksimplewifi': 'blynk',         'blynksimpleesp32': 'blynk',
    'wificlientsecure': 'wifi_generic', 'wifi': 'wifi_generic',
    'esp8266wifi': 'wifi_generic',      'esp32wifi': 'wifi_generic',
    'wifis3': 'wifi_generic',           'tinygps++': 'tinygps',
    'tinygpsplus': 'tinygps',           'nrf24l01': 'rf24',
    'mfrc522': 'rc522',                 'firebaseesp8266': 'firebase',
    'firebaseesp32': 'firebase',        'adafruit_bmp280': 'bmp280',
    'adafruit_bme280': 'bme280',        'arduino_led_matrix': 'arduino_led_matrix',
}

PLATFORM_LIBS = {
    'spi', 'wire', 'eeprom', 'softwareserial', 'wifi_generic',
    'wifis3', 'esp8266wifi', 'esp32wifi', 'ethernet', 'sd',
    'arduinojson', 'httpclient', 'time', 'rtclib', 'freertos',
    'arduino', 'wifi', 'wificlientsecure', 'blynk', 'firebase',
    'mqtt', 'pubsubclient', 'esp_sleep', 'pngdec', 'animatedgif',
    'sipeed_st7789',
}

BAD_KEYWORDS = ['python', 'sdk', 'matlab', 'ros', 'dependency', 'pip', 'npm']

def is_valid_lib(lib: str) -> bool:
    return not any(b in lib for b in BAD_KEYWORDS)

def canonicalize_set(lib_set: set) -> set:
    final = set()
    for lib in lib_set:
        for p in split_libs(lib):
            norm = normalize_lib(p)
            if not norm or not is_valid_lib(norm): continue
            norm = ALIASES.get(norm, norm)
            final.add(norm)
    return final

LIB_EQUIVALENTS = {
    'dht':                {'dht', 'dht11', 'dht22', 'dht_u', 'dht sensor'},
    'tinygps':            {'tinygps', 'tinygps++', 'gps'},
    'rf24':               {'rf24', 'nrf24l01'},
    'rc522':              {'rc522', 'mfrc522'},
    'ssd1306':            {'ssd1306', 'oled', 'adafruit_ssd1306'},
    'liquidcrystal':      {'liquidcrystal', 'liquidcrystal_i2c'},
    'wifi_generic':       {'wifi_generic', 'wifi', 'esp8266wifi', 'esp32wifi'},
    'adafruit_neopixel':  {'adafruit_neopixel', 'neopixel', 'fastled'},
    'blynk':              {'blynk', 'blynksimplewifi', 'blynksimpleesp8266'},
    'mpu6050':            {'mpu6050', 'adafruit_mpu6050', 'wire'},
    'bmp280':             {'bmp280', 'adafruit_bmp280'},
    'irremote':           {'irremote', 'irrecv', 'ir'},
    'arduino_led_matrix': {'arduino_led_matrix', 'adafruit_neomatrix',
                           'ledcontrol', 'md_max72xx', 'fastled'},
    'tft_espi':           {'tft_espi', 'tft', 'adafruit_tft', 'adafruit_ili9341',
                           'adafruit_st7735', 'spfd5408_adafruit_tftlcd'},
    'servo':              {'servo'},
    'dfrobot_urm09':      {'dfrobot_urm09', 'newping'},
    'adafruit_ina219':    {'adafruit_ina219', 'ina219'},
    'arduinoble':         {'arduinoble', 'adafruit_ble', 'ble', 'adafruit_nrf52_arduino'},
    'adafruit_gfx':       {'adafruit_gfx', 'gfx'},
}

def match_with_equivalence(gt_set: set, pred_set: set) -> int:
    matched, used_gt, used_pred = 0, set(), set()
    for g in gt_set:
        for p in pred_set:
            if p in used_pred or g in used_gt: continue
            if g == p:
                matched += 1; used_gt.add(g); used_pred.add(p); break
            for group in LIB_EQUIVALENTS.values():
                if g in group and p in group:
                    matched += 1; used_gt.add(g); used_pred.add(p); break
            if g in used_gt: break
    return matched


# ── Batched Task 2 Agent ──────────────────────────────────────────────────────
def run_task2_agent(component_names: list, description: str,
                    client, board: str = 'Arduino Uno') -> list:
    """
    Task 2 Agent (batched): ONE LLM call for ALL components in a project.
    Returns list of {component, library, confidence, status} dicts.
    """
    if not component_names:
        return []

    user_prompt = USER_PROMPT_MODULE2_TEMPLATE.format(
        component_list = json.dumps(component_names),
        description    = description,
        board          = board,
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_MODULE2,
    )

    # result should be a list
    if not isinstance(result, list):
        # Return error entry for each component
        return [
            {'component': n, 'library': None, 'confidence': 'low',
             'reasoning': 'parse_failed', 'latency': latency, 'status': 'parse_failed'}
            for n in component_names
        ]

    # Process each returned item
    processed = []
    result_map = {
        str(item.get('component', '')).lower(): item
        for item in result if isinstance(item, dict)
    }

    for name in component_names:
        item = result_map.get(name.lower(), {})

        library    = item.get('library')
        confidence = item.get('confidence', 'low')
        reasoning  = item.get('reasoning', '')

        # No library needed
        if library is None:
            processed.append({
                'component':  name,
                'library':    None,
                'confidence': confidence,
                'reasoning':  reasoning,
                'latency':    latency,
                'status':     'no_library_needed',
            })
            continue

        # Normalize + alias
        lib_norm = normalize_lib(str(library))
        lib_norm = ALIASES.get(lib_norm, lib_norm)

        # Skip platform libs
        if lib_norm in PLATFORM_LIBS:
            processed.append({
                'component':  name,
                'library':    None,
                'confidence': confidence,
                'reasoning':  reasoning,
                'latency':    latency,
                'status':     'platform_only',
            })
            continue

        processed.append({
            'component':  name,
            'library':    lib_norm,
            'confidence': confidence,
            'reasoning':  reasoning,
            'latency':    latency,
            'status':     'ok',
        })

    return processed


print('Task 2 batched agent defined')

---
# 🔗 Connected Pipeline — Task 1 + Task 2 (Full Agent)

In [ ]:
# ── CELL 10: SANITY CHECK — 3 PROJECTS ───────────────────────────────────────
print('── Sanity check (3 projects) ──\n')

for proj in projects[:3]:
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')

    # Task 1
    t1       = run_task1_agent(proj, active_client)
    gt_names = get_gt_names(proj)

    # Task 2 — ONE batched call for all components
    gt_libs    = set(proj.get('ground_truth_libraries', []))
    t2_results = run_task2_agent(
        t1['predicted_names'], desc, active_client, board=board
    )

    ret_libs  = {r['library'] for r in t2_results if r['library']}
    gt_canon  = canonicalize_set(gt_libs)
    ret_canon = canonicalize_set(ret_libs)
    gt_canon  = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    print(f'Project {pid} [{proj.get("complexity")}] | Board: {board}')
    print(f'  T1 Predicted : {t1["predicted_names"]}')
    print(f'  T1 GT names  : {sorted(gt_names)}')
    print(f'  T2 Results   :')
    for r in t2_results:
        print(f'    {r["component"]:<35} → {str(r["library"]):<30} ({r["status"]})')
    print(f'  T2 GT libs   : {sorted(gt_canon)}')
    print(f'  T2 Retrieved : {sorted(ret_canon)}')
    print(f'  API calls    : 2 total (1 T1 + 1 T2)')
    print()

In [ ]:
# ── CELL 11: FULL CONNECTED RUN — ALL 434 PROJECTS ───────────────────────────
SAVE_PATH         = 'connected_results_full_agent.json'
connected_results = []
error_count       = 0

for proj in tqdm(projects[:10], desc='Task 1+2 Full Agent'):
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')
    gt_libs = set(proj.get('ground_truth_libraries', []))

    try:
        t1_output = run_task1_agent(proj, active_client)
    except Exception as e:
        error_count += 1
        tqdm.write(f'  ERROR T1 project {pid}: {e}')
        continue

    gt_names        = get_gt_names(proj)
    predicted_names = t1_output['predicted_names']

    if not gt_names:
        continue

    t1_scores    = match_predicted_to_gt(predicted_names, gt_names)
    t1_tp        = t1_scores['tp']
    t1_fp        = t1_scores['fp']
    t1_fn        = t1_scores['fn']
    t1_precision = t1_tp / (t1_tp + t1_fp) if (t1_tp + t1_fp) > 0 else 0
    t1_recall    = t1_tp / (t1_tp + t1_fn) if (t1_tp + t1_fn) > 0 else 0
    t1_f1        = (2 * t1_precision * t1_recall / (t1_precision + t1_recall)
                    if (t1_precision + t1_recall) > 0 else 0)

    try:
        t2_details = run_task2_agent(
            predicted_names, desc, active_client, board=board
        )
    except Exception as e:
        tqdm.write(f'  ERROR T2 project {pid}: {e}')
        t2_details = [
            {'component': n, 'library': None, 'status': 'error',
             'confidence': 'low', 'reasoning': str(e), 'latency': 0.0}
            for n in predicted_names
        ]

    retrieved_libs = {r['library'] for r in t2_details if r['library']}

    gt_canon  = canonicalize_set(gt_libs)
    ret_canon = canonicalize_set(retrieved_libs)
    gt_canon  = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if gt_canon:
        t2_tp        = match_with_equivalence(gt_canon, ret_canon)
        t2_precision = t2_tp / len(ret_canon) if ret_canon else 0
        t2_recall    = t2_tp / len(gt_canon)  if gt_canon  else 0
        t2_f1        = (2 * t2_precision * t2_recall / (t2_precision + t2_recall)
                        if (t2_precision + t2_recall) > 0 else 0)
    else:
        t2_tp = t2_precision = t2_recall = t2_f1 = None

    connected_results.append({
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,
        'task1': {
            'predicted_names': predicted_names,
            'gt_names':        list(gt_names),
            'matched':         list(t1_scores['matched_gt']),
            'precision':       round(t1_precision, 4),
            'recall':          round(t1_recall,    4),
            'f1':              round(t1_f1,        4),
            'latency':         t1_output['latency'],
        },
        'task2': {
            'gt_libs':        list(gt_canon)  if gt_canon  else [],
            'retrieved_libs': list(ret_canon) if ret_canon else [],
            'tp':             t2_tp,
            'precision':      round(t2_precision, 4) if t2_precision is not None else None,
            'recall':         round(t2_recall,    4) if t2_recall    is not None else None,
            'f1':             round(t2_f1,        4) if t2_f1        is not None else None,
            'details':        t2_details,
        },
    })

    if len(connected_results) % 10 == 0:
        with open(SAVE_PATH, 'w') as f:
            json.dump(connected_results, f, indent=2)
        tqdm.write(f'  💾 Saved {len(connected_results)} results...')

with open(SAVE_PATH, 'w') as f:
    json.dump(connected_results, f, indent=2)

print(f' Done — saved to {SAVE_PATH}')
print(f'   Evaluated : {len(connected_results)} | Errors: {error_count}')

In [ ]:
# ── CELL 12: PRINT RESULTS ────────────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]

print('=' * 55)
print(' Full Agent Pipeline — Task 1 + Task 2')
print('=' * 55)
print(f'  Model              : {active_client.model_key}')
print(f'  Projects evaluated : {len(connected_results)}')
print(f'  Errors             : {error_count}')
print()
print(f'  Task 1 — Component Extraction')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t1_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t1_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t1_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]   for r in t1_all]):.2f}s')
print()
print(f'  Task 2 — Library Retrieval')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t2_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t2_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t2_all]):.3f}')
print(f'    Scored           : {len(t2_all)} / {len(connected_results)} projects')
print('=' * 55)

print('\n── Per Project Sample (first 5) ──')
for r in connected_results[:5]:
    print(f"\n  Project {r['project_id']} [{r['complexity']}]")
    print(f"  T1 Predicted  : {r['task1']['predicted_names']}")
    print(f"  T1 GT         : {r['task1']['gt_names']}")
    print(f"  T1 F1         : {r['task1']['f1']:.2f}")
    print(f"  T2 GT libs    : {r['task2']['gt_libs']}")
    print(f"  T2 Retrieved  : {r['task2']['retrieved_libs']}")
    print(f"  T2 F1         : {r['task2']['f1']}")
    print(f"  T2 Reasoning  :")
    for d in r['task2']['details']:
        print(f"    {d['component']:<35} → {str(d['library']):<25} ({d['status']})")

In [ ]:
# ── CELL 13: FAILURE ANALYSIS ─────────────────────────────────────────────────
t1_low  = [r for r in connected_results if r['task1']['f1'] < 0.2]
t1_zero = [r for r in connected_results if r['task1']['f1'] == 0.0]
t1_perf = [r for r in connected_results if r['task1']['f1'] == 1.0]
t2_low  = [r for r in connected_results
           if r['task2']['f1'] is not None and r['task2']['f1'] < 0.2]
t2_zero = [r for r in connected_results
           if r['task2']['f1'] is not None and r['task2']['f1'] == 0.0]
t2_perf = [r for r in connected_results
           if r['task2']['f1'] is not None and r['task2']['f1'] == 1.0]
t2_scored = [r for r in connected_results if r['task2']['f1'] is not None]

print('=' * 60)
print(' Failure Analysis')
print('=' * 60)
print(f'  Total projects evaluated   : {len(connected_results)}')
print()
print(f'  Task 1 — Perfect F1 (1.00) : {len(t1_perf)} / {len(connected_results)}')
print(f'  Task 1 — Low F1   (< 0.20) : {len(t1_low)} / {len(connected_results)}')
print(f'  Task 1 — Zero F1  (0.00)   : {len(t1_zero)} / {len(connected_results)}')
print()
print(f'  Task 2 — Scored            : {len(t2_scored)} / {len(connected_results)}')
print(f'  Task 2 — Perfect F1 (1.00) : {len(t2_perf)} / {len(t2_scored)}')
print(f'  Task 2 — Low F1   (< 0.20) : {len(t2_low)} / {len(t2_scored)}')
print(f'  Task 2 — Zero F1  (0.00)   : {len(t2_zero)} / {len(t2_scored)}')
print('=' * 60)

# ── Precision distribution ────────────────────────────────────────────────────
print('\n── Task 1 F1 Distribution ──')
buckets_t1 = {'1.00': 0, '0.75-0.99': 0, '0.50-0.74': 0,
              '0.25-0.49': 0, '0.00-0.24': 0}
for r in connected_results:
    f = r['task1']['f1']
    if f == 1.0:        buckets_t1['1.00']      += 1
    elif f >= 0.75:     buckets_t1['0.75-0.99'] += 1
    elif f >= 0.50:     buckets_t1['0.50-0.74'] += 1
    elif f >= 0.25:     buckets_t1['0.25-0.49'] += 1
    else:               buckets_t1['0.00-0.24'] += 1
for bucket, count in buckets_t1.items():
    bar = '█' * (count // max(1, len(connected_results) // 20))
    print(f'  {bucket} : {count:>4}  {bar}')

print('\n── Task 2 F1 Distribution ──')
buckets_t2 = {'1.00': 0, '0.75-0.99': 0, '0.50-0.74': 0,
              '0.25-0.49': 0, '0.00-0.24': 0}
for r in t2_scored:
    f = r['task2']['f1']
    if f == 1.0:        buckets_t2['1.00']      += 1
    elif f >= 0.75:     buckets_t2['0.75-0.99'] += 1
    elif f >= 0.50:     buckets_t2['0.50-0.74'] += 1
    elif f >= 0.25:     buckets_t2['0.25-0.49'] += 1
    else:               buckets_t2['0.00-0.24'] += 1
for bucket, count in buckets_t2.items():
    bar = '█' * (count // max(1, len(t2_scored) // 20))
    print(f'  {bucket} : {count:>4}  {bar}')

# ── Most missed components T1 ─────────────────────────────────────────────────
missed_comps = Counter()
extra_comps  = Counter()
for r in connected_results:
    matched  = set(r['task1']['matched'])
    gt_set   = set(r['task1']['gt_names'])
    pred_set = set(r['task1']['predicted_names'])
    for c in gt_set   - matched: missed_comps[c] += 1
    for c in pred_set - matched: extra_comps[c]  += 1

print('\n── Top 10 Most Missed Components (T1 — FN) ──')
for name, count in missed_comps.most_common(10):
    print(f'  {name:<45} missed {count}x')

print('\n── Top 10 Most Hallucinated Components (T1 — FP) ──')
for name, count in extra_comps.most_common(10):
    print(f'  {name:<45} extra  {count}x')

# ── Most missed libraries T2 ──────────────────────────────────────────────────
missed_libs = Counter()
extra_libs  = Counter()
for r in connected_results:
    if r['task2']['f1'] is None: continue
    gt_set  = set(r['task2']['gt_libs'])
    ret_set = set(r['task2']['retrieved_libs'])
    for lib in gt_set  - ret_set: missed_libs[lib] += 1
    for lib in ret_set - gt_set:  extra_libs[lib]  += 1

print('\n── Top 10 Most Missed Libraries (T2 — FN) ──')
for lib, count in missed_libs.most_common(10):
    print(f'  {lib:<45} missed {count}x')

print('\n── Top 10 Over-Retrieved Libraries (T2 — FP) ──')
for lib, count in extra_libs.most_common(10):
    print(f'  {lib:<45} extra  {count}x')

# ── T2 status breakdown ───────────────────────────────────────────────────────
status_counts = Counter()
for r in connected_results:
    for d in r['task2'].get('details', []):
        status_counts[d.get('status', 'unknown')] += 1

print('\n── Task 2 Component Status Breakdown ──')
for status, count in status_counts.most_common():
    print(f'  {status:<25} : {count}')

# ── Sample failure cases ──────────────────────────────────────────────────────
if t1_zero:
    print(f'\n── Sample Zero-F1 T1 Projects (first 3) ──')
    for r in t1_zero[:3]:
        print(f"\n  Project {r['project_id']} [{r['complexity']}]")
        print(f"  GT        : {r['task1']['gt_names']}")
        print(f"  Predicted : {r['task1']['predicted_names']}")

if t2_zero:
    print(f'\n── Sample Zero-F1 T2 Projects (first 3) ──')
    for r in t2_zero[:3]:
        print(f"\n  Project {r['project_id']} [{r['complexity']}]")
        print(f"  GT libs   : {r['task2']['gt_libs']}")
        print(f"  Retrieved : {r['task2']['retrieved_libs']}")
        print(f"  Details   :")
        for d in r['task2']['details']:
            print(f"    {d['component']:<35} → {str(d['library']):<25} ({d['status']})")

In [ ]:
# ── CELL 14: FINAL SAVE + SUMMARY ────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]

summary = {
    'model':           active_client.model_key,
    'total_evaluated': len(connected_results),
    'errors':          error_count,
    'task1': {
        'avg_precision': round(np.mean([r['precision'] for r in t1_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t1_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t1_all]), 4),
        'avg_latency':   round(np.mean([r['latency']   for r in t1_all]), 3),
        'perfect_f1':    len([r for r in t1_all if r['f1'] == 1.0]),
        'zero_f1':       len([r for r in t1_all if r['f1'] == 0.0]),
    },
    'task2': {
        'avg_precision': round(np.mean([r['precision'] for r in t2_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t2_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t2_all]), 4),
        'scored':        len(t2_all),
        'perfect_f1':    len([r for r in t2_all if r['f1'] == 1.0]),
        'zero_f1':       len([r for r in t2_all if r['f1'] == 0.0]),
    },
    'results': connected_results,
}

with open('connected_results_full_agent_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(' Saved → connected_results_full_agent_summary.json')
print()
print(f"  Task 1 — Avg F1 : {summary['task1']['avg_f1']}")
print(f"  Task 2 — Avg F1 : {summary['task2']['avg_f1']}")

In [ ]:
# ── TEST ON CUSTOM DESCRIPTION ────────────────────────────────────────────────

custom_project = {
    "project_id":               999,
    "description":              "I want to build a smart dustbin system",  
    "ground_truth_libraries":   [],
    "ground_truth_component_ids": [],
    "board":                    "Arduino Uno",
    "complexity":               "Low",
}

desc  = custom_project['description']
board = custom_project['board']

print(f'Description : {desc}')
print(f'Board       : {board}')
print()

# ── Task 1 WITH RAG ───────────────────────────────────────────────────────────
print('── Task 1: Component Extraction (with RAG) ──\n')
t1 = run_task1_agent(custom_project, active_client, use_rag=True)

print('RAG examples injected:')
for ex in t1['rag_examples']:
    if ex['similarity'] >= 0.10:
        print(f"  sim:{ex['similarity']:.2f} | {ex['description'][:70]}")
        print(f"           → {ex['components']}")
print()

print('Predicted components:')
if t1['predicted_names']:
    for name in t1['predicted_names']:
        print(f'  ✅ {name}')
else:
    print('  ⚠️  No components predicted')
print(f'Latency: {t1["latency"]:.2f}s')
print()

# ── Task 2 ────────────────────────────────────────────────────────────────────
print('── Task 2: Library Retrieval ──\n')
if t1['predicted_names']:
    t2 = run_task2_agent(
        t1['predicted_names'], desc, active_client, board=board
    )
    print(f'  {"Component":<33} {"Library":<30} Status')
    print('  ' + '-' * 73)
    for r in t2:
        icon = '✅' if r['library'] else '⚪'
        print(f'  {icon} {r["component"]:<33} {str(r["library"]):<30} {r["status"]}')

    print()
    print('  #include statements needed:')
    libs = [r['library'] for r in t2 if r['library']]
    if libs:
        for lib in libs:
            print(f'    #include <{lib}.h>')
    else:
        print('    None — all components use native Arduino functions')
else:
    print('  ⚠️  Skipped — no components from Task 1')

print()

# ── Dictionary entries ────────────────────────────────────────────────────────
print('── Dictionary Entries Retrieved ──\n')
for name in t1['predicted_names']:
    name_lower  = name.lower().strip()
    name_tokens = set(name_lower.split())
    best_score  = 0.0
    best_row    = None

    for _, row in df_comp.iterrows():
        comp_name   = str(row['Component name']).lower().strip()
        comp_tokens = set(comp_name.split())
        if name_lower == comp_name:
            best_row = row; break
        if name_lower in comp_name or comp_name in name_lower:
            score = 0.95
        else:
            overlap = (len(name_tokens & comp_tokens) / len(name_tokens | comp_tokens)
                       if name_tokens and comp_tokens else 0.0)
            fuzzy   = difflib.SequenceMatcher(None, name_lower, comp_name).ratio()
            score   = max(overlap, fuzzy)
        if score > best_score:
            best_score = score
            best_row   = row

    if best_row is not None and best_score >= 0.40:
        print(f'  Component   : {name}')
        print(f'  Dict match  : {best_row["Component name"]} (score: {best_score:.2f})')
        print(f'  Category    : {best_row["Category"]} > {best_row["Sub-category"]}')
        print(f'  Libraries   : {best_row["Associated Libraries"]}')
        print(f'  Pin Type    : {best_row["Pin Type"]}')
        print(f'  Use Cases   : {str(best_row["Typical Use Cases"])[:80]}...')
        print()
    else:
        print(f'  Component   : {name}')
        print(f'  Dict match  : ❌ No match in dictionary')
        print()

# ── Similar projects ──────────────────────────────────────────────────────────
print('── Similar Projects in Dataset ──\n')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

all_descs  = [str(p.get('description', '')) for p in projects]
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_mat  = vectorizer.fit_transform(all_descs + [desc])
sims       = cos_sim(tfidf_mat[-1], tfidf_mat[:-1])[0]
top_indices = np.argsort(sims)[::-1][:5]

for rank, idx in enumerate(top_indices, 1):
    proj      = projects[idx]
    gt_ids    = proj.get('ground_truth_component_ids', [])
    gt_names  = [comp_lookup.get(int(c), {}).get('Component name', '?')
                 for c in gt_ids]
    print(f'  #{rank} Project {proj["project_id"]} [{proj["complexity"]}]'
          f'  similarity: {sims[idx]:.3f}')
    print(f'     Description : {str(proj.get("description",""))[:90]}')
    print(f'     Components  : {gt_names}')
    print(f'     Libraries   : {proj.get("ground_truth_libraries", [])}')
    print()

print('=' * 65)

# 🧠 Task 3: Intent Understanding
> **Measures:** Completeness and Accuracy of extracted project intent

**Method:** Module 3A agent reads description + predicted components →
outputs structured intent JSON (tasks, conditions, thresholds, inputs, outputs).
Scored by LLM-as-judge against original description.

**Status:** ✅ Complete (10-project sample)

| Metric | Score |
|---|---|
| Avg Completeness | 0.915 |
| Avg Accuracy | 0.959 |
| Avg F1 | 0.937 |
| Projects scored | 10 |

| Complexity | F1 | Completeness | Accuracy |
|---|---|---|---|
| Low | 0.938 | 0.917 | 0.960 |
| Medium | 0.932 | 0.910 | 0.956 |
| High | 0.945 | 0.925 | 0.965 |
```



In [ ]:
# ── CELL 2: TASK 3 — PROMPTS ──────────────────────────────────────────────────
SYSTEM_PROMPT_MODULE3A = """You are an Arduino Logic Architect specializing in intent extraction.

Given a project description and list of hardware components, extract the complete 
project intent as a structured JSON object.

Rules:
- Extract ALL tasks the system must perform
- Identify ALL conditions that trigger actions (if/when statements)
- Extract ALL thresholds (numeric values, time limits, distances)
- List ALL inputs (sensors, buttons, signals)
- List ALL outputs (actuators, displays, LEDs, motors)
- Be specific — use exact values from description when available
- Output ONLY a JSON object. No explanation, no markdown, no preamble.

Output format:
{
  "tasks": [
    "description of each task the system performs"
  ],
  "conditions": [
    "condition that triggers an action"
  ],
  "thresholds": [
    "numeric value or limit mentioned"
  ],
  "inputs": [
    "input device or signal"
  ],
  "outputs": [
    "output device or signal"
  ],
  "logic_summary": "one sentence describing what the system does"
}

--- EXAMPLES ---

Example 1:
Description: "Smart dustbin that opens lid when object detected within 20cm, closes after 3 seconds"
Components: ["Ultrasonic Sensor", "Servo Motor"]
Output:
{
  "tasks":         ["detect nearby object", "open lid", "close lid after delay"],
  "conditions":    ["distance < 20cm triggers lid open", "3 seconds elapsed triggers lid close"],
  "thresholds":    ["20cm detection distance", "3 second delay"],
  "inputs":        ["ultrasonic echo pin"],
  "outputs":       ["servo motor signal"],
  "logic_summary": "Opens dustbin lid when object is detected within 20cm and closes it after 3 seconds"
}

Example 2:
Description: "Plant watering system that monitors soil moisture and activates pump when moisture drops below 30%"
Components: ["Soil Moisture Sensor", "Relay Module", "Water Pump"]
Output:
{
  "tasks":         ["monitor soil moisture", "activate water pump", "deactivate pump when sufficient"],
  "conditions":    ["moisture < 30% triggers pump on", "moisture >= 30% triggers pump off"],
  "thresholds":    ["30% moisture threshold"],
  "inputs":        ["soil moisture sensor analog pin"],
  "outputs":       ["relay module", "water pump"],
  "logic_summary": "Monitors soil moisture and automatically waters plant when moisture drops below 30%"
}

Example 3:
Description: "Temperature monitoring system that displays readings on OLED and triggers buzzer alarm when temperature exceeds 40 degrees"
Components: ["DHT22 Sensor", "OLED Display", "Buzzer"]
Output:
{
  "tasks":         ["read temperature", "display reading on OLED", "trigger alarm"],
  "conditions":    ["temperature > 40C triggers buzzer"],
  "thresholds":    ["40 degrees Celsius alarm threshold"],
  "inputs":        ["DHT22 temperature pin"],
  "outputs":       ["OLED display", "buzzer"],
  "logic_summary": "Monitors temperature, displays readings on OLED, and sounds buzzer alarm above 40 degrees"
}

--- END EXAMPLES ---"""


USER_PROMPT_MODULE3A_TEMPLATE = """Extract the complete project intent from the following Arduino project.

Project description:
{description}

Hardware components identified:
{components}

Board: {board}

Return ONLY the JSON object."""


print(' Task 3 prompts defined')

In [ ]:
# ── CELL 3: TASK 3 — AGENT FUNCTION ──────────────────────────────────────────

def run_task3_agent(project: dict, predicted_names: list, client) -> dict:
    """
    Task 3 Agent: LLM extracts structured intent from description + components.
    Input  → description + predicted component names from Task 1
    Output → tasks, conditions, thresholds, inputs, outputs, logic_summary
    """
    desc       = str(project.get('description', ''))
    board      = project.get('board', 'Arduino Uno')
    comp_str   = json.dumps(predicted_names)

    user_prompt = USER_PROMPT_MODULE3A_TEMPLATE.format(
        description = desc,
        components  = comp_str,
        board       = board,
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_MODULE3A,
    )

    # ── Validate output structure ─────────────────────────────────────────────
    REQUIRED_KEYS = ['tasks', 'conditions', 'thresholds',
                     'inputs', 'outputs', 'logic_summary']

    if not isinstance(result, dict):
        return {
            'project_id':    project.get('project_id'),
            'status':        'parse_failed',
            'latency':       latency,
            'tasks':         [],
            'conditions':    [],
            'thresholds':    [],
            'inputs':        [],
            'outputs':       [],
            'logic_summary': '',
            'raw':           result,
        }

    # Fill missing keys with empty defaults
    for key in REQUIRED_KEYS:
        if key not in result:
            result[key] = [] if key != 'logic_summary' else ''

    # Ensure all list fields are actually lists
    for key in ['tasks', 'conditions', 'thresholds', 'inputs', 'outputs']:
        if not isinstance(result[key], list):
            result[key] = [str(result[key])] if result[key] else []

    return {
        'project_id':    project.get('project_id'),
        'status':        'ok',
        'latency':       latency,
        'tasks':         result['tasks'],
        'conditions':    result['conditions'],
        'thresholds':    result['thresholds'],
        'inputs':        result['inputs'],
        'outputs':       result['outputs'],
        'logic_summary': result['logic_summary'],
        'raw':           result,
    }


print(' run_task3_agent defined')

In [ ]:
# ── CELL 4: TASK 3 — SCORING ──────────────────────────────────────────────────

SYSTEM_PROMPT_JUDGE = """You are an expert Arduino project evaluator.

You will be given:
1. Original project description
2. Predicted intent (tasks, conditions, thresholds, inputs, outputs)

Score the predicted intent on two dimensions:

COMPLETENESS (0.0 to 1.0):
- Did it capture ALL tasks the system performs?
- Did it identify ALL conditions/triggers?
- Did it find ALL thresholds/numeric values?
- 1.0 = nothing missing, 0.0 = completely missed the point

ACCURACY (0.0 to 1.0):
- Are the extracted tasks correct and specific?
- Are conditions stated correctly (right direction, right value)?
- Are inputs/outputs correctly identified?
- 1.0 = everything correct, 0.0 = everything wrong

Rules:
- Be strict but fair
- Partial credit is allowed
- Minor wording differences are fine as long as meaning is correct
- Output ONLY a JSON object. No explanation, no markdown.

Output format:
{
  "completeness": 0.85,
  "accuracy":     0.90,
  "reasoning":    "Brief explanation of scores"
}"""


USER_PROMPT_JUDGE_TEMPLATE = """Evaluate this intent extraction.

Original description:
{description}

Predicted intent:
Tasks      : {tasks}
Conditions : {conditions}
Thresholds : {thresholds}
Inputs     : {inputs}
Outputs    : {outputs}
Summary    : {logic_summary}

Return ONLY the JSON object with completeness, accuracy, and reasoning."""


def score_intent_with_judge(project: dict, t3_output: dict, client) -> dict:
    """
    LLM-as-judge scores Task 3 output on completeness + accuracy.
    Returns completeness (0-1), accuracy (0-1), f1 (harmonic mean).
    """
    desc = str(project.get('description', ''))

    # Skip if Task 3 failed
    if t3_output['status'] != 'ok':
        return {
            'completeness': 0.0,
            'accuracy':     0.0,
            'f1':           0.0,
            'reasoning':    'Task 3 parse failed — nothing to score',
            'latency':      0.0,
        }

    user_prompt = USER_PROMPT_JUDGE_TEMPLATE.format(
        description   = desc,
        tasks         = json.dumps(t3_output['tasks']),
        conditions    = json.dumps(t3_output['conditions']),
        thresholds    = json.dumps(t3_output['thresholds']),
        inputs        = json.dumps(t3_output['inputs']),
        outputs       = json.dumps(t3_output['outputs']),
        logic_summary = t3_output['logic_summary'],
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_JUDGE,
    )

    if not isinstance(result, dict):
        return {
            'completeness': 0.0,
            'accuracy':     0.0,
            'f1':           0.0,
            'reasoning':    'Judge parse failed',
            'latency':      latency,
        }

    completeness = float(result.get('completeness', 0.0))
    accuracy     = float(result.get('accuracy',     0.0))

    # Clamp to [0, 1]
    completeness = max(0.0, min(1.0, completeness))
    accuracy     = max(0.0, min(1.0, accuracy))

    # F1 = harmonic mean of completeness and accuracy
    f1 = (2 * completeness * accuracy / (completeness + accuracy)
          if (completeness + accuracy) > 0 else 0.0)

    return {
        'completeness': round(completeness, 4),
        'accuracy':     round(accuracy,     4),
        'f1':           round(f1,           4),
        'reasoning':    result.get('reasoning', ''),
        'latency':      latency,
    }


print(' Task 3 scoring functions defined')
print('   score_intent_with_judge() — LLM-as-judge scores completeness + accuracy')

In [ ]:
# ── CELL 5: TASK 3 — SANITY CHECK (3 PROJECTS) ───────────────────────────────
print('── Task 3 Sanity Check (3 projects) ──\n')

for proj in projects[:3]:
    pid  = proj['project_id']
    desc = proj.get('description', '')

    # ── Task 1 first — get predicted components ───────────────────────────────
    t1 = run_task1_agent(proj, active_client, use_rag=True)

    # ── Task 3 — extract intent ───────────────────────────────────────────────
    t3 = run_task3_agent(proj, t1['predicted_names'], active_client)

    # ── Judge scoring ─────────────────────────────────────────────────────────
    score = score_intent_with_judge(proj, t3, active_client)

    print(f'Project {pid} [{proj.get("complexity")}]')
    print(f'  Description  : {desc[:100]}...')
    print(f'  Components   : {t1["predicted_names"]}')
    print()
    print(f'  ── Intent Extracted ──')
    print(f'  Tasks        : {t3["tasks"]}')
    print(f'  Conditions   : {t3["conditions"]}')
    print(f'  Thresholds   : {t3["thresholds"]}')
    print(f'  Inputs       : {t3["inputs"]}')
    print(f'  Outputs      : {t3["outputs"]}')
    print(f'  Summary      : {t3["logic_summary"]}')
    print()
    print(f'  ── Judge Scores ──')
    print(f'  Completeness : {score["completeness"]:.2f}')
    print(f'  Accuracy     : {score["accuracy"]:.2f}')
    print(f'  F1           : {score["f1"]:.2f}')
    print(f'  Reasoning    : {score["reasoning"]}')
    print(f'  T3 Latency   : {t3["latency"]:.2f}s')
    print(f'  Judge Latency: {score["latency"]:.2f}s')
    print()
    print('-' * 65)
    print()

In [ ]:
# ── CELL 6: FULL CONNECTED RUN — TASKS 1 + 2 + 3 ─────────────────────────────
SAVE_PATH         = 'connected_results_t1_t2_t3.json'
connected_results = []
error_count       = 0

for proj in tqdm(projects[:10], desc='Tasks 1+2+3'):
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')
    gt_libs = set(proj.get('ground_truth_libraries', []))

    # ══════════════════════════════════════════════════
    # TASK 1 — Component extraction
    # ══════════════════════════════════════════════════
    try:
        t1_output = run_task1_agent(proj, active_client, use_rag=True)
    except Exception as e:
        error_count += 1
        tqdm.write(f'  ERROR T1 project {pid}: {e}')
        continue

    gt_names        = get_gt_names(proj)
    predicted_names = t1_output['predicted_names']

    if not gt_names:
        continue

    # Task 1 scoring
    t1_scores    = match_predicted_to_gt(predicted_names, gt_names)
    t1_tp        = t1_scores['tp']
    t1_fp        = t1_scores['fp']
    t1_fn        = t1_scores['fn']
    t1_precision = t1_tp / (t1_tp + t1_fp) if (t1_tp + t1_fp) > 0 else 0
    t1_recall    = t1_tp / (t1_tp + t1_fn) if (t1_tp + t1_fn) > 0 else 0
    t1_f1        = (2 * t1_precision * t1_recall / (t1_precision + t1_recall)
                    if (t1_precision + t1_recall) > 0 else 0)

    # ══════════════════════════════════════════════════
    # TASK 2 — Library retrieval (batched)
    # ══════════════════════════════════════════════════
    try:
        t2_details = run_task2_agent(
            predicted_names, desc, active_client, board=board
        )
    except Exception as e:
        tqdm.write(f'  ERROR T2 project {pid}: {e}')
        t2_details = [
            {'component': n, 'library': None, 'status': 'error',
             'confidence': 'low', 'reasoning': str(e), 'latency': 0.0}
            for n in predicted_names
        ]

    retrieved_libs = {r['library'] for r in t2_details if r['library']}
    gt_canon       = canonicalize_set(gt_libs)
    ret_canon      = canonicalize_set(retrieved_libs)
    gt_canon       = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon      = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if gt_canon:
        t2_tp        = match_with_equivalence(gt_canon, ret_canon)
        t2_precision = t2_tp / len(ret_canon) if ret_canon else 0
        t2_recall    = t2_tp / len(gt_canon)  if gt_canon  else 0
        t2_f1        = (2 * t2_precision * t2_recall / (t2_precision + t2_recall)
                        if (t2_precision + t2_recall) > 0 else 0)
    else:
        t2_tp = t2_precision = t2_recall = t2_f1 = None

    # ══════════════════════════════════════════════════
    # TASK 3 — Intent understanding
    # ══════════════════════════════════════════════════
    try:
        t3_output = run_task3_agent(proj, predicted_names, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T3 project {pid}: {e}')
        t3_output = {
            'project_id': pid, 'status': 'error',
            'tasks': [], 'conditions': [], 'thresholds': [],
            'inputs': [], 'outputs': [], 'logic_summary': '',
            'latency': 0.0, 'raw': {},
        }

    # Judge scoring
    try:
        t3_score = score_intent_with_judge(proj, t3_output, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR Judge project {pid}: {e}')
        t3_score = {
            'completeness': 0.0, 'accuracy': 0.0,
            'f1': 0.0, 'reasoning': str(e), 'latency': 0.0,
        }

    # ══════════════════════════════════════════════════
    # SAVE
    # ══════════════════════════════════════════════════
    connected_results.append({
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,

        'task1': {
            'predicted_names': predicted_names,
            'gt_names':        list(gt_names),
            'matched':         list(t1_scores['matched_gt']),
            'precision':       round(t1_precision, 4),
            'recall':          round(t1_recall,    4),
            'f1':              round(t1_f1,        4),
            'latency':         t1_output['latency'],
        },

        'task2': {
            'gt_libs':        list(gt_canon)  if gt_canon  else [],
            'retrieved_libs': list(ret_canon) if ret_canon else [],
            'tp':             t2_tp,
            'precision':      round(t2_precision, 4) if t2_precision is not None else None,
            'recall':         round(t2_recall,    4) if t2_recall    is not None else None,
            'f1':             round(t2_f1,        4) if t2_f1        is not None else None,
            'details':        t2_details,
        },

        'task3': {
            'tasks':         t3_output['tasks'],
            'conditions':    t3_output['conditions'],
            'thresholds':    t3_output['thresholds'],
            'inputs':        t3_output['inputs'],
            'outputs':       t3_output['outputs'],
            'logic_summary': t3_output['logic_summary'],
            'completeness':  t3_score['completeness'],
            'accuracy':      t3_score['accuracy'],
            'f1':            t3_score['f1'],
            'reasoning':     t3_score['reasoning'],
            'status':        t3_output['status'],
            'latency':       t3_output['latency'],
            'judge_latency': t3_score['latency'],
        },
    })

    # Incremental save every 10 projects
    if len(connected_results) % 10 == 0:
        with open(SAVE_PATH, 'w') as f:
            json.dump(connected_results, f, indent=2)
        tqdm.write(f'  💾 Saved {len(connected_results)} results...')

# Final save
with open(SAVE_PATH, 'w') as f:
    json.dump(connected_results, f, indent=2)

print(f' Done — saved to {SAVE_PATH}')
print(f'   Evaluated : {len(connected_results)} | Errors: {error_count}')

In [ ]:
# ── CELL 7: PRINT RESULTS ─────────────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']

print('=' * 60)
print(' Connected Pipeline — Task 1 + Task 2 + Task 3')
print('=' * 60)
print(f'  Model              : {active_client.model_key}')
print(f'  Projects evaluated : {len(connected_results)}')
print(f'  Errors             : {error_count}')
print()

# ── Task 1 ────────────────────────────────────────────────────────────────────
print(f'  Task 1 — Component Extraction')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t1_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t1_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t1_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]   for r in t1_all]):.2f}s')
print()

# ── Task 2 ────────────────────────────────────────────────────────────────────
print(f'  Task 2 — Library Retrieval')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t2_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t2_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t2_all]):.3f}')
print(f'    Scored           : {len(t2_all)} / {len(connected_results)}')
print()

# ── Task 3 ────────────────────────────────────────────────────────────────────
print(f'  Task 3 — Intent Understanding')
print(f'    Avg Completeness : {np.mean([r["completeness"] for r in t3_all]):.3f}')
print(f'    Avg Accuracy     : {np.mean([r["accuracy"]     for r in t3_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]           for r in t3_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]      for r in t3_all]):.2f}s')
print(f'    Scored           : {len(t3_all)} / {len(connected_results)}')
print('=' * 60)

# ── F1 distributions ──────────────────────────────────────────────────────────
print('\n── Task 3 Score Distributions ──')

for label, values in [
    ('Completeness', [r['completeness'] for r in t3_all]),
    ('Accuracy',     [r['accuracy']     for r in t3_all]),
    ('F1',           [r['f1']           for r in t3_all]),
]:
    buckets = {'1.00': 0, '0.75-0.99': 0, '0.50-0.74': 0,
               '0.25-0.49': 0, '0.00-0.24': 0}
    for v in values:
        if v == 1.0:        buckets['1.00']      += 1
        elif v >= 0.75:     buckets['0.75-0.99'] += 1
        elif v >= 0.50:     buckets['0.50-0.74'] += 1
        elif v >= 0.25:     buckets['0.25-0.49'] += 1
        else:               buckets['0.00-0.24'] += 1
    print(f'\n  {label}:')
    for bucket, count in buckets.items():
        bar = '█' * (count // max(1, len(t3_all) // 20))
        print(f'    {bucket} : {count:>4}  {bar}')

# ── Per project sample ────────────────────────────────────────────────────────
print('\n── Per Project Sample (first 5) ──')
for r in connected_results[:5]:
    print(f"\n  Project {r['project_id']} [{r['complexity']}]")
    print(f"  T1 Predicted  : {r['task1']['predicted_names']}")
    print(f"  T1 F1         : {r['task1']['f1']:.2f}")
    print(f"  T2 F1         : {r['task2']['f1']}")
    print(f"  T3 Tasks      : {r['task3']['tasks']}")
    print(f"  T3 Conditions : {r['task3']['conditions']}")
    print(f"  T3 Thresholds : {r['task3']['thresholds']}")
    print(f"  T3 Summary    : {r['task3']['logic_summary']}")
    print(f"  T3 Complete   : {r['task3']['completeness']:.2f}")
    print(f"  T3 Accuracy   : {r['task3']['accuracy']:.2f}")
    print(f"  T3 F1         : {r['task3']['f1']:.2f}")
    print(f"  T3 Reasoning  : {r['task3']['reasoning']}")

In [ ]:
# ── CELL 8: TASK 3 — FAILURE ANALYSIS ────────────────────────────────────────
t3_all    = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t3_failed = [r for r in connected_results if r['task3']['status'] != 'ok']
t3_low    = [r for r in connected_results if r['task3']['f1'] < 0.2]
t3_zero   = [r for r in connected_results if r['task3']['f1'] == 0.0]
t3_perf   = [r for r in connected_results if r['task3']['f1'] == 1.0]

print('=' * 60)
print(' Task 3 — Failure Analysis')
print('=' * 60)
print(f'  Total evaluated    : {len(connected_results)}')
print(f'  Successfully scored: {len(t3_all)}')
print(f'  Parse failed       : {len(t3_failed)}')
print()
print(f'  Perfect F1  (1.00) : {len(t3_perf)} / {len(t3_all)}')
print(f'  Low F1  (< 0.20)   : {len(t3_low)} / {len(t3_all)}')
print(f'  Zero F1 (0.00)     : {len(t3_zero)} / {len(t3_all)}')
print('=' * 60)

# ── Score breakdown ───────────────────────────────────────────────────────────
comp_scores = [r['task3']['completeness'] for r in connected_results
               if r['task3']['status'] == 'ok']
acc_scores  = [r['task3']['accuracy']     for r in connected_results
               if r['task3']['status'] == 'ok']

print(f'\n── Score Breakdown ──')
print(f'  Completeness — min:{min(comp_scores):.2f}  '
      f'max:{max(comp_scores):.2f}  '
      f'avg:{np.mean(comp_scores):.2f}  '
      f'std:{np.std(comp_scores):.2f}')
print(f'  Accuracy     — min:{min(acc_scores):.2f}  '
      f'max:{max(acc_scores):.2f}  '
      f'avg:{np.mean(acc_scores):.2f}  '
      f'std:{np.std(acc_scores):.2f}')

# ── What's being missed ───────────────────────────────────────────────────────
empty_tasks      = [r for r in connected_results if len(r['task3']['tasks'])      == 0]
empty_conditions = [r for r in connected_results if len(r['task3']['conditions']) == 0]
empty_thresholds = [r for r in connected_results if len(r['task3']['thresholds']) == 0]
empty_inputs     = [r for r in connected_results if len(r['task3']['inputs'])     == 0]
empty_outputs    = [r for r in connected_results if len(r['task3']['outputs'])    == 0]

print(f'\n── Empty Fields (missing extractions) ──')
print(f'  No tasks extracted      : {len(empty_tasks)}')
print(f'  No conditions extracted : {len(empty_conditions)}')
print(f'  No thresholds extracted : {len(empty_thresholds)}')
print(f'  No inputs extracted     : {len(empty_inputs)}')
print(f'  No outputs extracted    : {len(empty_outputs)}')

# ── Complexity breakdown ──────────────────────────────────────────────────────
print(f'\n── T3 F1 by Complexity ──')
for complexity in ['Low', 'Medium', 'High', 'Unknown']:
    subset = [r for r in connected_results
              if r['complexity'] == complexity
              and r['task3']['status'] == 'ok']
    if not subset: continue
    avg_f1  = np.mean([r['task3']['f1']           for r in subset])
    avg_com = np.mean([r['task3']['completeness']  for r in subset])
    avg_acc = np.mean([r['task3']['accuracy']      for r in subset])
    print(f'  {complexity:<8} ({len(subset):>3} projects) — '
          f'F1:{avg_f1:.2f}  '
          f'Completeness:{avg_com:.2f}  '
          f'Accuracy:{avg_acc:.2f}')

# ── Low F1 sample cases ───────────────────────────────────────────────────────
if t3_low:
    print(f'\n── Sample Low-F1 Projects (first 3) ──')
    for r in t3_low[:3]:
        print(f"\n  Project {r['project_id']} [{r['complexity']}]")
        print(f"  Description  : {next((p.get('description','') for p in projects if p['project_id'] == r['project_id']), '')[:100]}")
        print(f"  Tasks        : {r['task3']['tasks']}")
        print(f"  Conditions   : {r['task3']['conditions']}")
        print(f"  Thresholds   : {r['task3']['thresholds']}")
        print(f"  Completeness : {r['task3']['completeness']:.2f}")
        print(f"  Accuracy     : {r['task3']['accuracy']:.2f}")
        print(f"  F1           : {r['task3']['f1']:.2f}")
        print(f"  Reasoning    : {r['task3']['reasoning']}")

# ── Parse failures ────────────────────────────────────────────────────────────
if t3_failed:
    print(f'\n── Parse Failed Projects ({len(t3_failed)}) ──')
    for r in t3_failed[:5]:
        print(f"  Project {r['project_id']} — status: {r['task3']['status']}")

In [ ]:
# ── CELL 9: SAVE RESULTS ──────────────────────────────────────────────────────
import json
from pathlib import Path

SAVE_PATH = 'task3_results_final.json'

# ── Build summary ─────────────────────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']

summary = {
    'model':           active_client.model_key,
    'total_evaluated': len(connected_results),
    'errors':          error_count,

    'task1': {
        'avg_precision': round(np.mean([r['precision'] for r in t1_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t1_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t1_all]), 4),
        'avg_latency':   round(np.mean([r['latency']   for r in t1_all]), 3),
        'perfect_f1':    len([r for r in t1_all if r['f1'] == 1.0]),
        'zero_f1':       len([r for r in t1_all if r['f1'] == 0.0]),
    },

    'task2': {
        'avg_precision': round(np.mean([r['precision'] for r in t2_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t2_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t2_all]), 4),
        'scored':        len(t2_all),
        'perfect_f1':    len([r for r in t2_all if r['f1'] == 1.0]),
        'zero_f1':       len([r for r in t2_all if r['f1'] == 0.0]),
    },

    'task3': {
        'avg_completeness': round(np.mean([r['completeness'] for r in t3_all]), 4),
        'avg_accuracy':     round(np.mean([r['accuracy']     for r in t3_all]), 4),
        'avg_f1':           round(np.mean([r['f1']           for r in t3_all]), 4),
        'avg_latency':      round(np.mean([r['latency']      for r in t3_all]), 3),
        'scored':           len(t3_all),
        'perfect_f1':       len([r for r in t3_all if r['f1'] == 1.0]),
        'zero_f1':          len([r for r in t3_all if r['f1'] == 0.0]),
        'parse_failed':     len([r for r in connected_results
                                 if r['task3']['status'] != 'ok']),
        # Complexity breakdown
        'by_complexity': {
            c: {
                'count':           len([r for r in connected_results
                                        if r['complexity'] == c
                                        and r['task3']['status'] == 'ok']),
                'avg_f1':          round(np.mean([r['task3']['f1']
                                        for r in connected_results
                                        if r['complexity'] == c
                                        and r['task3']['status'] == 'ok'] or [0]), 4),
                'avg_completeness':round(np.mean([r['task3']['completeness']
                                        for r in connected_results
                                        if r['complexity'] == c
                                        and r['task3']['status'] == 'ok'] or [0]), 4),
                'avg_accuracy':    round(np.mean([r['task3']['accuracy']
                                        for r in connected_results
                                        if r['complexity'] == c
                                        and r['task3']['status'] == 'ok'] or [0]), 4),
            }
            for c in ['Low', 'Medium', 'High', 'Unknown']
        },
    },

    'results': connected_results,
}

# ── Save ──────────────────────────────────────────────────────────────────────
with open(SAVE_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(f' Saved → {SAVE_PATH}')
print()
print('── Final Summary ──')
print(f'  Task 1 — Avg F1           : {summary["task1"]["avg_f1"]}')
print(f'  Task 2 — Avg F1           : {summary["task2"]["avg_f1"]}')
print(f'  Task 3 — Avg F1           : {summary["task3"]["avg_f1"]}')
print(f'  Task 3 — Avg Completeness : {summary["task3"]["avg_completeness"]}')
print(f'  Task 3 — Avg Accuracy     : {summary["task3"]["avg_accuracy"]}')
print()
print('── Task 3 by Complexity ──')
for c, stats in summary['task3']['by_complexity'].items():
    if stats['count'] == 0:
        continue
    print(f'  {c:<8} ({stats["count"]:>3} projects) — '
          f'F1:{stats["avg_f1"]:.3f}  '
          f'Completeness:{stats["avg_completeness"]:.3f}  '
          f'Accuracy:{stats["avg_accuracy"]:.3f}')

# ── Update markdown ───────────────────────────────────────────────────────────
print(f"""
── Update Task 3 markdown to ──

**Status:** Complete

| Metric           | Score  |
|------------------|--------|
| Avg Completeness | {summary['task3']['avg_completeness']} |
| Avg Accuracy     | {summary['task3']['avg_accuracy']}     |
| Avg F1           | {summary['task3']['avg_f1']}           |
| Projects scored  | {summary['task3']['scored']}           |
""")


# ⚙️ Task 4: Hardware Feasibility Mapping
**Status:** ✅ Complete (10-project sample)

| Metric | Score |
|---|---|
| Avg Pin Validity | 0.980 |
| Avg Coverage | 1.000 |
| Avg Conflict Free | 1.000 |
| Avg F1 | 0.992 |
| Projects scored | 10 |

| Complexity | F1 | Pin Validity | Coverage |
|---|---|---|---|
| Low | 1.000 | 1.000 | 1.000 |
| Medium | 1.000 | 1.000 | 1.000 |
| High | 0.962 | 0.900 | 1.000 |

In [ ]:
print('── Pin-related columns in df_comp ──\n')
print(df_comp[['Component_Id', 'Component name', 'Pin Type', 
               'Pin Description']].head(10).to_string())

print('\n── Sample Pin Descriptions ──\n')
for _, row in df_comp.head(5).iterrows():
    print(f"ID {row['Component_Id']}: {row['Component name']}")
    print(f"  Pin Type : {row['Pin Type']}")
    print(f"  Pin Desc : {str(row['Pin Description'])[:200]}")
    print()

In [ ]:
# ── FORCE REDEFINE — PIN SPEC EXTRACTOR ──────────────────────────────────────

BOARD_PIN_RULES = {
    'Arduino Uno': {
        'digital':  list(range(2, 14)),
        'pwm':      [3, 5, 6, 9, 10, 11],
        'analog':   ['A0','A1','A2','A3','A4','A5'],
        'i2c':      {'sda': 'A4', 'scl': 'A5'},
        'spi':      {'ss': 10, 'mosi': 11, 'miso': 12, 'sck': 13},
        'uart':     {'rx': 0, 'tx': 1},
        'avoid':    [0, 1],
    },
    'Arduino Mega': {
        'digital':  list(range(2, 54)),
        'pwm':      [2,3,4,5,6,7,8,9,10,11,12,13,44,45,46],
        'analog':   [f'A{i}' for i in range(16)],
        'i2c':      {'sda': 20, 'scl': 21},
        'spi':      {'ss': 53, 'mosi': 51, 'miso': 50, 'sck': 52},
        'uart':     {'rx': 0, 'tx': 1},
        'avoid':    [0, 1],
    },
    'Arduino Uno R4 Wifi Module': {
        'digital':  list(range(2, 14)),
        'pwm':      [3, 5, 6, 9, 10, 11],
        'analog':   ['A0','A1','A2','A3','A4','A5'],
        'i2c':      {'sda': 'A4', 'scl': 'A5'},
        'spi':      {'ss': 10, 'mosi': 11, 'miso': 12, 'sck': 13},
        'uart':     {'rx': 0, 'tx': 1},
        'avoid':    [0, 1],
    },
}

DEFAULT_BOARD = 'Arduino Uno'

def get_board_rules(board: str) -> dict:
    return BOARD_PIN_RULES.get(board, BOARD_PIN_RULES[DEFAULT_BOARD])


def parse_pin_description(pin_desc_str: str) -> list:
    if not pin_desc_str or str(pin_desc_str).strip().lower() in ('nan', 'none', ''):
        return []
    try:
        pins = json.loads(str(pin_desc_str))
    except Exception:
        return []
    return [
        p for p in pins
        if isinstance(p, dict)
        and p.get('kind') not in ('power', 'ground')
    ]


COMP_LOOKUP_ALIASES = {
    'dht22 sensor':         'grove temperature & humidity sensor pro',
    'dht22':                'grove temperature & humidity sensor pro',
    'dht11 sensor':         'grove temperature & humidity sensor',
    'dht11':                'grove temperature & humidity sensor',
    'dht sensor':           'grove temperature & humidity sensor',
    'temperature sensor':   'grove temperature & humidity sensor',
    'humidity sensor':      'grove temperature & humidity sensor',
    'ultrasonic sensor':    'waterproof ultrasonic sensor',
    'ir sensor':            'grove ir distance interrupter',
    'oled display':         'oled display',
    'bluetooth module':     'hc-05 bluetooth module',
    'soil moisture sensor': 'gravity analog capacitive soil moisture sensor',
    'relay module':         'gravity digital 5a relay module',
}


def get_component_pin_specs(component_name: str, df_components) -> dict:
    # Apply alias first
    name_lower  = component_name.lower().strip()
    name_lower  = COMP_LOOKUP_ALIASES.get(name_lower, name_lower)
    name_tokens = set(name_lower.split())

    best_score = 0.0
    best_row   = None

    for _, row in df_components.iterrows():
        comp_name   = str(row['Component name']).lower().strip()
        comp_tokens = set(comp_name.split())

        if name_lower == comp_name:
            best_row = row
            best_score = 1.0 
            break

        if name_lower in comp_name or comp_name in name_lower:
            score = 0.95
        else:
            overlap = (len(name_tokens & comp_tokens) / len(name_tokens | comp_tokens)
                       if name_tokens and comp_tokens else 0.0)
            fuzzy   = difflib.SequenceMatcher(None, name_lower, comp_name).ratio()
            score   = max(overlap, fuzzy)

        if score > best_score:
            best_score = score
            best_row   = row

    if best_row is None or best_score < 0.40:
        return {
            'matched_name': None,
            'pin_type':     None,
            'pins':         [],
            'status':       'no_match',
        }

    pins = parse_pin_description(best_row.get('Pin Description', ''))

    if not pins:
        return {
            'matched_name': str(best_row['Component name']),
            'pin_type':     str(best_row['Pin Type']),
            'pins':         [],
            'status':       'no_pins',
        }

    return {
        'matched_name': str(best_row['Component name']),
        'pin_type':     str(best_row['Pin Type']),
        'pins':         pins,
        'status':       'ok',
    }


print('── get_component_pin_specs test ──\n')

test_components = [
    'Ultrasonic Sensor',
    'Servo Motor',
    'Soil Moisture Sensor',
    'OLED Display',
    'DHT22 Sensor',
    'Relay Module',
    'DC Motor',
    'Bluetooth Module',
]

for name in test_components:
    result = get_component_pin_specs(name, df_comp)
    print(f'  {name:<30} → matched: {str(result["matched_name"]):<45} '
          f'type: {str(result["pin_type"]):<15} '
          f'pins: {[p["name"] for p in result["pins"]]}  '
          f'status: {result["status"]}')

print('\n── Alias check ──')
print(f'  "dht22 sensor" → {COMP_LOOKUP_ALIASES.get("dht22 sensor")}')
print(f'  "dht22"        → {COMP_LOOKUP_ALIASES.get("dht22")}')

In [ ]:
# ── CELL 3: TASK 4 — PROMPTS ──────────────────────────────────────────────────
SYSTEM_PROMPT_MODULE3B = """You are a Physical Hardware Mapper for Arduino projects.

Given a list of components with their pin specifications, assign each signal pin 
to a specific physical pin number on the target board.

Rules:
- NEVER assign two components to the same pin number
- Digital sensors/actuators → use digital pins (2-13)
- Analog sensors → MUST use analog pins (A0-A5)
- PWM components (servo, motor speed) → MUST use PWM pins: 3, 5, 6, 9, 10, 11
- I2C components → MUST use SDA=A4, SCL=A5 (shared bus, multiple devices ok)
- UART/Serial → use pins 0(RX) and 1(TX) — avoid if possible, use SoftwareSerial
- SPI → SS=10, MOSI=11, MISO=12, SCK=13
- Skip VCC, GND, NC, STATE, EN pins — these do not need Arduino pin assignment
- Output ONLY a JSON object. No explanation, no markdown, no preamble.

Output format:
{
  "assignments": [
    {
      "component":  "Ultrasonic Sensor",
      "pin_type":   "Digital",
      "pins": {
        "TRIG": 5,
        "ECHO": 6
      }
    },
    {
      "component":  "Servo Motor",
      "pin_type":   "PWM",
      "pins": {
        "SIGNAL": 9
      }
    }
  ],
  "conflicts":   [],
  "feasibility": "feasible"
}

feasibility values:
- "feasible"              → all components can be connected without conflicts
- "feasible_with_notes"   → works but with constraints (e.g. shared I2C bus)
- "infeasible"            → not enough pins or irresolvable conflicts

--- EXAMPLES ---

Example 1:
Board: Arduino Uno
Components:
  - Ultrasonic Sensor | Digital | pins: TRIG(input), ECHO(output)
  - Servo Motor       | PWM     | pins: SIGNAL
Output:
{
  "assignments": [
    {"component": "Ultrasonic Sensor", "pin_type": "Digital",
     "pins": {"TRIG": 5, "ECHO": 6}},
    {"component": "Servo Motor",       "pin_type": "PWM",
     "pins": {"SIGNAL": 9}}
  ],
  "conflicts":   [],
  "feasibility": "feasible"
}

Example 2:
Board: Arduino Uno
Components:
  - OLED Display      | I2C     | pins: SDA, SCL
  - DHT22 Sensor      | Digital | pins: DATA
  - Relay Module      | Digital | pins: IN
Output:
{
  "assignments": [
    {"component": "OLED Display",  "pin_type": "I2C",
     "pins": {"SDA": "A4", "SCL": "A5"}},
    {"component": "DHT22 Sensor",  "pin_type": "Digital",
     "pins": {"DATA": 4}},
    {"component": "Relay Module",  "pin_type": "Digital",
     "pins": {"IN": 7}}
  ],
  "conflicts":   [],
  "feasibility": "feasible"
}

Example 3:
Board: Arduino Uno
Components:
  - Soil Moisture Sensor | Analog | pins: AOUT
  - Bluetooth Module     | UART   | pins: TXD, RXD
  - DC Motor             | PWM    | pins: DIR, SPD
Output:
{
  "assignments": [
    {"component": "Soil Moisture Sensor", "pin_type": "Analog",
     "pins": {"AOUT": "A0"}},
    {"component": "Bluetooth Module",     "pin_type": "UART",
     "pins": {"TXD": 2, "RXD": 3}},
    {"component": "DC Motor",             "pin_type": "Digital/PWM",
     "pins": {"DIR": 4, "SPD": 5}}
  ],
  "conflicts":   [],
  "feasibility": "feasible"
}

--- END EXAMPLES ---"""


USER_PROMPT_MODULE3B_TEMPLATE = """Assign Arduino pin numbers for each component below.

Board: {board}

Components and their pin specifications:
{component_pin_specs}

Rules reminder:
- Analog sensors → A0-A5 only
- PWM outputs    → pins 3, 5, 6, 9, 10, 11 only
- I2C            → SDA=A4, SCL=A5
- No pin conflicts

Return ONLY the JSON object."""


print(' Task 4 prompts defined')

In [ ]:
# ── CELL 4: TASK 4 — AGENT FUNCTION ──────────────────────────────────────────

def build_component_pin_specs_str(predicted_names: list, df_components) -> tuple:
    """
    Build formatted string of component pin specs for the prompt.
    Returns (formatted_string, list of spec dicts for scoring)
    """
    lines      = []
    spec_list  = []

    for name in predicted_names:
        spec = get_component_pin_specs(name, df_components)

        spec_list.append({
            'component':    name,
            'matched_name': spec['matched_name'],
            'pin_type':     spec['pin_type'],
            'pins':         spec['pins'],
            'status':       spec['status'],
        })

        if spec['status'] == 'no_match':
            lines.append(f'  - {name} | Unknown | pins: unknown')
            continue

        if spec['status'] == 'no_pins':
            lines.append(f'  - {name} | {spec["pin_type"]} | pins: none required')
            continue

        # Format signal pins
        pin_strs = []
        for p in spec['pins']:
            pin_name      = p.get('name', '')
            pin_kind      = p.get('kind', '')
            pin_direction = p.get('direction', '')
            bus_role      = p.get('bus_role', '')

            if bus_role:
                pin_strs.append(f'{pin_name}({bus_role})')
            elif pin_direction:
                pin_strs.append(f'{pin_name}({pin_direction})')
            else:
                pin_strs.append(pin_name)

        lines.append(
            f'  - {name} | {spec["pin_type"]} | '
            f'pins: {", ".join(pin_strs)}'
        )

    return '\n'.join(lines), spec_list


def run_task4_agent(project: dict, predicted_names: list,
                    client, df_components) -> dict:
    """
    Task 4 Agent: LLM assigns physical pin numbers to each component.
    Input  → predicted component names + dictionary pin specs
    Output → pin assignments, conflicts, feasibility
    """
    board = project.get('board', 'Arduino Uno')

    # Build component pin spec string for prompt
    comp_spec_str, spec_list = build_component_pin_specs_str(
        predicted_names, df_components
    )

    user_prompt = USER_PROMPT_MODULE3B_TEMPLATE.format(
        board               = board,
        component_pin_specs = comp_spec_str,
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_MODULE3B,
    )

    if not isinstance(result, dict):
        return {
            'project_id':   project.get('project_id'),
            'status':       'parse_failed',
            'assignments':  [],
            'conflicts':    [],
            'feasibility':  'unknown',
            'spec_list':    spec_list,
            'latency':      latency,
        }

    assignments = result.get('assignments', [])
    conflicts   = result.get('conflicts',   [])
    feasibility = result.get('feasibility', 'unknown')

    # Ensure assignments is a list
    if not isinstance(assignments, list):
        assignments = []

    return {
        'project_id':   project.get('project_id'),
        'status':       'ok',
        'assignments':  assignments,
        'conflicts':    conflicts,
        'feasibility':  feasibility,
        'spec_list':    spec_list,
        'latency':      latency,
        'raw':          result,
    }


print(' run_task4_agent defined')

In [ ]:
# ── CELL 5: TASK 4 — SCORING + VALIDATION ────────────────────────────────────

PIN_KIND_RULES = {
    'digital':  'digital',
    'analog':   'analog',
    'bus':      'bus',      
    'pwm':      'pwm',
}

SKIP_PINS = {'vcc', 'gnd', 'nc', 'state', 'en', 'power', 'ground'}


def normalize_pin_number(pin_val) -> str:
    """Normalize pin number to string for comparison. e.g. 9 → '9', 'A4' → 'a4'"""
    return str(pin_val).lower().strip()


def validate_pin_assignment(pin_name: str, pin_number,
                             pin_kind: str, board_rules: dict) -> dict:
    """
    Validate a single pin assignment against board rules.
    Returns {valid: bool, reason: str}
    """
    pin_name_lower = pin_name.lower()
    pin_str        = normalize_pin_number(pin_number)

    # Skip non-signal pins
    if pin_name_lower in SKIP_PINS:
        return {'valid': True, 'reason': 'skipped — power/ground/nc pin'}

    # ── I2C bus pins ──────────────────────────────────────────────────────────
    if pin_kind == 'bus':
        i2c = board_rules.get('i2c', {})
        spi = board_rules.get('spi', {})

        if 'sda' in pin_name_lower:
            expected = normalize_pin_number(i2c.get('sda', 'a4'))
            if pin_str == expected:
                return {'valid': True, 'reason': 'correct SDA pin'}
            return {'valid': False,
                    'reason': f'SDA must be {expected}, got {pin_str}'}

        if 'scl' in pin_name_lower:
            expected = normalize_pin_number(i2c.get('scl', 'a5'))
            if pin_str == expected:
                return {'valid': True, 'reason': 'correct SCL pin'}
            return {'valid': False,
                    'reason': f'SCL must be {expected}, got {pin_str}'}

        if 'i2c_sda' in pin_name_lower or 'i2c_scl' in pin_name_lower:
            return {'valid': True, 'reason': 'I2C bus pin'}

        # UART bus
        if 'tx' in pin_name_lower or 'rx' in pin_name_lower:
            return {'valid': True, 'reason': 'UART pin — valid'}

        # SPI bus
        if any(k in pin_name_lower for k in ['mosi', 'miso', 'sck', 'ss', 'cs']):
            return {'valid': True, 'reason': 'SPI pin — valid'}

        return {'valid': True, 'reason': 'bus pin — assumed valid'}

    # ── Analog pins ───────────────────────────────────────────────────────────
    if pin_kind == 'analog':
        analog_pins = [normalize_pin_number(p)
                       for p in board_rules.get('analog', [])]
        if pin_str in analog_pins:
            return {'valid': True, 'reason': f'correct analog pin {pin_str}'}
        return {'valid': False,
                'reason': f'analog sensor must use A0-A5, got {pin_str}'}

    # ── PWM pins ─────────────────────────────────────────────────────────────
    if pin_kind == 'pwm' or 'signal' in pin_name_lower or 'spd' in pin_name_lower:
        pwm_pins = [normalize_pin_number(p)
                    for p in board_rules.get('pwm', [])]
        if pin_str in pwm_pins:
            return {'valid': True, 'reason': f'correct PWM pin {pin_str}'}
        # PWM components can also use digital for direction pins
        digital_pins = [normalize_pin_number(p)
                        for p in board_rules.get('digital', [])]
        if pin_str in digital_pins:
            return {'valid': True,
                    'reason': f'digital pin {pin_str} acceptable for direction'}
        return {'valid': False,
                'reason': f'PWM component needs PWM pin, got {pin_str}'}

    # ── Digital pins ─────────────────────────────────────────────────────────
    if pin_kind == 'digital':
        digital_pins = [normalize_pin_number(p)
                        for p in board_rules.get('digital', [])]
        analog_pins  = [normalize_pin_number(p)
                        for p in board_rules.get('analog', [])]
        if pin_str in digital_pins or pin_str in analog_pins:
            return {'valid': True, 'reason': f'valid digital pin {pin_str}'}
        return {'valid': False,
                'reason': f'invalid digital pin {pin_str}'}

    # Default — unknown kind
    return {'valid': True, 'reason': f'unknown pin kind — assumed valid'}


def check_pin_conflicts(assignments: list) -> list:
    """
    Check for pin conflicts — two components using the same pin number.
    I2C pins (A4, A5) are shared bus — allowed for multiple I2C devices.
    Returns list of conflict descriptions.
    """
    pin_usage  = {}   # pin_number → list of (component, pin_name)
    conflicts  = []
    I2C_SHARED = {'a4', 'a5'}   # shared I2C bus pins

    for assignment in assignments:
        comp  = assignment.get('component', '')
        pins  = assignment.get('pins', {})
        ptype = assignment.get('pin_type', '').lower()

        for pin_name, pin_number in pins.items():
            if pin_name.lower() in SKIP_PINS:
                continue
            pin_str = normalize_pin_number(pin_number)

            # I2C shared bus — allow multiple devices on A4/A5
            if pin_str in I2C_SHARED and 'i2c' in ptype:
                continue

            if pin_str not in pin_usage:
                pin_usage[pin_str] = []
            pin_usage[pin_str].append((comp, pin_name))

    for pin_str, usages in pin_usage.items():
        if len(usages) > 1:
            desc = f'Pin {pin_str} conflict: ' + \
                   ' vs '.join([f'{c}({p})' for c, p in usages])
            conflicts.append(desc)

    return conflicts


def score_task4(t4_output: dict, board: str) -> dict:
    """
    Score Task 4 output on three metrics:
    1. Pin Validity Rate  → % pins assigned to correct type
    2. Conflict Rate      → 1.0 if no conflicts, 0.0 if any
    3. Coverage Rate      → % required components assigned
    """
    if t4_output['status'] != 'ok':
        return {
            'pin_validity':  0.0,
            'conflict_free': 0.0,
            'coverage':      0.0,
            'f1':            0.0,
            'conflicts':     [],
            'details':       [],
        }

    board_rules  = get_board_rules(board)
    assignments  = t4_output['assignments']
    spec_list    = t4_output['spec_list']

    # ── 1. Pin Validity ───────────────────────────────────────────────────────
    total_pins  = 0
    valid_pins  = 0
    details     = []

    for assignment in assignments:
        comp      = assignment.get('component', '')
        pins      = assignment.get('pins', {})
        pin_type  = assignment.get('pin_type', '')

        # Get original pin specs from dictionary
        orig_spec = next(
            (s for s in spec_list if s['component'] == comp), {}
        )
        orig_pins = {
            p['name'].lower(): p
            for p in orig_spec.get('pins', [])
        }

        for pin_name, pin_number in pins.items():
            if pin_name.lower() in SKIP_PINS:
                continue

            total_pins += 1

            # Get pin kind from original spec
            orig_pin  = orig_pins.get(pin_name.lower(), {})
            pin_kind  = orig_pin.get('kind', 'digital')
            bus_role  = orig_pin.get('bus_role', '')

            # Use bus_role for I2C/SPI/UART if available
            if bus_role:
                pin_kind = 'bus'

            validation = validate_pin_assignment(
                pin_name, pin_number, pin_kind, board_rules
            )

            if validation['valid']:
                valid_pins += 1

            details.append({
                'component':  comp,
                'pin_name':   pin_name,
                'pin_number': pin_number,
                'pin_kind':   pin_kind,
                'valid':      validation['valid'],
                'reason':     validation['reason'],
            })

    pin_validity = valid_pins / total_pins if total_pins > 0 else 0.0

    # ── 2. Conflict check ─────────────────────────────────────────────────────
    detected_conflicts = check_pin_conflicts(assignments)
    conflict_free      = 1.0 if not detected_conflicts else 0.0

    # ── 3. Coverage ───────────────────────────────────────────────────────────
    assigned_comps = {a.get('component', '').lower() for a in assignments}
    required_comps = {
        s['component'].lower() for s in spec_list
        if s['status'] in ('ok', 'no_pins')
    }
    coverage = (len(assigned_comps & required_comps) / len(required_comps)
                if required_comps else 0.0)

    # ── F1 = harmonic mean of all three ──────────────────────────────────────
    scores = [pin_validity, conflict_free, coverage]
    f1     = (3 / sum(1/s for s in scores if s > 0)
              if all(s > 0 for s in scores) else 0.0)

    return {
        'pin_validity':  round(pin_validity,  4),
        'conflict_free': round(conflict_free, 4),
        'coverage':      round(coverage,      4),
        'f1':            round(f1,            4),
        'conflicts':     detected_conflicts,
        'details':       details,
    }


# ── Quick sanity test ─────────────────────────────────────────────────────────
print(' Task 4 scoring functions defined')
print()
print('── Board rules check ──')
rules = get_board_rules('Arduino Uno')
print(f'  PWM pins   : {rules["pwm"]}')
print(f'  Analog pins: {rules["analog"]}')
print(f'  I2C        : {rules["i2c"]}')

In [ ]:
# ── CELL 6: TASK 4 — SANITY CHECK (3 PROJECTS) ───────────────────────────────
print('── Task 4 Sanity Check (3 projects) ──\n')

for proj in projects[:3]:
    pid  = proj['project_id']
    board = proj.get('board', 'Arduino Uno')

    # Task 1 first
    t1 = run_task1_agent(proj, active_client, use_rag=True)
    predicted_names = t1['predicted_names']

    print(f'Project {pid} [{proj.get("complexity")}] | Board: {board}')
    print(f'  Components : {predicted_names}')
    print()

    # Task 4
    t4 = run_task4_agent(proj, predicted_names, active_client, df_comp)

    # Score
    score = score_task4(t4, board)

    print(f'  ── Pin Assignments ──')
    for a in t4['assignments']:
        print(f'    {a["component"]:<35} | {a["pin_type"]:<15} | {a["pins"]}')

    print()
    print(f'  ── Validation ──')
    for d in score['details']:
        icon = 'Yes' if d['valid'] else 'No'
        print(f'    {icon} {d["component"]:<30} '
              f'{d["pin_name"]:<8} → {str(d["pin_number"]):<6} '
              f'({d["pin_kind"]}) — {d["reason"]}')

    print()
    print(f'  ── Scores ──')
    print(f'    Pin Validity  : {score["pin_validity"]:.2f}')
    print(f'    Conflict Free : {score["conflict_free"]:.2f}')
    print(f'    Coverage      : {score["coverage"]:.2f}')
    print(f'    F1            : {score["f1"]:.2f}')

    if score['conflicts']:
        print(f'      Conflicts  : {score["conflicts"]}')
    else:
        print(f'     No conflicts')

    print(f'    Feasibility   : {t4["feasibility"]}')
    print(f'    Latency       : {t4["latency"]:.2f}s')
    print()
    print('-' * 65)
    print()

In [ ]:
# ── CELL 7: FULL CONNECTED RUN — TASKS 1 + 2 + 3 + 4 ─────────────────────────
SAVE_PATH         = 'connected_results_t1_t2_t3_t4.json'
connected_results = []
error_count       = 0

for proj in tqdm(projects[:10], desc='Tasks 1+2+3+4'):
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')
    gt_libs = set(proj.get('ground_truth_libraries', []))

    # TASK 1 — Component extraction
    try:
        t1_output = run_task1_agent(proj, active_client, use_rag=True)
    except Exception as e:
        error_count += 1
        tqdm.write(f'  ERROR T1 project {pid}: {e}')
        continue

    gt_names        = get_gt_names(proj)
    predicted_names = t1_output['predicted_names']

    if not gt_names:
        continue

    # Task 1 scoring
    t1_scores    = match_predicted_to_gt(predicted_names, gt_names)
    t1_tp        = t1_scores['tp']
    t1_fp        = t1_scores['fp']
    t1_fn        = t1_scores['fn']
    t1_precision = t1_tp / (t1_tp + t1_fp) if (t1_tp + t1_fp) > 0 else 0
    t1_recall    = t1_tp / (t1_tp + t1_fn) if (t1_tp + t1_fn) > 0 else 0
    t1_f1        = (2 * t1_precision * t1_recall / (t1_precision + t1_recall)
                    if (t1_precision + t1_recall) > 0 else 0)

    # TASK 2 — Library retrieval (batched)
    try:
        t2_details = run_task2_agent(
            predicted_names, desc, active_client, board=board
        )
    except Exception as e:
        tqdm.write(f'  ERROR T2 project {pid}: {e}')
        t2_details = [
            {'component': n, 'library': None, 'status': 'error',
             'confidence': 'low', 'reasoning': str(e), 'latency': 0.0}
            for n in predicted_names
        ]

    retrieved_libs = {r['library'] for r in t2_details if r['library']}
    gt_canon       = canonicalize_set(gt_libs)
    ret_canon      = canonicalize_set(retrieved_libs)
    gt_canon       = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon      = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if gt_canon:
        t2_tp        = match_with_equivalence(gt_canon, ret_canon)
        t2_precision = t2_tp / len(ret_canon) if ret_canon else 0
        t2_recall    = t2_tp / len(gt_canon)  if gt_canon  else 0
        t2_f1        = (2 * t2_precision * t2_recall / (t2_precision + t2_recall)
                        if (t2_precision + t2_recall) > 0 else 0)
    else:
        t2_tp = t2_precision = t2_recall = t2_f1 = None

    # TASK 3 — Intent understanding
    try:
        t3_output = run_task3_agent(proj, predicted_names, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T3 project {pid}: {e}')
        t3_output = {
            'project_id': pid, 'status': 'error',
            'tasks': [], 'conditions': [], 'thresholds': [],
            'inputs': [], 'outputs': [], 'logic_summary': '',
            'latency': 0.0, 'raw': {},
        }

    try:
        t3_score = score_intent_with_judge(proj, t3_output, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR Judge project {pid}: {e}')
        t3_score = {
            'completeness': 0.0, 'accuracy': 0.0,
            'f1': 0.0, 'reasoning': str(e), 'latency': 0.0,
        }

    # TASK 4 — Hardware pin mapping
    try:
        t4_output = run_task4_agent(
            proj, predicted_names, active_client, df_comp
        )
    except Exception as e:
        tqdm.write(f'  ERROR T4 project {pid}: {e}')
        t4_output = {
            'project_id': pid, 'status': 'error',
            'assignments': [], 'conflicts': [],
            'feasibility': 'unknown', 'spec_list': [],
            'latency': 0.0,
        }

    try:
        t4_score = score_task4(t4_output, board)
    except Exception as e:
        tqdm.write(f'  ERROR T4 score project {pid}: {e}')
        t4_score = {
            'pin_validity': 0.0, 'conflict_free': 0.0,
            'coverage': 0.0, 'f1': 0.0,
            'conflicts': [], 'details': [],
        }

    connected_results.append({
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,

        'task1': {
            'predicted_names': predicted_names,
            'gt_names':        list(gt_names),
            'matched':         list(t1_scores['matched_gt']),
            'precision':       round(t1_precision, 4),
            'recall':          round(t1_recall,    4),
            'f1':              round(t1_f1,        4),
            'latency':         t1_output['latency'],
        },

        'task2': {
            'gt_libs':        list(gt_canon)  if gt_canon  else [],
            'retrieved_libs': list(ret_canon) if ret_canon else [],
            'tp':             t2_tp,
            'precision':      round(t2_precision, 4) if t2_precision is not None else None,
            'recall':         round(t2_recall,    4) if t2_recall    is not None else None,
            'f1':             round(t2_f1,        4) if t2_f1        is not None else None,
            'details':        t2_details,
        },

        'task3': {
            'tasks':         t3_output['tasks'],
            'conditions':    t3_output['conditions'],
            'thresholds':    t3_output['thresholds'],
            'inputs':        t3_output['inputs'],
            'outputs':       t3_output['outputs'],
            'logic_summary': t3_output['logic_summary'],
            'completeness':  t3_score['completeness'],
            'accuracy':      t3_score['accuracy'],
            'f1':            t3_score['f1'],
            'reasoning':     t3_score['reasoning'],
            'status':        t3_output['status'],
            'latency':       t3_output['latency'],
            'judge_latency': t3_score['latency'],
        },

        'task4': {
            'assignments':   t4_output['assignments'],
            'conflicts':     t4_score['conflicts'],
            'feasibility':   t4_output['feasibility'],
            'pin_validity':  t4_score['pin_validity'],
            'conflict_free': t4_score['conflict_free'],
            'coverage':      t4_score['coverage'],
            'f1':            t4_score['f1'],
            'details':       t4_score['details'],
            'status':        t4_output['status'],
            'latency':       t4_output['latency'],
        },
    })

    # Incremental save every 10 projects
    if len(connected_results) % 10 == 0:
        with open(SAVE_PATH, 'w') as f:
            json.dump(connected_results, f, indent=2)
        tqdm.write(f'  💾 Saved {len(connected_results)} results...')

# Final save
with open(SAVE_PATH, 'w') as f:
    json.dump(connected_results, f, indent=2)

print(f' Done — saved to {SAVE_PATH}')
print(f'   Evaluated : {len(connected_results)} | Errors: {error_count}')

In [ ]:
# ── CELL 8: PRINT RESULTS ─────────────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t4_all = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']

print('=' * 65)
print(' Connected Pipeline — Tasks 1 + 2 + 3 + 4')
print('=' * 65)
print(f'  Model              : {active_client.model_key}')
print(f'  Projects evaluated : {len(connected_results)}')
print(f'  Errors             : {error_count}')
print()

#  Task 1 
print(f'  Task 1 — Component Extraction')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t1_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t1_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t1_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]   for r in t1_all]):.2f}s')
print()

# Task 2 
print(f'  Task 2 — Library Retrieval')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t2_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t2_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t2_all]):.3f}')
print(f'    Scored           : {len(t2_all)} / {len(connected_results)}')
print()

# Task 3
print(f'  Task 3 — Intent Understanding')
print(f'    Avg Completeness : {np.mean([r["completeness"] for r in t3_all]):.3f}')
print(f'    Avg Accuracy     : {np.mean([r["accuracy"]     for r in t3_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]           for r in t3_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]      for r in t3_all]):.2f}s')
print(f'    Scored           : {len(t3_all)} / {len(connected_results)}')
print()

#  Task 4 
print(f'  Task 4 — Hardware Pin Mapping')
print(f'    Avg Pin Validity : {np.mean([r["pin_validity"]  for r in t4_all]):.3f}')
print(f'    Avg Conflict Free: {np.mean([r["conflict_free"] for r in t4_all]):.3f}')
print(f'    Avg Coverage     : {np.mean([r["coverage"]      for r in t4_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]            for r in t4_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]       for r in t4_all]):.2f}s')
print(f'    Scored           : {len(t4_all)} / {len(connected_results)}')
print('=' * 65)

# F1 distributions 
print('\n── Task 4 Score Distributions ──')
for label, values in [
    ('Pin Validity',  [r['pin_validity']  for r in t4_all]),
    ('Conflict Free', [r['conflict_free'] for r in t4_all]),
    ('Coverage',      [r['coverage']      for r in t4_all]),
    ('F1',            [r['f1']            for r in t4_all]),
]:
    buckets = {'1.00': 0, '0.75-0.99': 0, '0.50-0.74': 0,
               '0.25-0.49': 0, '0.00-0.24': 0}
    for v in values:
        if v == 1.0:      buckets['1.00']      += 1
        elif v >= 0.75:   buckets['0.75-0.99'] += 1
        elif v >= 0.50:   buckets['0.50-0.74'] += 1
        elif v >= 0.25:   buckets['0.25-0.49'] += 1
        else:             buckets['0.00-0.24'] += 1
    print(f'\n  {label}:')
    for bucket, count in buckets.items():
        bar = '█' * (count // max(1, len(t4_all) // 20))
        print(f'    {bucket} : {count:>4}  {bar}')

print('\n── Per Project Sample (first 10) ──')
for r in connected_results[:10]:
    print(f"\n  Project {r['project_id']} [{r['complexity']}] | {r['board']}")
    print(f"  T1 Predicted  : {r['task1']['predicted_names']}")
    print(f"  T1 F1         : {r['task1']['f1']:.2f}")
    print(f"  T2 F1         : {r['task2']['f1']}")
    print(f"  T3 F1         : {r['task3']['f1']:.2f}")
    print(f"  T4 Assignments:")
    for a in r['task4']['assignments']:
        print(f"    {a['component']:<35} → {a['pins']}")
    print(f"  T4 Pin Validity  : {r['task4']['pin_validity']:.2f}")
    print(f"  T4 Conflict Free : {r['task4']['conflict_free']:.2f}")
    print(f"  T4 Coverage      : {r['task4']['coverage']:.2f}")
    print(f"  T4 F1            : {r['task4']['f1']:.2f}")
    if r['task4']['conflicts']:
        print(f"    Conflicts : {r['task4']['conflicts']}")
    else:
        print(f"   No conflicts")
    print(f"  Feasibility      : {r['task4']['feasibility']}")

In [ ]:
# ── CELL 9: TASK 4 — FAILURE ANALYSIS ────────────────────────────────────────
t4_all    = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']
t4_failed = [r for r in connected_results if r['task4']['status'] != 'ok']
t4_low    = [r for r in connected_results if r['task4']['f1'] < 0.5]
t4_perf   = [r for r in connected_results if r['task4']['f1'] == 1.0]
t4_conf   = [r for r in connected_results if len(r['task4']['conflicts']) > 0]
t4_infeas = [r for r in connected_results
             if r['task4']['feasibility'] == 'infeasible']

print('=' * 60)
print(' Task 4 — Failure Analysis')
print('=' * 60)
print(f'  Total evaluated    : {len(connected_results)}')
print(f'  Successfully scored: {len(t4_all)}')
print(f'  Parse failed       : {len(t4_failed)}')
print()
print(f'  Perfect F1  (1.00) : {len(t4_perf)} / {len(t4_all)}')
print(f'  Low F1  (< 0.50)   : {len(t4_low)} / {len(t4_all)}')
print(f'  With conflicts     : {len(t4_conf)} / {len(t4_all)}')
print(f'  Infeasible         : {len(t4_infeas)} / {len(t4_all)}')
print('=' * 60)

# Pin validity breakdown 
invalid_pins = Counter()
for r in connected_results:
    for d in r['task4'].get('details', []):
        if not d.get('valid'):
            key = f"{d['pin_name']} ({d['pin_kind']}) → {d['pin_number']}"
            invalid_pins[key] += 1

print('\n── Top 10 Invalid Pin Assignments ──')
if invalid_pins:
    for pin, count in invalid_pins.most_common(10):
        print(f'  {pin:<45} invalid {count}x')
else:
    print('   No invalid pin assignments found')

# Conflict breakdown 
all_conflicts = []
for r in connected_results:
    for c in r['task4'].get('conflicts', []):
        all_conflicts.append(c)

print(f'\n── Pin Conflicts ({len(all_conflicts)} total) ──')
if all_conflicts:
    for c in all_conflicts[:10]:
        print(f'    {c}')
else:
    print('   No pin conflicts detected')

# Coverage breakdown
low_coverage = [r for r in connected_results
                if r['task4']['coverage'] < 1.0
                and r['task4']['status'] == 'ok']

print(f'\n── Low Coverage Projects ({len(low_coverage)}) ──')
for r in low_coverage[:5]:
    assigned = {a['component'] for a in r['task4']['assignments']}
    t1_preds = set(r['task1']['predicted_names'])
    missing  = t1_preds - assigned
    print(f"  Project {r['project_id']} [{r['complexity']}]")
    print(f"    Predicted  : {list(t1_preds)}")
    print(f"    Assigned   : {list(assigned)}")
    print(f"    Missing    : {list(missing)}")
    print(f"    Coverage   : {r['task4']['coverage']:.2f}")
    print()

# Feasibility breakdown 
feasibility_counts = Counter()
for r in connected_results:
    feasibility_counts[r['task4']['feasibility']] += 1

print('── Feasibility Distribution ──')
for status, count in feasibility_counts.most_common():
    bar = '█' * (count // max(1, len(connected_results) // 20))
    print(f'  {status:<25} : {count:>4}  {bar}')

# Complexity breakdown 
print('\n── T4 F1 by Complexity ──')
for complexity in ['Low', 'Medium', 'High', 'Unknown']:
    subset = [r for r in connected_results
              if r['complexity'] == complexity
              and r['task4']['status'] == 'ok']
    if not subset: continue
    avg_f1  = np.mean([r['task4']['f1']           for r in subset])
    avg_pv  = np.mean([r['task4']['pin_validity']  for r in subset])
    avg_cf  = np.mean([r['task4']['conflict_free'] for r in subset])
    avg_cov = np.mean([r['task4']['coverage']      for r in subset])
    print(f'  {complexity:<8} ({len(subset):>3} projects) — '
          f'F1:{avg_f1:.2f}  '
          f'PinValid:{avg_pv:.2f}  '
          f'ConflictFree:{avg_cf:.2f}  '
          f'Coverage:{avg_cov:.2f}')

# Sample low F1 cases 
if t4_low:
    print(f'\n── Sample Low-F1 T4 Projects (first 3) ──')
    for r in t4_low[:3]:
        print(f"\n  Project {r['project_id']} [{r['complexity']}] | {r['board']}")
        print(f"  Components : {r['task1']['predicted_names']}")
        print(f"  Assignments:")
        for a in r['task4']['assignments']:
            print(f"    {a['component']:<35} → {a['pins']}")
        print(f"  Pin Validity  : {r['task4']['pin_validity']:.2f}")
        print(f"  Conflict Free : {r['task4']['conflict_free']:.2f}")
        print(f"  Coverage      : {r['task4']['coverage']:.2f}")
        print(f"  F1            : {r['task4']['f1']:.2f}")
        if r['task4']['conflicts']:
            for c in r['task4']['conflicts']:
                print(f"    {c}")
        print()

# Parse failures 
if t4_failed:
    print(f'\n── Parse Failed Projects ({len(t4_failed)}) ──')
    for r in t4_failed[:5]:
        print(f"  Project {r['project_id']} — status: {r['task4']['status']}")

In [ ]:
# ── CELL 10: SAVE RESULTS ─────────────────────────────────────────────────────
SAVE_PATH = 'pipeline_t1_t2_t3_t4_final.json'

t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t4_all = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']

summary = {
    'model':           active_client.model_key,
    'total_evaluated': len(connected_results),
    'errors':          error_count,

    'task1': {
        'avg_precision': round(np.mean([r['precision'] for r in t1_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t1_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t1_all]), 4),
        'avg_latency':   round(np.mean([r['latency']   for r in t1_all]), 3),
        'perfect_f1':    len([r for r in t1_all if r['f1'] == 1.0]),
        'zero_f1':       len([r for r in t1_all if r['f1'] == 0.0]),
    },

    'task2': {
        'avg_precision': round(np.mean([r['precision'] for r in t2_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t2_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t2_all]), 4),
        'scored':        len(t2_all),
        'perfect_f1':    len([r for r in t2_all if r['f1'] == 1.0]),
        'zero_f1':       len([r for r in t2_all if r['f1'] == 0.0]),
    },

    'task3': {
        'avg_completeness': round(np.mean([r['completeness'] for r in t3_all]), 4),
        'avg_accuracy':     round(np.mean([r['accuracy']     for r in t3_all]), 4),
        'avg_f1':           round(np.mean([r['f1']           for r in t3_all]), 4),
        'avg_latency':      round(np.mean([r['latency']      for r in t3_all]), 3),
        'scored':           len(t3_all),
        'perfect_f1':       len([r for r in t3_all if r['f1'] == 1.0]),
        'zero_f1':          len([r for r in t3_all if r['f1'] == 0.0]),
        'by_complexity': {
            c: {
                'count':            len([r for r in connected_results
                                         if r['complexity'] == c
                                         and r['task3']['status'] == 'ok']),
                'avg_f1':           round(np.mean([r['task3']['f1']
                                         for r in connected_results
                                         if r['complexity'] == c
                                         and r['task3']['status'] == 'ok'] or [0]), 4),
                'avg_completeness': round(np.mean([r['task3']['completeness']
                                         for r in connected_results
                                         if r['complexity'] == c
                                         and r['task3']['status'] == 'ok'] or [0]), 4),
                'avg_accuracy':     round(np.mean([r['task3']['accuracy']
                                         for r in connected_results
                                         if r['complexity'] == c
                                         and r['task3']['status'] == 'ok'] or [0]), 4),
            }
            for c in ['Low', 'Medium', 'High', 'Unknown']
        },
    },

    'task4': {
        'avg_pin_validity':  round(np.mean([r['pin_validity']  for r in t4_all]), 4),
        'avg_conflict_free': round(np.mean([r['conflict_free'] for r in t4_all]), 4),
        'avg_coverage':      round(np.mean([r['coverage']      for r in t4_all]), 4),
        'avg_f1':            round(np.mean([r['f1']            for r in t4_all]), 4),
        'avg_latency':       round(np.mean([r['latency']       for r in t4_all]), 3),
        'scored':            len(t4_all),
        'perfect_f1':        len([r for r in t4_all if r['f1'] == 1.0]),
        'zero_f1':           len([r for r in t4_all if r['f1'] == 0.0]),
        'total_conflicts':   sum(len(r['conflicts']) for r in t4_all),
        'conflict_projects': len([r for r in connected_results
                                   if len(r['task4']['conflicts']) > 0]),
        'feasibility_dist':  dict(Counter(
                                  r['task4']['feasibility']
                                  for r in connected_results)),
        'by_complexity': {
            c: {
                'count':            len([r for r in connected_results
                                         if r['complexity'] == c
                                         and r['task4']['status'] == 'ok']),
                'avg_f1':           round(np.mean([r['task4']['f1']
                                         for r in connected_results
                                         if r['complexity'] == c
                                         and r['task4']['status'] == 'ok'] or [0]), 4),
                'avg_pin_validity': round(np.mean([r['task4']['pin_validity']
                                         for r in connected_results
                                         if r['complexity'] == c
                                         and r['task4']['status'] == 'ok'] or [0]), 4),
                'avg_coverage':     round(np.mean([r['task4']['coverage']
                                         for r in connected_results
                                         if r['complexity'] == c
                                         and r['task4']['status'] == 'ok'] or [0]), 4),
            }
            for c in ['Low', 'Medium', 'High', 'Unknown']
        },
    },

    'results': connected_results,
}

with open(SAVE_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(f' Saved → {SAVE_PATH}')
print()
print('── Final Pipeline Summary ──')
print(f'  Task 1 — Avg F1           : {summary["task1"]["avg_f1"]}')
print(f'  Task 2 — Avg F1           : {summary["task2"]["avg_f1"]}')
print(f'  Task 3 — Avg F1           : {summary["task3"]["avg_f1"]}')
print(f'  Task 3 — Avg Completeness : {summary["task3"]["avg_completeness"]}')
print(f'  Task 3 — Avg Accuracy     : {summary["task3"]["avg_accuracy"]}')
print(f'  Task 4 — Avg F1           : {summary["task4"]["avg_f1"]}')
print(f'  Task 4 — Avg Pin Validity : {summary["task4"]["avg_pin_validity"]}')
print(f'  Task 4 — Avg Coverage     : {summary["task4"]["avg_coverage"]}')
print(f'  Task 4 — Conflict Free    : {summary["task4"]["avg_conflict_free"]}')
print()
print('── Task 4 by Complexity ──')
for c, stats in summary['task4']['by_complexity'].items():
    if stats['count'] == 0:
        continue
    print(f'  {c:<8} ({stats["count"]:>3} projects) — '
          f'F1:{stats["avg_f1"]:.3f}  '
          f'PinValid:{stats["avg_pin_validity"]:.3f}  '
          f'Coverage:{stats["avg_coverage"]:.3f}')
print()
print('── Feasibility Distribution ──')
for status, count in summary['task4']['feasibility_dist'].items():
    print(f'  {status:<25} : {count}')
print(f'\n── Update Task 4 markdown to ──')
print(f"""
**Status:**  Complete

| Metric           | Score  |
|------------------|--------|
| Avg Pin Validity | {summary['task4']['avg_pin_validity']} |
| Avg Coverage     | {summary['task4']['avg_coverage']}     |
| Avg Conflict Free| {summary['task4']['avg_conflict_free']}|
| Avg F1           | {summary['task4']['avg_f1']}           |
| Projects scored  | {summary['task4']['scored']}           |
""")

---
# ✅ Task 5 Complete — Code Planning Quality

| Metric                  | Score  |
|-------------------------|--------|
| Avg Library Coverage    | 0.710  |
| Avg Setup Completeness  | 0.935  |
| Avg Loop Correctness    | 0.910  |
| Avg F1                  | 0.845  |
| Projects scored         | 10/10  |
| Parse failed            | 0      |

**Complexity breakdown:**
- Low: F1=0.73 (lib score affected by judge bug on no-library components)
- Medium: F1=0.87
- High: F1=0.95

---

In [ ]:
# ── CELL 2: TASK 5 — PROMPTS ──────────────────────────────────────────────────

SYSTEM_PROMPT_MODULE3C = """You are an Arduino Senior Developer specialising in code architecture.

Given a project intent, pin assignments, and component list, produce a complete
structured coding plan a developer can follow to write the final Arduino code.

Rules:
- Include ALL necessary libraries in the libraries list (without .h extension)
- Declare ALL global variables needed (pins, objects, state variables)
- Setup logic must initialise EVERY component
- Loop logic must implement ALL tasks and conditions from the intent
- Use actual pin numbers from the assignments provided
- Output ONLY a valid JSON object. No explanation, no markdown, no preamble.

Output format:
{
  "libraries": ["Servo"],
  "global_vars": [
    {"name": "servo",   "type": "Servo", "value": null, "description": "Servo object for lid"},
    {"name": "trigPin", "type": "int",   "value": 5,    "description": "Ultrasonic trigger pin"}
  ],
  "setup_logic": [
    "Serial.begin(9600)",
    "servo.attach(servoPin)",
    "pinMode(trigPin, OUTPUT)",
    "pinMode(echoPin, INPUT)"
  ],
  "loop_logic": [
    "Send 10us pulse on trigPin",
    "Read pulseIn duration on echoPin",
    "Calculate dist = duration * 0.034 / 2",
    "If dist < 20: servo.write(90)",
    "delay(3000)",
    "servo.write(0)"
  ],
  "notes": "Optional warnings or implementation notes"
}

--- EXAMPLES ---

Example 1 — Ultrasonic + Servo:
Components   : ["Ultrasonic Sensor", "Servo Motor"]
Board        : Arduino Uno
Pin Assignments:
  Ultrasonic Sensor (digital) → TRIG:5, ECHO:6
  Servo Motor (PWM)           → SIGNAL:9
Intent:
  Tasks      : ["detect nearby object", "open lid", "close lid after delay"]
  Conditions : ["distance < 20cm triggers lid open"]
  Thresholds : ["20cm detection distance", "3 second delay"]
Output:
{"libraries":["Servo"],"global_vars":[{"name":"servo","type":"Servo","value":null,"description":"Servo object"},{"name":"trigPin","type":"int","value":5,"description":"Ultrasonic trigger"},{"name":"echoPin","type":"int","value":6,"description":"Ultrasonic echo"},{"name":"servoPin","type":"int","value":9,"description":"Servo signal pin"},{"name":"dist","type":"long","value":null,"description":"Distance in cm"}],"setup_logic":["Serial.begin(9600)","servo.attach(servoPin)","pinMode(trigPin, OUTPUT)","pinMode(echoPin, INPUT)","servo.write(0)"],"loop_logic":["Send 10us pulse on trigPin","Read pulseIn duration on echoPin","dist = duration * 0.034 / 2","If dist < 20: servo.write(90)","delay(3000)","servo.write(0)"],"notes":"Use pulseIn() — no external library needed"}

Example 2 — Soil Moisture + Relay:
Components   : ["Soil Moisture Sensor", "Relay Module"]
Board        : Arduino Uno
Pin Assignments:
  Soil Moisture (analog) → AOUT:A0
  Relay Module (digital) → IN:7
Intent:
  Tasks      : ["monitor soil moisture", "activate pump when dry"]
  Conditions : ["moisture < 30% triggers pump on"]
  Thresholds : ["30% moisture threshold"]
Output:
{"libraries":[],"global_vars":[{"name":"moisturePin","type":"int","value":"A0","description":"Soil moisture analog pin"},{"name":"relayPin","type":"int","value":7,"description":"Relay control pin"},{"name":"moisture","type":"int","value":null,"description":"Raw moisture reading"},{"name":"threshold","type":"int","value":300,"description":"Dry soil threshold"}],"setup_logic":["pinMode(relayPin, OUTPUT)","digitalWrite(relayPin, HIGH)","Serial.begin(9600)"],"loop_logic":["moisture = analogRead(moisturePin)","If moisture < threshold: digitalWrite(relayPin, LOW)","Else: digitalWrite(relayPin, HIGH)","delay(1000)"],"notes":"Relay is active LOW — LOW turns pump ON"}

--- END EXAMPLES ---"""


USER_PROMPT_MODULE3C_TEMPLATE = """Create a structured Arduino code plan for the following project.

Project description:
{description}

Components and pin assignments:
{pin_assignments}

Project intent:
  Tasks      : {tasks}
  Conditions : {conditions}
  Thresholds : {thresholds}
  Inputs     : {inputs}
  Outputs    : {outputs}
  Summary    : {logic_summary}

Libraries available from the hardware dictionary (include only what is needed):
{libraries}

Board: {board}

Return ONLY the JSON object."""


print(' Task 5 prompts defined')

In [ ]:
# ── CELL 3: TASK 5 — AGENT FUNCTION ──────────────────────────────────────────

# ── max_tokens budget for Task 5 ─────────────────────────────────────────────
# A full plan for a 4-component project is ~800-1200 tokens.
# The old 512 limit was the single biggest cause of parse_failed results.
T5_MAX_TOKENS = 1500


def normalize_lib_name(lib: str) -> str:
    """Strip .h suffix (may already be present) before we re-add it in prompts."""
    return lib.strip().removesuffix('.h').removesuffix('.H')


def format_pin_assignments(t4_output: dict) -> str:
    """Format Task 4 pin assignments into readable string for the prompt."""
    if not t4_output or t4_output.get('status') != 'ok':
        return '  No pin assignments available — assign pins based on component type'
    lines = []
    for a in t4_output.get('assignments', []):
        comp     = a.get('component', '')
        pin_type = a.get('pin_type', '')
        pins     = a.get('pins', {})
        pin_str  = ', '.join(f'{k}:{v}' for k, v in pins.items())
        lines.append(f'  {comp} ({pin_type}) → {pin_str}')
    return '\n'.join(lines) if lines else '  No assignments available'


def format_libraries_from_t2(t2_details: list) -> str:
    """
    Format Task 2 library results for the prompt.
    FIX: strip existing .h before listing, so we never produce 'Servo.h.h'.
    """
    libs = []
    seen = set()
    for r in t2_details:
        if not isinstance(r, dict):
            continue
        lib = r.get('library')
        if not lib:
            continue
        clean = normalize_lib_name(lib)
        if clean.lower() not in seen:
            seen.add(clean.lower())
            libs.append(clean)
    if not libs:
        return '  None — components use native Arduino functions only'
    return '\n'.join(f'  - {lib}' for lib in libs)


def run_task5_agent(project: dict,
                    t3_output: dict,
                    t4_output: dict,
                    t2_details: list,
                    client) -> dict:
    """
    Task 5 Agent: LLM produces a structured code plan.
    Input  → description + intent (T3) + pin assignments (T4) + libraries (T2)
    Output → libraries, global_vars, setup_logic, loop_logic

    FIX 1: passes max_tokens=T5_MAX_TOKENS to avoid JSON truncation.
    FIX 2: description is always included so the model has context even
            when T3/T4 outputs are empty (graceful upstream fallback).
    """
    pid   = project.get('project_id')
    board = project.get('board', 'Arduino Uno')
    desc  = str(project.get('description', ''))

    pin_assignments_str = format_pin_assignments(t4_output)
    libraries_str       = format_libraries_from_t2(t2_details)

    # Graceful fallback: if T3 failed / returned empty, degrade gracefully.
    # The description is always passed so the model isn't flying blind.
    t3_ok       = isinstance(t3_output, dict) and t3_output.get('status') == 'ok'
    tasks         = t3_output.get('tasks',         []) if t3_ok else []
    conditions    = t3_output.get('conditions',    []) if t3_ok else []
    thresholds    = t3_output.get('thresholds',    []) if t3_ok else []
    inputs        = t3_output.get('inputs',        []) if t3_ok else []
    outputs       = t3_output.get('outputs',       []) if t3_ok else []
    logic_summary = t3_output.get('logic_summary', '') if t3_ok else ''

    user_prompt = USER_PROMPT_MODULE3C_TEMPLATE.format(
        description     = desc,
        pin_assignments = pin_assignments_str,
        tasks           = json.dumps(tasks),
        conditions      = json.dumps(conditions),
        thresholds      = json.dumps(thresholds),
        inputs          = json.dumps(inputs),
        outputs         = json.dumps(outputs),
        logic_summary   = logic_summary,
        libraries       = libraries_str,
        board           = board,
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_MODULE3C,
        max_tokens    = T5_MAX_TOKENS,      # ← KEY FIX: was 512 globally
    )

    REQUIRED_KEYS = ['libraries', 'global_vars', 'setup_logic', 'loop_logic']

    if not isinstance(result, dict) or not any(result.get(k) for k in REQUIRED_KEYS):
        return {
            'project_id':  pid,
            'status':      'parse_failed',
            'libraries':   [],
            'global_vars': [],
            'setup_logic': [],
            'loop_logic':  [],
            'notes':       '',
            'latency':     latency,
            'raw':         result,
        }

    # Fill missing keys
    for key in REQUIRED_KEYS:
        if key not in result:
            result[key] = []

    # Coerce non-list fields
    for key in ['libraries', 'setup_logic', 'loop_logic']:
        if not isinstance(result[key], list):
            result[key] = [str(result[key])] if result[key] else []
    if not isinstance(result.get('global_vars'), list):
        result['global_vars'] = []

    # Normalise library names (strip stray .h suffixes from model output)
    result['libraries'] = [normalize_lib_name(l) for l in result['libraries'] if l]

    return {
        'project_id':  pid,
        'status':      'ok',
        'libraries':   result['libraries'],
        'global_vars': result['global_vars'],
        'setup_logic': result['setup_logic'],
        'loop_logic':  result['loop_logic'],
        'notes':       result.get('notes', ''),
        'latency':     latency,
        'raw':         result,
    }


print(' run_task5_agent defined')
print('   Input : description + T3 intent + T4 pins + T2 libraries')
print('   Output: libraries, global_vars, setup_logic, loop_logic')
print(f'   max_tokens for T5: {T5_MAX_TOKENS}  (was 512 — this was the main failure cause)')

In [ ]:
# ── CELL 4: TASK 5 — SCORING ──────────────────────────────────────────────────

SYSTEM_PROMPT_JUDGE_T5 = """You are an expert Arduino code architect and evaluator.

Score the generated code plan on THREE dimensions (each 0.0–1.0):

LIBRARY_COVERAGE:
- Are all required libraries present?
- 1.0 = all libraries correct, 0.0 = critical libraries missing

SETUP_COMPLETENESS:
- Does setup() initialise every component (pin modes, attach, begin)?
- 1.0 = complete, 0.0 = missing critical initialisation

LOOP_CORRECTNESS:
- Does loop() implement all tasks, conditions, and thresholds from intent?
- 1.0 = fully correct logic, 0.0 = completely wrong

Rules:
- Be strict but give partial credit for partial correctness
- Minor variable name differences are acceptable
- Output ONLY a JSON object. No explanation, no markdown.
- Keep reasoning under 30 words — be concise

Output format:
{"library_coverage": 0.90, "setup_completeness": 0.85, "loop_correctness": 0.80, "reasoning": "brief explanation"}"""


USER_PROMPT_JUDGE_T5_TEMPLATE = """Evaluate this Arduino code plan.

Project description:
{description}

Required libraries (from dictionary):
{required_libs}

Pin assignments:
{pin_assignments}

Intent:
  Tasks      : {tasks}
  Conditions : {conditions}
  Thresholds : {thresholds}
  Summary    : {logic_summary}

Generated plan:
  Libraries   : {plan_libraries}
  Global vars : {global_vars}
  Setup logic : {setup_logic}
  Loop logic  : {loop_logic}
  Notes       : {notes}

Return ONLY the JSON object with library_coverage, setup_completeness, loop_correctness, reasoning."""


def score_task5_with_judge(project: dict,
                            t2_details: list,
                            t3_output:  dict,
                            t4_output:  dict,
                            t5_output:  dict,
                            client) -> dict:
    """
    LLM-as-judge scores Task 5 on three dimensions.

    FIX 1: extra 2s sleep before the judge call so the rate limiter has
           time to recover after the T5 generation call (they share quota).
    FIX 2: F1 uses arithmetic mean when any dimension is 0, instead of
           collapsing to 0.0 — prevents a single missing library from
           wiping out a plan that otherwise scored well.
    """
    if t5_output.get('status') != 'ok':
        return {
            'library_coverage':   0.0,
            'setup_completeness': 0.0,
            'loop_correctness':   0.0,
            'f1':                 0.0,
            'reasoning':          'Task 5 parse failed — skipping judge',
            'latency':            0.0,
        }

    # ── Extra delay before judge to reduce consecutive-429 rate ──────────────
    _time.sleep(2.0)

    desc          = str(project.get('description', ''))
    t3_ok         = isinstance(t3_output, dict) and t3_output.get('status') == 'ok'
    required_libs = [
        normalize_lib_name(r['library'])
        for r in t2_details
        if isinstance(r, dict) and r.get('library')
    ]

    user_prompt = USER_PROMPT_JUDGE_T5_TEMPLATE.format(
        description     = desc,
        required_libs   = json.dumps(required_libs),
        pin_assignments = format_pin_assignments(t4_output),
        tasks           = json.dumps(t3_output.get('tasks',      []) if t3_ok else []),
        conditions      = json.dumps(t3_output.get('conditions', []) if t3_ok else []),
        thresholds      = json.dumps(t3_output.get('thresholds', []) if t3_ok else []),
        logic_summary   = t3_output.get('logic_summary', '') if t3_ok else '',
        plan_libraries  = json.dumps(t5_output['libraries']),
        global_vars     = json.dumps(t5_output['global_vars']),
        setup_logic     = json.dumps(t5_output['setup_logic']),
        loop_logic      = json.dumps(t5_output['loop_logic']),
        notes           = t5_output.get('notes', ''),
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_JUDGE_T5,
        max_tokens    = 256,   # judge output is small — keep tight
    )

    if not isinstance(result, dict):
        return {
            'library_coverage':   0.0,
            'setup_completeness': 0.0,
            'loop_correctness':   0.0,
            'f1':                 0.0,
            'reasoning':          'Judge parse failed',
            'latency':            latency,
        }

    lib_cov   = max(0.0, min(1.0, float(result.get('library_coverage',   0.0))))
    setup_com = max(0.0, min(1.0, float(result.get('setup_completeness', 0.0))))
    loop_cor  = max(0.0, min(1.0, float(result.get('loop_correctness',   0.0))))

    scores = [lib_cov, setup_com, loop_cor]

    # ── FIX: harmonic mean only when all three are > 0 ───────────────────────
    # Previously: any zero dimension collapsed F1 to 0.0 even when two
    # dimensions scored 0.9. Now we always fall back to arithmetic mean.
    if all(s > 0 for s in scores):
        f1 = 3 / sum(1 / s for s in scores)   # harmonic mean
    else:
        f1 = float(np.mean(scores))            # arithmetic mean fallback

    return {
        'library_coverage':   round(lib_cov,   4),
        'setup_completeness': round(setup_com, 4),
        'loop_correctness':   round(loop_cor,  4),
        'f1':                 round(f1,        4),
        'reasoning':          result.get('reasoning', ''),
        'latency':            latency,
    }


print(' Task 5 scoring functions defined')
print('   Metrics: library_coverage + setup_completeness + loop_correctness')
print('   Judge max_tokens: 256  |  Extra pre-judge delay: 2.0s')
print('   F1: harmonic mean when all > 0, arithmetic mean when any = 0')

In [ ]:
# ── CELL 5: TASK 5 — SANITY CHECK (3 PROJECTS) ───────────────────────────────
print('── Task 5 Sanity Check (3 projects) ──\n')

for proj in projects[:3]:
    pid   = proj['project_id']
    board = proj.get('board', 'Arduino Uno')

    t1 = run_task1_agent(proj, active_client, use_rag=True)
    predicted_names = t1['predicted_names']

    t2_details = run_task2_agent(
        predicted_names, proj.get('description', ''),
        active_client, board=board
    )

    t3 = run_task3_agent(proj, predicted_names, active_client)
    t4 = run_task4_agent(proj, predicted_names, active_client, df_comp)
    t5 = run_task5_agent(proj, t3, t4, t2_details, active_client)

    score = score_task5_with_judge(proj, t2_details, t3, t4, t5, active_client)

    print(f'Project {pid} [{proj.get("complexity")}] | Board: {board}')
    print(f'  Status       : {t5["status"]}')
    print(f'  Components   : {predicted_names}')
    print(f'  Libraries    : {t5["libraries"]}')
    print(f'  Global vars  : {len(t5["global_vars"])} declared')
    for v in t5['global_vars']:
        print(f'    {v.get("type",""):<8} {v.get("name",""):<15} '
              f'= {str(v.get("value","")):<8} // {v.get("description","")}')
    print(f'  Setup steps  : {len(t5["setup_logic"])}')
    for s in t5['setup_logic']:
        print(f'    → {s}')
    print(f'  Loop steps   : {len(t5["loop_logic"])}')
    for l in t5['loop_logic']:
        print(f'    → {l}')
    if t5['notes']:
        print(f'  Notes        : {t5["notes"]}')
    print()
    print(f'  ── Judge Scores ──')
    print(f'  Lib Coverage  : {score["library_coverage"]:.2f}')
    print(f'  Setup Complete: {score["setup_completeness"]:.2f}')
    print(f'  Loop Correct  : {score["loop_correctness"]:.2f}')
    print(f'  F1            : {score["f1"]:.2f}')
    print(f'  Reasoning     : {score["reasoning"]}')
    print(f'  T5 Latency    : {t5["latency"]:.2f}s  | Judge: {score["latency"]:.2f}s')
    print()
    print('-' * 65)
    print()

In [ ]:
# ── CELL 6: FULL CONNECTED RUN — TASKS 1 + 2 + 3 + 4 + 5 ────────────────────
SAVE_PATH         = 'connected_results_t1_t2_t3_t4_t5.json'
connected_results = []
error_count       = 0

_EMPTY_T3 = {
    'project_id': None, 'status': 'error',
    'tasks': [], 'conditions': [], 'thresholds': [],
    'inputs': [], 'outputs': [], 'logic_summary': '', 'latency': 0.0, 'raw': {},
}
_EMPTY_T4 = {
    'project_id': None, 'status': 'error',
    'assignments': [], 'conflicts': [], 'feasibility': 'unknown',
    'spec_list': [], 'latency': 0.0,
}
_EMPTY_T5 = {
    'project_id': None, 'status': 'error',
    'libraries': [], 'global_vars': [], 'setup_logic': [], 'loop_logic': [],
    'notes': '', 'latency': 0.0,
}
_ZERO_T5_SCORE = {
    'library_coverage': 0.0, 'setup_completeness': 0.0,
    'loop_correctness': 0.0, 'f1': 0.0,
    'reasoning': 'upstream error', 'latency': 0.0,
}

for proj in tqdm(projects[:10], desc='Tasks 1→5'):
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')
    gt_libs = set(proj.get('ground_truth_libraries', []))

    # ── Task 1 ───────────────────────────────────────────────────────────────
    try:
        t1_output = run_task1_agent(proj, active_client, use_rag=True)
    except Exception as e:
        error_count += 1
        tqdm.write(f'  ERROR T1 project {pid}: {e}')
        continue

    gt_names        = get_gt_names(proj)
    predicted_names = t1_output['predicted_names']
    if not gt_names:
        continue

    t1_scores    = match_predicted_to_gt(predicted_names, gt_names)
    t1_tp        = t1_scores['tp']
    t1_fp        = t1_scores['fp']
    t1_fn        = t1_scores['fn']
    t1_precision = t1_tp / (t1_tp + t1_fp) if (t1_tp + t1_fp) > 0 else 0
    t1_recall    = t1_tp / (t1_tp + t1_fn) if (t1_tp + t1_fn) > 0 else 0
    t1_f1        = (2 * t1_precision * t1_recall / (t1_precision + t1_recall)
                    if (t1_precision + t1_recall) > 0 else 0)

    # ── Task 2 ───────────────────────────────────────────────────────────────
    try:
        t2_details = run_task2_agent(
            predicted_names, desc, active_client, board=board
        )
    except Exception as e:
        tqdm.write(f'  ERROR T2 project {pid}: {e}')
        t2_details = [
            {'component': n, 'library': None, 'status': 'error',
             'confidence': 'low', 'reasoning': str(e), 'latency': 0.0}
            for n in predicted_names
        ]

    retrieved_libs = {r['library'] for r in t2_details if r.get('library')}
    gt_canon       = canonicalize_set(gt_libs)
    ret_canon      = canonicalize_set(retrieved_libs)
    gt_canon       = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon      = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if gt_canon:
        t2_tp        = match_with_equivalence(gt_canon, ret_canon)
        t2_precision = t2_tp / len(ret_canon) if ret_canon else 0
        t2_recall    = t2_tp / len(gt_canon)  if gt_canon  else 0
        t2_f1        = (2 * t2_precision * t2_recall / (t2_precision + t2_recall)
                        if (t2_precision + t2_recall) > 0 else 0)
    else:
        t2_tp = t2_precision = t2_recall = t2_f1 = None

    # ── Task 3 ───────────────────────────────────────────────────────────────
    try:
        t3_output = run_task3_agent(proj, predicted_names, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T3 project {pid}: {e}')
        t3_output = {**_EMPTY_T3, 'project_id': pid}

    try:
        t3_score = score_intent_with_judge(proj, t3_output, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T3 judge project {pid}: {e}')
        t3_score = {'completeness': 0.0, 'accuracy': 0.0,
                    'f1': 0.0, 'reasoning': str(e), 'latency': 0.0}

    # ── Task 4 ───────────────────────────────────────────────────────────────
    try:
        t4_output = run_task4_agent(proj, predicted_names, active_client, df_comp)
    except Exception as e:
        tqdm.write(f'  ERROR T4 project {pid}: {e}')
        t4_output = {**_EMPTY_T4, 'project_id': pid}

    try:
        t4_score = score_task4(t4_output, board)
    except Exception as e:
        tqdm.write(f'  ERROR T4 score project {pid}: {e}')
        t4_score = {'pin_validity': 0.0, 'conflict_free': 0.0,
                    'coverage': 0.0, 'f1': 0.0, 'conflicts': [], 'details': []}

    # ── Task 5 ───────────────────────────────────────────────────────────────
    try:
        t5_output = run_task5_agent(
            proj, t3_output, t4_output, t2_details, active_client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T5 project {pid}: {e}')
        t5_output = {**_EMPTY_T5, 'project_id': pid}

    try:
        t5_score = score_task5_with_judge(
            proj, t2_details, t3_output, t4_output, t5_output, active_client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T5 judge project {pid}: {e}')
        t5_score = {**_ZERO_T5_SCORE, 'reasoning': str(e)}

    connected_results.append({
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,

        'task1': {
            'predicted_names': predicted_names,
            'gt_names':        list(gt_names),
            'matched':         list(t1_scores['matched_gt']),
            'precision':       round(t1_precision, 4),
            'recall':          round(t1_recall,    4),
            'f1':              round(t1_f1,        4),
            'latency':         t1_output['latency'],
        },

        'task2': {
            'gt_libs':        list(gt_canon)  if gt_canon  else [],
            'retrieved_libs': list(ret_canon) if ret_canon else [],
            'tp':             t2_tp,
            'precision':      round(t2_precision, 4) if t2_precision is not None else None,
            'recall':         round(t2_recall,    4) if t2_recall    is not None else None,
            'f1':             round(t2_f1,        4) if t2_f1        is not None else None,
            'details':        t2_details,
        },

        'task3': {
            'tasks':         t3_output.get('tasks', []),
            'conditions':    t3_output.get('conditions', []),
            'thresholds':    t3_output.get('thresholds', []),
            'inputs':        t3_output.get('inputs', []),
            'outputs':       t3_output.get('outputs', []),
            'logic_summary': t3_output.get('logic_summary', ''),
            'completeness':  t3_score['completeness'],
            'accuracy':      t3_score['accuracy'],
            'f1':            t3_score['f1'],
            'reasoning':     t3_score['reasoning'],
            'status':        t3_output.get('status', 'error'),
            'latency':       t3_output.get('latency', 0.0),
            'judge_latency': t3_score['latency'],
        },

        'task4': {
            'assignments':   t4_output.get('assignments', []),
            'conflicts':     t4_score['conflicts'],
            'feasibility':   t4_output.get('feasibility', 'unknown'),
            'pin_validity':  t4_score['pin_validity'],
            'conflict_free': t4_score['conflict_free'],
            'coverage':      t4_score['coverage'],
            'f1':            t4_score['f1'],
            'details':       t4_score['details'],
            'status':        t4_output.get('status', 'error'),
            'latency':       t4_output.get('latency', 0.0),
        },

        'task5': {
            'libraries':          t5_output['libraries'],
            'global_vars':        t5_output['global_vars'],
            'setup_logic':        t5_output['setup_logic'],
            'loop_logic':         t5_output['loop_logic'],
            'notes':              t5_output.get('notes', ''),
            'library_coverage':   t5_score['library_coverage'],
            'setup_completeness': t5_score['setup_completeness'],
            'loop_correctness':   t5_score['loop_correctness'],
            'f1':                 t5_score['f1'],
            'reasoning':          t5_score['reasoning'],
            'status':             t5_output.get('status', 'error'),
            'latency':            t5_output.get('latency', 0.0),
            'judge_latency':      t5_score['latency'],
        },
    })

    # Incremental save every 10 projects
    if len(connected_results) % 10 == 0:
        with open(SAVE_PATH, 'w') as f:
            json.dump(connected_results, f, indent=2)
        tqdm.write(f'  💾 Saved {len(connected_results)} results...')

with open(SAVE_PATH, 'w') as f:
    json.dump(connected_results, f, indent=2)

print(f' Done — saved to {SAVE_PATH}')
print(f'   Evaluated: {len(connected_results)} | Errors: {error_count}')

In [ ]:
# ── CELL 7: PRINT RESULTS ─────────────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t4_all = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']
t5_all = [r['task5'] for r in connected_results if r['task5']['status'] == 'ok']

print('=' * 65)
print(' Connected Pipeline — Tasks 1 + 2 + 3 + 4 + 5')
print('=' * 65)
print(f'  Model              : {active_client.model_key}')
print(f'  Projects evaluated : {len(connected_results)}')
print(f'  Errors             : {error_count}')
print()

print(f'  Task 1 — Component Extraction')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t1_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t1_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t1_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]   for r in t1_all]):.2f}s')
print()

print(f'  Task 2 — Library Retrieval')
if t2_all:
    print(f'    Avg Precision    : {np.mean([r["precision"] for r in t2_all]):.3f}')
    print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t2_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]        for r in t2_all]):.3f}')
print(f'    Scored           : {len(t2_all)} / {len(connected_results)}')
print()

print(f'  Task 3 — Intent Understanding')
if t3_all:
    print(f'    Avg Completeness : {np.mean([r["completeness"] for r in t3_all]):.3f}')
    print(f'    Avg Accuracy     : {np.mean([r["accuracy"]     for r in t3_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]           for r in t3_all]):.3f}')
    print(f'    Avg Latency      : {np.mean([r["latency"]      for r in t3_all]):.2f}s')
print(f'    Scored           : {len(t3_all)} / {len(connected_results)}')
print()

print(f'  Task 4 — Hardware Pin Mapping')
if t4_all:
    print(f'    Avg Pin Validity : {np.mean([r["pin_validity"]  for r in t4_all]):.3f}')
    print(f'    Avg Conflict Free: {np.mean([r["conflict_free"] for r in t4_all]):.3f}')
    print(f'    Avg Coverage     : {np.mean([r["coverage"]      for r in t4_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]            for r in t4_all]):.3f}')
print(f'    Scored           : {len(t4_all)} / {len(connected_results)}')
print()

print(f'  Task 5 — Code Planning')
if t5_all:
    print(f'    Avg Lib Coverage : {np.mean([r["library_coverage"]   for r in t5_all]):.3f}')
    print(f'    Avg Setup Comp   : {np.mean([r["setup_completeness"] for r in t5_all]):.3f}')
    print(f'    Avg Loop Correct : {np.mean([r["loop_correctness"]   for r in t5_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]                 for r in t5_all]):.3f}')
    print(f'    Avg Latency      : {np.mean([r["latency"]            for r in t5_all]):.2f}s')
    t5_failed_ct = sum(1 for r in connected_results if r['task5']['status'] != 'ok')
    print(f'    Parse failed     : {t5_failed_ct}')
print(f'    Scored           : {len(t5_all)} / {len(connected_results)}')
print('=' * 65)

# ── T5 score distributions ────────────────────────────────────────────────────
if t5_all:
    print('\n── Task 5 Score Distributions ──')
    for label, values in [
        ('Lib Coverage',  [r['library_coverage']   for r in t5_all]),
        ('Setup Complete',[r['setup_completeness'] for r in t5_all]),
        ('Loop Correct',  [r['loop_correctness']   for r in t5_all]),
        ('F1',            [r['f1']                 for r in t5_all]),
    ]:
        buckets = {'1.00': 0, '0.75-0.99': 0, '0.50-0.74': 0, '0.25-0.49': 0, '0.00-0.24': 0}
        for v in values:
            if   v >= 1.0:  buckets['1.00']      += 1
            elif v >= 0.75: buckets['0.75-0.99'] += 1
            elif v >= 0.50: buckets['0.50-0.74'] += 1
            elif v >= 0.25: buckets['0.25-0.49'] += 1
            else:           buckets['0.00-0.24'] += 1
        print(f'\n  {label}:')
        for bucket, count in buckets.items():
            bar = '█' * (count * 20 // max(len(t5_all), 1))
            print(f'    {bucket} : {count:>4}  {bar}')

# ── T5 by complexity ──────────────────────────────────────────────────────────
print('\n── Task 5 F1 by Complexity ──')
for complexity in ['Low', 'Medium', 'High', 'Unknown']:
    subset = [r for r in connected_results
              if r['complexity'] == complexity and r['task5']['status'] == 'ok']
    if not subset:
        continue
    print(f'  {complexity:<8} ({len(subset):>3}) — '
          f'F1:{np.mean([r["task5"]["f1"] for r in subset]):.2f}  '
          f'Lib:{np.mean([r["task5"]["library_coverage"] for r in subset]):.2f}  '
          f'Setup:{np.mean([r["task5"]["setup_completeness"] for r in subset]):.2f}  '
          f'Loop:{np.mean([r["task5"]["loop_correctness"] for r in subset]):.2f}')

# ── Per project sample ────────────────────────────────────────────────────────
print('\n── Per Project Sample (first 3) ──')
for r in connected_results[:3]:
    print(f"\n  Project {r['project_id']} [{r['complexity']}] | {r['board']}")
    print(f"  T1 F1 : {r['task1']['f1']:.2f}  "
          f"T2 F1 : {r['task2']['f1']}  "
          f"T3 F1 : {r['task3']['f1']:.2f}  "
          f"T4 F1 : {r['task4']['f1']:.2f}  "
          f"T5 F1 : {r['task5']['f1']:.2f}")
    print(f"  T5 Libraries : {r['task5']['libraries']}")
    print(f"  T5 Setup     : {len(r['task5']['setup_logic'])} steps")
    print(f"  T5 Loop      : {len(r['task5']['loop_logic'])} steps")
    print(f"  T5 Reasoning : {r['task5']['reasoning']}")

In [ ]:
# ── CELL 8: TASK 5 — FAILURE ANALYSIS ────────────────────────────────────────
t5_all    = [r for r in connected_results if r['task5']['status'] == 'ok']
t5_failed = [r for r in connected_results if r['task5']['status'] != 'ok']
t5_low    = [r for r in t5_all if r['task5']['f1'] < 0.5]
t5_perf   = [r for r in t5_all if r['task5']['f1'] == 1.0]

print('=' * 60)
print(' Task 5 — Failure Analysis')
print('=' * 60)
print(f'  Total evaluated    : {len(connected_results)}')
print(f'  Successfully scored: {len(t5_all)}')
print(f'  Parse failed       : {len(t5_failed)}')
print(f'  Perfect F1  (1.00) : {len(t5_perf)}')
print(f'  Low F1  (< 0.50)   : {len(t5_low)}')
print('=' * 60)

if t5_all:
    lib_scores   = [r['task5']['library_coverage']   for r in t5_all]
    setup_scores = [r['task5']['setup_completeness'] for r in t5_all]
    loop_scores  = [r['task5']['loop_correctness']   for r in t5_all]

    print('\n── Score Breakdown ──')
    for label, vals in [('Lib Coverage ', lib_scores),
                        ('Setup Complete', setup_scores),
                        ('Loop Correct  ', loop_scores)]:
        print(f'  {label} — '
              f'min:{min(vals):.2f}  max:{max(vals):.2f}  '
              f'avg:{np.mean(vals):.2f}  std:{np.std(vals):.2f}')

    weakest = min(
        [('Lib Coverage',   np.mean(lib_scores)),
         ('Setup Complete', np.mean(setup_scores)),
         ('Loop Correct',   np.mean(loop_scores))],
        key=lambda x: x[1]
    )
    print(f'\n  ⚠ Weakest dimension: {weakest[0]} ({weakest[1]:.3f})')

# ── Missing library analysis ──────────────────────────────────────────────────
missing_libs = Counter()
for r in t5_all:
    t2_libs = {normalize_lib_name(d['library'])
               for d in r['task2']['details']
               if isinstance(d, dict) and d.get('library')}
    t5_libs = {normalize_lib_name(l)
               for l in r['task5']['libraries'] if l}
    for lib in t2_libs - t5_libs:
        missing_libs[lib] += 1

print('\n── Top 10 Libraries Missing from Plan (T5 vs T2) ──')
if missing_libs:
    for lib, count in missing_libs.most_common(10):
        print(f'  {lib:<40} missing {count}x')
else:
    print('  No systematic library gaps')

# ── Empty field counts ────────────────────────────────────────────────────────
print('\n── Empty Fields ──')
print(f'  No libraries   : {sum(1 for r in t5_all if not r["task5"]["libraries"])}')
print(f'  No setup steps : {sum(1 for r in t5_all if not r["task5"]["setup_logic"])}')
print(f'  No loop steps  : {sum(1 for r in t5_all if not r["task5"]["loop_logic"])}')
print(f'  No global vars : {sum(1 for r in t5_all if not r["task5"]["global_vars"])}')

# ── Low F1 sample ─────────────────────────────────────────────────────────────
if t5_low:
    print(f'\n── Sample Low-F1 Projects (first 3) ──')
    for r in t5_low[:3]:
        desc = next((p.get('description', '')
                     for p in projects if p['project_id'] == r['project_id']), '')
        print(f"\n  Project {r['project_id']} [{r['complexity']}]")
        print(f"  Desc      : {desc[:100]}")
        print(f"  Libraries : {r['task5']['libraries']}")
        print(f"  Lib Cov   : {r['task5']['library_coverage']:.2f}  "
              f"Setup: {r['task5']['setup_completeness']:.2f}  "
              f"Loop: {r['task5']['loop_correctness']:.2f}  "
              f"F1: {r['task5']['f1']:.2f}")
        print(f"  Reasoning : {r['task5']['reasoning']}")

# ── Parse failures ────────────────────────────────────────────────────────────
if t5_failed:
    print(f'\n── Parse Failed Projects ({len(t5_failed)}) ──')
    for r in t5_failed[:5]:
        print(f"  Project {r['project_id']} — status: {r['task5']['status']}")

In [ ]:
# ── CELL 9: SAVE RESULTS ──────────────────────────────────────────────────────
SAVE_PATH = 'pipeline_t1_t2_t3_t4_t5_final.json'

t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t4_all = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']
t5_all = [r['task5'] for r in connected_results if r['task5']['status'] == 'ok']

def _by_complexity(results, task_key, metrics):
    out = {}
    for c in ['Low', 'Medium', 'High', 'Unknown']:
        subset = [r[task_key] for r in results
                  if r['complexity'] == c and r[task_key].get('status') == 'ok']
        if not subset:
            out[c] = {'count': 0}
            continue
        out[c] = {'count': len(subset)}
        for m in metrics:
            out[c][f'avg_{m}'] = round(float(np.mean([s[m] for s in subset])), 4)
    return out

summary = {
    'model':           active_client.model_key,
    'total_evaluated': len(connected_results),
    'errors':          error_count,

    'task1': {
        'avg_precision': round(np.mean([r['precision'] for r in t1_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t1_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t1_all]), 4),
        'avg_latency':   round(np.mean([r['latency']   for r in t1_all]), 3),
        'perfect_f1':    sum(1 for r in t1_all if r['f1'] == 1.0),
        'zero_f1':       sum(1 for r in t1_all if r['f1'] == 0.0),
    },

    'task2': {
        'avg_precision': round(np.mean([r['precision'] for r in t2_all]), 4) if t2_all else None,
        'avg_recall':    round(np.mean([r['recall']    for r in t2_all]), 4) if t2_all else None,
        'avg_f1':        round(np.mean([r['f1']        for r in t2_all]), 4) if t2_all else None,
        'scored':        len(t2_all),
        'perfect_f1':    sum(1 for r in t2_all if r['f1'] == 1.0),
        'zero_f1':       sum(1 for r in t2_all if r['f1'] == 0.0),
    },

    'task3': {
        'avg_completeness': round(np.mean([r['completeness'] for r in t3_all]), 4) if t3_all else None,
        'avg_accuracy':     round(np.mean([r['accuracy']     for r in t3_all]), 4) if t3_all else None,
        'avg_f1':           round(np.mean([r['f1']           for r in t3_all]), 4) if t3_all else None,
        'scored':           len(t3_all),
        'by_complexity':    _by_complexity(connected_results, 'task3',
                                           ['completeness', 'accuracy', 'f1']),
    },

    'task4': {
        'avg_pin_validity':  round(np.mean([r['pin_validity']  for r in t4_all]), 4) if t4_all else None,
        'avg_conflict_free': round(np.mean([r['conflict_free'] for r in t4_all]), 4) if t4_all else None,
        'avg_coverage':      round(np.mean([r['coverage']      for r in t4_all]), 4) if t4_all else None,
        'avg_f1':            round(np.mean([r['f1']            for r in t4_all]), 4) if t4_all else None,
        'scored':            len(t4_all),
        'total_conflicts':   sum(len(r['conflicts']) for r in t4_all),
        'feasibility_dist':  dict(Counter(r['task4']['feasibility'] for r in connected_results)),
        'by_complexity':     _by_complexity(connected_results, 'task4',
                                            ['pin_validity', 'conflict_free', 'coverage', 'f1']),
    },

    'task5': {
        'avg_library_coverage':   round(np.mean([r['library_coverage']   for r in t5_all]), 4) if t5_all else None,
        'avg_setup_completeness': round(np.mean([r['setup_completeness'] for r in t5_all]), 4) if t5_all else None,
        'avg_loop_correctness':   round(np.mean([r['loop_correctness']   for r in t5_all]), 4) if t5_all else None,
        'avg_f1':                 round(np.mean([r['f1']                 for r in t5_all]), 4) if t5_all else None,
        'avg_latency':            round(np.mean([r['latency']            for r in t5_all]), 3) if t5_all else None,
        'scored':                 len(t5_all),
        'parse_failed':           sum(1 for r in connected_results if r['task5']['status'] != 'ok'),
        'perfect_f1':             sum(1 for r in t5_all if r['f1'] == 1.0),
        'zero_f1':                sum(1 for r in t5_all if r['f1'] == 0.0),
        'by_complexity':          _by_complexity(connected_results, 'task5',
                                                 ['library_coverage', 'setup_completeness',
                                                  'loop_correctness', 'f1']),
    },

    'results': connected_results,
}

with open(SAVE_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(f' Saved → {SAVE_PATH}')
print()
print('── Final Pipeline Summary ──')
for task, key, metrics in [
    ('Task 1', 'task1', [('Avg F1', 'avg_f1')]),
    ('Task 2', 'task2', [('Avg F1', 'avg_f1')]),
    ('Task 3', 'task3', [('Avg F1', 'avg_f1'), ('Completeness', 'avg_completeness'), ('Accuracy', 'avg_accuracy')]),
    ('Task 4', 'task4', [('Avg F1', 'avg_f1'), ('Pin Validity', 'avg_pin_validity'), ('Coverage', 'avg_coverage')]),
    ('Task 5', 'task5', [('Avg F1', 'avg_f1'), ('Lib Cov', 'avg_library_coverage'),
                          ('Setup', 'avg_setup_completeness'), ('Loop', 'avg_loop_correctness')]),
]:
    vals = '  '.join(f'{lbl}: {summary[key].get(k)}' for lbl, k in metrics)
    print(f'  {task} — {vals}')

print()
print('── Task 5 by Complexity ──')
for c, stats in summary['task5']['by_complexity'].items():
    if not stats.get('count'):
        continue
    print(f'  {c:<8} ({stats["count"]:>3}) — '
          f'F1:{stats.get("avg_f1", 0):.3f}  '
          f'Lib:{stats.get("avg_library_coverage", 0):.3f}  '
          f'Setup:{stats.get("avg_setup_completeness", 0):.3f}  '
          f'Loop:{stats.get("avg_loop_correctness", 0):.3f}')


---
# 🔧 Task 6: Code Synthesis
> **Measures:** Whether the LLM can generate a complete, compilable `.ino` Arduino sketch
> from the structured plan produced by Task 5.

### Pipeline Flow
```
T1 components + T2 libraries + T3 intent + T4 pins + T5 plan
        ↓
  [Task 6 Agent] LLM → generates .ino code
        ↓
  Write to t6_output/sketch_<pid>/sketch_<pid>.ino
        ↓
  arduino-cli compile --fqbn <board_fqbn>
        ↓
  ┌─ libraries installed → real compilation → pass/fail
  └─ library missing     → record 'library_missing', skip
        ↓
  [LLM Judge] → logic_correctness (0–5) + code_quality (0–5)
```

**Metrics:**
- `component_correct`    — correct sensors/actuators present in code (binary)
- `library_correct`      — required #include statements present (precision/recall)
- `pin_valid`            — pins match T4 assignments (%)
- `compilation_success`  — compiles with Arduino CLI (binary) ← objective ground truth
- `logic_correctness`    — LLM judge rating 0–5
- `code_quality`         — LLM judge rating 0–5

**Results (llama3_3 — 10 projects):**

| Complexity | Compiled | Logic | Quality | Lib F1 |
|------------|----------|-------|---------|--------|
| Low (3)    | 3/3      | 4.3/5 | 4.3/5   | 1.00   |
| Medium (5) | 2/5      | 4.4/5 | 4.2/5   | 0.89   |
| High (2)   | 0/2      | 4.0/5 | 4.0/5   | 0.90   |
| **Total**  | **5/10** | **4.2/5** | **4.1/5** | **0.93** |

**Key finding:** Compilation failures are caused by library API hallucinations,
not logic errors — Logic scores remain high (4.0+) even for non-compiling sketches.

**Status:** ✅ Complete


In [ ]:
# ── CELL 1: TASK 6 — SETUP (fully automatic) ─────────────────────────────────
import subprocess
import shutil
from pathlib import Path

T6_OUTPUT_DIR = Path('t6_output')
T6_OUTPUT_DIR.mkdir(exist_ok=True)

ARDUINO_CLI = 'arduino-cli'

FQBN_MAP = {
    'Arduino Uno':                'arduino:avr:uno',
    'Arduino Uno R4 Wifi Module': 'arduino:renesas_uno:unor4wifi',
    'Arduino Mega':               'arduino:avr:mega',
}

CORE_BUNDLED = {
    'arduino:renesas_uno:unor4wifi': {
        'wifi', 'wifis3', 'arduinoledmatrix', 'arduino_led_matrix',
        'arduinographics', 'arduinoble',
    },
    'arduino:avr:uno':  set(),
    'arduino:avr:mega': set(),
}

# Libraries built into every Arduino core — never search or install these
ALWAYS_AVAILABLE = {
    'spi', 'wire', 'arduino', 'eeprom', 'softwareserial',
    'wifis3', 'wifi', 'serial', 'math', 'string',
}

NO_LIBRARY_NEEDED = {'', 'none', 'no_library_needed', 'native'}


def normalise_lib(lib: str) -> str:
    return re.sub(r'[\s_.\-]', '', lib.lower().removesuffix('.h'))


def get_installed_libs() -> dict:
    """
    Query arduino-cli for all installed libraries.
    Handles arduino-cli v1.4.1 JSON format.
    Returns dict: normalised_name → exact_install_name
    """
    result = subprocess.run(
        [ARDUINO_CLI, 'lib', 'list', '--format', 'json'],
        capture_output=True, text=True
    )
    installed = {}
    if result.returncode != 0:
        print(f'  arduino-cli lib list failed: {result.stderr[:100]}')
        return installed

    try:
        data = json.loads(result.stdout)

        # v1.4.1 format: {"installed_libraries": [{"library": {"name": ...}}]}
        # old format   : [{"library": {"name": ...}}]
        # fallback     : {"libraries": [...]}
        if isinstance(data, dict):
            libs = (
                data.get('installed_libraries') or
                data.get('libraries')           or
                []
            )
        elif isinstance(data, list):
            libs = data
        else:
            libs = []

        for entry in libs:
            lib_info = entry.get('library', entry)
            name = lib_info.get('name', '')
            if name:
                installed[normalise_lib(name)] = name

    except Exception as e:
        print(f'  JSON parse error: {e}')
        print(f'   Raw: {result.stdout[:300]}')

    return installed


def search_lib(lib_name: str) -> str | None:
    """
    Search arduino-cli registry for a library.
    Handles v1.4.1 format where results are under 'libraries' key.
    Tries multiple name variations automatically.
    """
    variations = [
        lib_name,
        lib_name.replace('_', ' '),
        lib_name.replace('_', '-'),
        lib_name.capitalize(),
        lib_name.upper(),
    ]

    for variant in variations:
        result = subprocess.run(
            [ARDUINO_CLI, 'lib', 'search', variant, '--format', 'json'],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            continue
        try:
            data = json.loads(result.stdout)

            if isinstance(data, dict):
                libs = (
                    data.get('libraries') or
                    data.get('installed_libraries') or
                    []
                )
            elif isinstance(data, list):
                libs = data
            else:
                continue

            if not libs:
                continue

            # Prefer exact normalised match first
            target = normalise_lib(lib_name)
            for lib in libs:
                if normalise_lib(lib.get('name', '')) == target:
                    return lib['name']

            # Fall back to first result
            return libs[0].get('name')

        except Exception:
            continue

    return None


def smart_install_lib(lib: str) -> tuple[bool, str]:
    """
    Intelligently install a library:
    1. Check if already installed
    2. Search registry for correct name
    3. Install correct name
    Returns (success, actual_name_used)
    """
    installed = get_installed_libs()
    n = normalise_lib(lib)

    # Already installed
    if n in installed:
        return True, installed[n]

    # Search registry for correct name
    print(f'     Searching registry for: {lib}')
    correct_name = search_lib(lib)
    print(f'     Found: {correct_name}')

    if not correct_name:
        print(f'      Not found in registry: {lib}')
        return False, lib

    # Install it
    result = subprocess.run(
        [ARDUINO_CLI, 'lib', 'install', correct_name],
        capture_output=True, text=True, timeout=60
    )
    if result.returncode == 0:
        print(f'     Installed: {correct_name}')
        return True, correct_name
    else:
        print(f'      Install failed: {result.stderr.strip()[:100]}')
        return False, lib


def is_lib_available(lib: str, fqbn: str) -> bool:
    n = normalise_lib(lib)
    if n in NO_LIBRARY_NEEDED:              return True
    if n in ALWAYS_AVAILABLE:               return True  
    if n in CORE_BUNDLED.get(fqbn, set()): return True
    installed = get_installed_libs()
    return n in installed


# ── Build installed libs cache on startup ─────────────────────────────────────
INSTALLED_CACHE = get_installed_libs()
print(f' arduino-cli found')
print(f' Installed libs detected: {len(INSTALLED_CACHE)}')
print(f'   {list(INSTALLED_CACHE.values())}')
print(f' t6_output dir: {T6_OUTPUT_DIR.resolve()}')

In [ ]:
# ── CELL 2: TASK 6 — PROMPTS ──────────────────────────────────────────────────

SYSTEM_PROMPT_T6 = """You are an expert Arduino firmware engineer.

Given a structured code plan, generate a complete, compilable Arduino .ino sketch.

Rules:
- Output ONLY the raw .ino code — no markdown fences, no explanation, no preamble
- First lines must be #include statements for all required libraries
- Use exact pin numbers from the plan
- Implement setup() exactly as described in setup_logic
- Implement loop() exactly as described in loop_logic
- Declare all global variables listed in global_vars
- Follow Arduino conventions: pinMode in setup(), no blocking delays > 100ms in ISRs
- Code must compile for the specified board
- Add brief inline comments for clarity
- For Arduino UNO R4 WiFi: always use #include <Arduino_LED_Matrix.h> not arduino_led_matrix.h
- For Arduino UNO R4 WiFi: always use #include <WiFiS3.h> for WiFi, not WiFi.h
- For Arduino UNO R4 WiFi: always use #include <BlynkSimpleWifi.h> not BlynkSimpleEsp8266.h
- If using Blynk library, always add these placeholder defines at the top before any #include:
  #define BLYNK_TEMPLATE_ID "YOUR_TEMPLATE_ID"
  #define BLYNK_TEMPLATE_NAME "YOUR_DEVICE_NAME"  
  #define BLYNK_AUTH_TOKEN "YOUR_AUTH_TOKEN"
- For Arduino_LED_Matrix library: class name is ArduinoLEDMatrix, not LED_Matrix or Arduino_LED_Matrix
  Usage: ArduinoLEDMatrix matrix; matrix.begin(); matrix.renderBitmap(frame, 8, 12);
- For TFT_eSPI library: class name is TFT_eSPI, not TFT
  Usage: TFT_eSPI tft = TFT_eSPI(); tft.init();
- For MPU6050 library: constructor requires Wire object
  Usage: MPU6050 mpu(Wire); void setup() { Wire.begin(); mpu.begin(); }
- VirtualWire is deprecated — use RadioHead instead
  Usage: #include <RH_ASK.h> RH_ASK driver; driver.init();
- If generated code exceeds board flash limit, reduce by:
  removing Serial.print debug statements, simplifying string literals,
  using F() macro for strings: Serial.println(F("text"));

--- EXAMPLE ---

Plan:
  Board      : Arduino Uno
  Libraries  : [Servo]
  Global vars: servo(Servo), trigPin(int)=5, echoPin(int)=6, servoPin(int)=9, dist(long)
  Setup logic: Serial.begin(9600), servo.attach(servoPin), pinMode(trigPin OUTPUT), pinMode(echoPin INPUT), servo.write(0)
  Loop logic : Send 10us pulse on trigPin, Read pulseIn on echoPin, dist = duration*0.034/2, If dist<20 servo.write(90), delay(3000), servo.write(0)

Output:
#include <Servo.h>

Servo servo;
int trigPin  = 5;
int echoPin  = 6;
int servoPin = 9;
long dist;

void setup() {
  Serial.begin(9600);
  servo.attach(servoPin);
  pinMode(trigPin, OUTPUT);
  pinMode(echoPin, INPUT);
  servo.write(0);
}

void loop() {
  digitalWrite(trigPin, LOW);
  delayMicroseconds(2);
  digitalWrite(trigPin, HIGH);
  delayMicroseconds(10);
  digitalWrite(trigPin, LOW);
  long duration = pulseIn(echoPin, HIGH);
  dist = duration * 0.034 / 2;
  if (dist < 20) {
    servo.write(90);
    delay(3000);
    servo.write(0);
  }
}

--- END EXAMPLE ---"""


USER_PROMPT_T6_TEMPLATE = """Generate a complete Arduino .ino sketch for this project.

Board: {board}

Components: {components}

Libraries to include: {libraries}

Global variables:
{global_vars}

Setup logic:
{setup_logic}

Loop logic:
{loop_logic}

Notes: {notes}

Output ONLY the raw .ino code."""


print(' Task 6 prompts defined')

In [ ]:
# ── CELL 3: TASK 6 — CODE GENERATION AGENT ───────────────────────────────────

T6_MAX_TOKENS = 2000

# ── Library name corrections ──────────────────────────────────────────────────
INSTALLED_LIBS = {
    'adafruit_busio', 'adafruitgfx', 'adafruitgfxlibrary',
    'adafruit_neopixel', 'adafruitneopixel',
    'adafruitssd1306', 'adafruitunifiedsensor',
    'dht', 'dhtsensorlibrary', 'dht11', 'dht22',
    'fastled', 'irremote', 'keypad', 'ledcontrol',
    'liquidcrystal', 'liquidcrystali2c',
    'newping', 'sd', 'servo', 'stepper', 'tinygpsplus',
    'tft_espi', 'tftespi',
    'blynk', 'blynkncpdriver',
    'mpu6050',
    'xpt2046_touchscreen', 'xpt2046touchscreen',
    'radiohead', 'rh_ask',
}

LIB_NAME_CORRECTIONS = {
    'blynksimpleesp8266':   'Blynk',
    'blynksimpleesp32':     'Blynk',
    'blynksimplewifi':      'Blynk',
    'blynk':                'Blynk',
    'ledmatrix':            'Arduino_LED_Matrix',
    'arduino_led_matrix':   'Arduino_LED_Matrix',
    'arduinoledmatrix':     'Arduino_LED_Matrix',
    'wifi':                 'WiFiS3',
    'wifis3':               'WiFiS3',
    'tft_espi':             'TFT_eSPI',
    'tftespi':              'TFT_eSPI',
    'xt2046_touchscreen':   'XPT2046_Touchscreen',
    'xpt2046touchscreen':   'XPT2046_Touchscreen',
    'xpt2046_touchscreen':  'XPT2046_Touchscreen',
    'rh_ask':               'RadioHead',
    'radiohead':            'RadioHead',
    'dht':                  'DHT sensor library',
    'virtualwire':          'RadioHead',
}


def correct_lib_names(libs: list, fqbn: str) -> list:
    """Map wrong/generic library names to correct Arduino registry names."""
    corrected = []
    seen      = set()
    for lib in libs:
        n = normalise_lib(lib)
        if fqbn == 'arduino:renesas_uno:unor4wifi':
            corrected_name = LIB_NAME_CORRECTIONS.get(n, lib)
        else:
            avr_corrections = {k: v for k, v in LIB_NAME_CORRECTIONS.items()
                               if k not in ('wifi', 'wifis3')}
            corrected_name = avr_corrections.get(n, lib)
        key = normalise_lib(corrected_name)
        if key not in seen:
            seen.add(key)
            corrected.append(corrected_name)
    return corrected


def format_global_vars(global_vars: list) -> str:
    lines = []
    for v in global_vars:
        name = v.get('name',        '')
        typ  = v.get('type',        'int')
        val  = v.get('value',       None)
        desc = v.get('description', '')
        line = f'  {typ} {name}'
        if val is not None:
            line += f' = {val}'
        if desc:
            line += f'  // {desc}'
        lines.append(line)
    return '\n'.join(lines) if lines else '  // no global vars'


def run_task6_agent(project:    dict,
                    t2_details: list,
                    t5_output:  dict,
                    client,
                    output_dir: Path = None) -> dict:   # ← ADD output_dir
    """
    Task 6 Agent: LLM generates complete .ino code from T5 plan.
    output_dir: custom output folder (defaults to T6_OUTPUT_DIR)
    Returns code string + sketch file path.
    """
    pid   = project.get('project_id')
    board = project.get('board', 'Arduino Uno')
    fqbn  = FQBN_MAP.get(board, 'arduino:avr:uno')

    # ── Build library list ────────────────────────────────────────────────────
    t5_libs = t5_output.get('libraries', [])
    if not t5_libs:
        t5_libs = [
            normalize_lib_name(r['library'])
            for r in t2_details
            if isinstance(r, dict) and r.get('library')
            and normalise_lib(r['library']) not in NO_LIBRARY_NEEDED
        ]

    t5_libs = correct_lib_names(t5_libs, fqbn)

    # ── Format prompt inputs ──────────────────────────────────────────────────
    components_str  = ', '.join(
        r.get('component', '') for r in t2_details if isinstance(r, dict)
    )
    libraries_str   = ', '.join(t5_libs) if t5_libs else 'None — use native Arduino functions'
    global_vars_str = format_global_vars(t5_output.get('global_vars', []))
    setup_str       = '\n'.join(f'  {s}' for s in t5_output.get('setup_logic', []))
    loop_str        = '\n'.join(f'  {s}' for s in t5_output.get('loop_logic',  []))
    notes_str       = t5_output.get('notes', '')

    user_prompt = USER_PROMPT_T6_TEMPLATE.format(
        board       = board,
        components  = components_str,
        libraries   = libraries_str,
        global_vars = global_vars_str,
        setup_logic = setup_str,
        loop_logic  = loop_str,
        notes       = notes_str,
    )

    if len(user_prompt) > MAX_PROMPT_CHARS:
        user_prompt = user_prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = client.generate(
        user_prompt,
        system_prompt = SYSTEM_PROMPT_T6,
        max_tokens    = T6_MAX_TOKENS,
    )

    code = raw.strip()
    code = re.sub(r'^```(?:cpp|arduino|ino|c\+\+)?\n?', '', code)
    code = re.sub(r'\n?```$', '', code).strip()

    if not code or len(code) < 50:
        return {
            'project_id':  pid,
            'status':      'generation_failed',
            'code':        '',
            'fqbn':        fqbn,
            'libs_used':   t5_libs,
            'sketch_path': None,
            'latency':     latency,
        }

    # ── Write sketch — use output_dir if provided, else T6_OUTPUT_DIR ─────────
    base_dir    = Path(output_dir) if output_dir else T6_OUTPUT_DIR  # ← KEY FIX
    sketch_dir  = base_dir / f'sketch_{pid}'
    sketch_dir.mkdir(parents=True, exist_ok=True)
    sketch_path = sketch_dir / f'sketch_{pid}.ino'
    sketch_path.write_text(code, encoding='utf-8')

    return {
        'project_id':  pid,
        'status':      'ok',
        'code':        code,
        'fqbn':        fqbn,
        'libs_used':   t5_libs,
        'sketch_path': str(sketch_path),
        'latency':     latency,
    }


print(' run_task6_agent defined')
print(f'   Output dir  : {T6_OUTPUT_DIR.resolve()}')
print(f'   max_tokens  : {T6_MAX_TOKENS}')
print(f'   Lib corrections: {len(LIB_NAME_CORRECTIONS)} mappings')
print('   output_dir param: supported ')

In [ ]:
# ── CELL 4: TASK 6 — COMPILATION (fully automatic) ───────────────────────────
COMPILE_TIMEOUT = 90


def compile_sketch(sketch_path: str, fqbn: str, libs_used: list) -> dict:
    """
    Fully automatic compilation:
    1. For each required lib — check installed, search registry, install
    2. Attempt real compilation
    3. If compile fails with 'No such file' — extract lib name, retry install
    """
    if not sketch_path or not Path(sketch_path).exists():
        return {
            'status': 'skipped', 'success': False,
            'error': 'no sketch file', 'missing_libs': [],
            'installed': [], 'mode': 'skipped',
        }

    installed_now = []
    still_missing = []

    # ── Phase 1: Pre-install all required libs ────────────────────────────────
    for lib in libs_used:
        if is_lib_available(lib, fqbn):
            continue
        success, actual_name = smart_install_lib(lib)
        if success:
            installed_now.append(actual_name)
        else:
            still_missing.append(lib)

    # ── Phase 2: Attempt compilation ──────────────────────────────────────────
    for attempt in range(2):   # allow one retry after auto-fix
        try:
            result = subprocess.run(
                [ARDUINO_CLI, 'compile', '--fqbn', fqbn, sketch_path],
                capture_output=True, text=True, timeout=COMPILE_TIMEOUT,
            )

            if result.returncode == 0:
                return {
                    'status':       'success',
                    'success':      True,
                    'error':        '',
                    'missing_libs': still_missing,
                    'installed':    installed_now,
                    'mode':         'real',
                }

            # ── Phase 3: Parse error — auto-fix missing includes ──────────────
            stderr = result.stderr or result.stdout

            # Extract missing header names from fatal error lines
            # e.g. fatal error: SomeLib.h: No such file or directory
            missing_headers = re.findall(
                r'fatal error:\s*([^\s:]+\.h):\s*No such file',
                stderr
            )

            if missing_headers and attempt == 0:
                for header in missing_headers:
                    lib_name = header.removesuffix('.h')
                    print(f'    🔍 Auto-detecting missing lib: {header}')
                    success, actual = smart_install_lib(lib_name)
                    if success:
                        installed_now.append(actual)
                    else:
                        still_missing.append(lib_name)
                continue   # retry compilation

            # Final failure
            error_lines = [l for l in stderr.splitlines() if 'error:' in l.lower()]
            error = '\n'.join(error_lines[:5]) if error_lines else stderr[:500]

            return {
                'status':       'failed',
                'success':      False,
                'error':        error,
                'missing_libs': still_missing,
                'installed':    installed_now,
                'mode':         'real',
            }

        except subprocess.TimeoutExpired:
            return {
                'status': 'timeout', 'success': False,
                'error': f'Timed out after {COMPILE_TIMEOUT}s',
                'missing_libs': still_missing, 'installed': installed_now,
                'mode': 'real',
            }
        except Exception as e:
            return {
                'status': 'error', 'success': False, 'error': str(e),
                'missing_libs': still_missing, 'installed': installed_now,
                'mode': 'real',
            }


print(' compile_sketch defined — fully automatic')
print('   Phase 1: pre-install all required libs via registry search')
print('   Phase 2: compile')
print('   Phase 3: parse errors → auto-detect missing headers → retry')

In [ ]:
# ── CELL 5: TASK 6 — SCORING ──────────────────────────────────────────────────

SYSTEM_PROMPT_JUDGE_T6 = """You are an expert Arduino firmware reviewer.

You will be given an Arduino .ino sketch and the project requirements.
Score it on TWO dimensions:

LOGIC_CORRECTNESS (0–5):
- 5 = fully implements all tasks, conditions, thresholds from intent
- 4 = implements most logic, minor gaps
- 3 = implements core functionality, missing some conditions
- 2 = partial implementation, significant gaps
- 1 = minimal correct logic
- 0 = wrong or empty

CODE_QUALITY (0–5):
- 5 = clean, correct Arduino conventions, good comments, readable
- 4 = mostly good, minor style issues
- 3 = works but messy or unconventional
- 2 = poor style, hard to follow
- 1 = barely readable
- 0 = unreadable or non-Arduino

Output ONLY a JSON object. No explanation, no markdown.
{"logic_correctness": 4, "code_quality": 4, "reasoning": "brief explanation"}"""


USER_PROMPT_JUDGE_T6_TEMPLATE = """Review this Arduino sketch.

Project description: {description}

Required components : {components}
Required libraries  : {libraries}
Pin assignments     : {pin_assignments}
Intent summary      : {logic_summary}

Generated sketch:
{code}

Return ONLY the JSON object."""


def score_task6_metrics(project:        dict,
                         t2_details:     list,
                         t4_output:      dict,
                         t6_output:      dict,
                         compile_result: dict) -> dict:
    """
    Automated metrics from generated code:
    - component_correct : required components appear in code (binary)
    - library_f1        : precision/recall of #include statements
    - pin_valid         : % of T4 pin assignments found in code
    """
    code = t6_output.get('code', '')
    if not code:
        return {
            'component_correct': 0,
            'library_precision': 0.0,
            'library_recall':    0.0,
            'library_f1':        0.0,
            'pin_valid':         0.0,
        }

    code_lower = code.lower()

    # ── Component correctness ─────────────────────────────────────────────────
    predicted  = [r.get('component', '') for r in t2_details if isinstance(r, dict)]
    comp_hits  = sum(
        1 for c in predicted
        if any(w.lower() in code_lower for w in c.split() if len(w) > 3)
    )
    component_correct = int(comp_hits >= max(1, len(predicted) * 0.5))

    # ── Library correctness ───────────────────────────────────────────────────
    # FIX: capture group extracts only filename — avoids stripping lib name chars
    # Old regex: re.sub(r'[\s<>"#include]', '', m) was stripping letters i,n,c,l,u,d,e
    includes_in_code = set(
        normalise_lib(m)
        for m in re.findall(r'#include\s*[<"]([^>"]+)[>"]', code)
    )

    # Required libs — combine T5 libs_used + T2 details for full coverage
    required_libs = set()
    for l in t6_output.get('libs_used', []):
        n = normalise_lib(l)
        if n and n not in NO_LIBRARY_NEEDED:
            required_libs.add(n)
    for r in t2_details:
        if isinstance(r, dict) and r.get('library'):
            n = normalise_lib(r['library'])
            if n and n not in NO_LIBRARY_NEEDED:
                required_libs.add(n)

    if required_libs:
        tp            = len(required_libs & includes_in_code)
        lib_precision = tp / len(includes_in_code) if includes_in_code else 0.0
        lib_recall    = tp / len(required_libs)
        lib_f1        = (2 * lib_precision * lib_recall /
                         (lib_precision + lib_recall)
                         if (lib_precision + lib_recall) > 0 else 0.0)
    else:
        # No libraries needed — correct if code also has no spurious includes
        lib_precision = lib_recall = lib_f1 = 1.0 if not includes_in_code else 0.8

    # ── Pin validity ──────────────────────────────────────────────────────────
    pin_hits  = 0
    pin_total = 0
    for assignment in t4_output.get('assignments', []):
        for pin_name, pin_val in assignment.get('pins', {}).items():
            pin_total += 1
            if str(pin_val) in code:
                pin_hits += 1
    pin_valid = pin_hits / pin_total if pin_total > 0 else 1.0

    return {
        'component_correct': component_correct,
        'library_precision': round(lib_precision, 4),
        'library_recall':    round(lib_recall,    4),
        'library_f1':        round(lib_f1,        4),
        'pin_valid':         round(pin_valid,      4),
    }


def score_task6_with_judge(project:    dict,
                            t2_details: list,
                            t4_output:  dict,
                            t3_output:  dict,
                            t6_output:  dict,
                            client) -> dict:
    """LLM judge scores logic correctness and code quality (0-5 each)."""
    code = t6_output.get('code', '')
    if not code:
        return {
            'logic_correctness': 0,
            'code_quality':      0,
            'reasoning':         'no code generated',
            'latency':           0.0,
        }

    _time.sleep(2.0)   # rate limit buffer before judge call

    t3_ok = isinstance(t3_output, dict) and t3_output.get('status') == 'ok'
    desc  = str(project.get('description', ''))
    comps = ', '.join(r.get('component', '') for r in t2_details if isinstance(r, dict))
    libs  = ', '.join(
        r['library'] for r in t2_details
        if isinstance(r, dict) and r.get('library')
        and normalise_lib(r.get('library', '')) not in NO_LIBRARY_NEEDED
    )

    user_prompt = USER_PROMPT_JUDGE_T6_TEMPLATE.format(
        description     = desc,
        components      = comps,
        libraries       = libs,
        pin_assignments = format_pin_assignments(t4_output),
        logic_summary   = t3_output.get('logic_summary', '') if t3_ok else '',
        code            = code[:3000],   # cap to avoid huge prompts
    )

    result, latency = call_llm_json(
        user_prompt,
        model         = client,
        system_prompt = SYSTEM_PROMPT_JUDGE_T6,
        max_tokens    = 512,
    )

    if not isinstance(result, dict):
        return {
            'logic_correctness': 0,
            'code_quality':      0,
            'reasoning':         'judge parse failed',
            'latency':           latency,
        }

    logic = max(0, min(5, int(result.get('logic_correctness', 0))))
    qual  = max(0, min(5, int(result.get('code_quality',      0))))

    return {
        'logic_correctness': logic,
        'code_quality':      qual,
        'reasoning':         result.get('reasoning', ''),
        'latency':           latency,
    }


print(' Task 6 scoring functions defined')
print('   Auto metrics : component_correct, library_f1, pin_valid')
print('   Judge metrics: logic_correctness (0-5), code_quality (0-5)')
print('   Library F1 fix: capture group regex — no longer strips lib name chars')

In [ ]:
# ── CELL 6: TASK 6 — SANITY CHECK (3 PROJECTS) ───────────────────────────────
print('── Task 6 Sanity Check (3 projects) ──\n')

for proj in projects[:3]:
    pid   = proj['project_id']
    board = proj.get('board', 'Arduino Uno')

    # Run full pipeline to get T2–T5 outputs
    t1 = run_task1_agent(proj, active_client, use_rag=True)
    predicted_names = t1['predicted_names']
    t2_details = run_task2_agent(predicted_names, proj.get('description',''),
                                 active_client, board=board)
    t3 = run_task3_agent(proj, predicted_names, active_client)
    t4 = run_task4_agent(proj, predicted_names, active_client, df_comp)
    t5 = run_task5_agent(proj, t3, t4, t2_details, active_client)

    # Task 6
    t6 = run_task6_agent(proj, t2_details, t5, active_client)
    compile_result = compile_sketch(t6['sketch_path'], t6['fqbn'], t6['libs_used'])
    auto_scores    = score_task6_metrics(proj, t2_details, t4, t6, compile_result)
    judge_scores   = score_task6_with_judge(proj, t2_details, t4, t3, t6, active_client)

    print(f'Project {pid} [{proj.get("complexity")}] | {board}')
    print(f'  T6 Status        : {t6["status"]}')
    print(f'  Sketch           : {t6["sketch_path"]}')
    print(f'  Libraries used   : {t6["libs_used"]}')
    print(f'  T6 Latency       : {t6["latency"]:.2f}s')
    print()
    print(f'  ── Compilation ──')
    print(f'  Status           : {compile_result["status"]}')
    print(f'  Success          : {compile_result["success"]}')
    print(f'  Mode             : {compile_result["mode"]}')
    if compile_result['error']:
        print(f'  Error            : {compile_result["error"][:150]}')
    if compile_result['missing_libs']:
        print(f'  Missing libs     : {compile_result["missing_libs"]}')
    print()
    print(f'  ── Auto Metrics ──')
    print(f'  Component Correct: {auto_scores["component_correct"]}')
    print(f'  Library F1       : {auto_scores["library_f1"]:.2f}')
    print(f'  Pin Valid        : {auto_scores["pin_valid"]:.2f}')
    print()
    print(f'  ── Judge Scores ──')
    print(f'  Logic Correct    : {judge_scores["logic_correctness"]} / 5')
    print(f'  Code Quality     : {judge_scores["code_quality"]} / 5')
    print(f'  Reasoning        : {judge_scores["reasoning"]}')
    print(f'  Judge Latency    : {judge_scores["latency"]:.2f}s')
    print()
    print('-' * 65)
    print()

In [ ]:
# ── CELL 7: FULL CONNECTED RUN — TASKS 1 + 2 + 3 + 4 + 5 + 6 ────────────────
SAVE_PATH         = 'connected_results_t1_t2_t3_t4_t5_t6.json'
connected_results = []
error_count       = 0

_EMPTY_T3 = {
    'project_id': None, 'status': 'error',
    'tasks': [], 'conditions': [], 'thresholds': [],
    'inputs': [], 'outputs': [], 'logic_summary': '', 'latency': 0.0, 'raw': {},
}
_EMPTY_T4 = {
    'project_id': None, 'status': 'error',
    'assignments': [], 'conflicts': [], 'feasibility': 'unknown',
    'spec_list': [], 'latency': 0.0,
}
_EMPTY_T5 = {
    'project_id': None, 'status': 'error',
    'libraries': [], 'global_vars': [], 'setup_logic': [], 'loop_logic': [],
    'notes': '', 'latency': 0.0,
}
_EMPTY_T6 = {
    'project_id': None, 'status': 'error',
    'code': '', 'fqbn': '', 'libs_used': [], 'sketch_path': None, 'latency': 0.0,
}
_ZERO_T5_SCORE = {
    'library_coverage': 0.0, 'setup_completeness': 0.0,
    'loop_correctness': 0.0, 'f1': 0.0,
    'reasoning': 'upstream error', 'latency': 0.0,
}
_ZERO_T6_AUTO = {
    'component_correct': 0, 'library_precision': 0.0,
    'library_recall': 0.0, 'library_f1': 0.0, 'pin_valid': 0.0,
}
_ZERO_T6_JUDGE = {
    'logic_correctness': 0, 'code_quality': 0,
    'reasoning': 'upstream error', 'latency': 0.0,
}
_ZERO_COMPILE = {
    'status': 'skipped', 'success': False, 'error': '',
    'missing_libs': [], 'installed': [], 'mode': 'skipped',
}

for proj in tqdm(projects[:10], desc='Tasks 1→6'):
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')
    gt_libs = set(proj.get('ground_truth_libraries', []))

    # ── Task 1 ───────────────────────────────────────────────────────────────
    try:
        t1_output = run_task1_agent(proj, active_client, use_rag=True)
    except Exception as e:
        error_count += 1
        tqdm.write(f'  ERROR T1 project {pid}: {e}')
        continue

    gt_names        = get_gt_names(proj)
    predicted_names = t1_output['predicted_names']
    if not gt_names:
        continue

    t1_scores    = match_predicted_to_gt(predicted_names, gt_names)
    t1_tp        = t1_scores['tp']
    t1_fp        = t1_scores['fp']
    t1_fn        = t1_scores['fn']
    t1_precision = t1_tp / (t1_tp + t1_fp) if (t1_tp + t1_fp) > 0 else 0
    t1_recall    = t1_tp / (t1_tp + t1_fn) if (t1_tp + t1_fn) > 0 else 0
    t1_f1        = (2 * t1_precision * t1_recall / (t1_precision + t1_recall)
                    if (t1_precision + t1_recall) > 0 else 0)

    # ── Task 2 ───────────────────────────────────────────────────────────────
    try:
        t2_details = run_task2_agent(
            predicted_names, desc, active_client, board=board
        )
    except Exception as e:
        tqdm.write(f'  ERROR T2 project {pid}: {e}')
        t2_details = [
            {'component': n, 'library': None, 'status': 'error',
             'confidence': 'low', 'reasoning': str(e), 'latency': 0.0}
            for n in predicted_names
        ]

    retrieved_libs = {r['library'] for r in t2_details if r.get('library')}
    gt_canon       = canonicalize_set(gt_libs)
    ret_canon      = canonicalize_set(retrieved_libs)
    gt_canon       = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon      = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if gt_canon:
        t2_tp        = match_with_equivalence(gt_canon, ret_canon)
        t2_precision = t2_tp / len(ret_canon) if ret_canon else 0
        t2_recall    = t2_tp / len(gt_canon)  if gt_canon  else 0
        t2_f1        = (2 * t2_precision * t2_recall / (t2_precision + t2_recall)
                        if (t2_precision + t2_recall) > 0 else 0)
    else:
        t2_tp = t2_precision = t2_recall = t2_f1 = None

    # ── Task 3 ───────────────────────────────────────────────────────────────
    try:
        t3_output = run_task3_agent(proj, predicted_names, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T3 project {pid}: {e}')
        t3_output = {**_EMPTY_T3, 'project_id': pid}

    try:
        t3_score = score_intent_with_judge(proj, t3_output, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T3 judge project {pid}: {e}')
        t3_score = {'completeness': 0.0, 'accuracy': 0.0,
                    'f1': 0.0, 'reasoning': str(e), 'latency': 0.0}

    # ── Task 4 ───────────────────────────────────────────────────────────────
    try:
        t4_output = run_task4_agent(proj, predicted_names, active_client, df_comp)
    except Exception as e:
        tqdm.write(f'  ERROR T4 project {pid}: {e}')
        t4_output = {**_EMPTY_T4, 'project_id': pid}

    try:
        t4_score = score_task4(t4_output, board)
    except Exception as e:
        tqdm.write(f'  ERROR T4 score project {pid}: {e}')
        t4_score = {'pin_validity': 0.0, 'conflict_free': 0.0,
                    'coverage': 0.0, 'f1': 0.0, 'conflicts': [], 'details': []}

    # ── Task 5 ───────────────────────────────────────────────────────────────
    try:
        t5_output = run_task5_agent(
            proj, t3_output, t4_output, t2_details, active_client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T5 project {pid}: {e}')
        t5_output = {**_EMPTY_T5, 'project_id': pid}

    try:
        t5_score = score_task5_with_judge(
            proj, t2_details, t3_output, t4_output, t5_output, active_client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T5 judge project {pid}: {e}')
        t5_score = {**_ZERO_T5_SCORE, 'reasoning': str(e)}

    # ── Task 6 ───────────────────────────────────────────────────────────────
    try:
        t6_output = run_task6_agent(proj, t2_details, t5_output, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR T6 project {pid}: {e}')
        t6_output = {**_EMPTY_T6, 'project_id': pid}

    try:
        compile_result = compile_sketch(
            t6_output['sketch_path'], t6_output['fqbn'], t6_output['libs_used']
        )
    except Exception as e:
        tqdm.write(f'  ERROR T6 compile project {pid}: {e}')
        compile_result = _ZERO_COMPILE.copy()

    try:
        t6_auto = score_task6_metrics(
            proj, t2_details, t4_output, t6_output, compile_result
        )
    except Exception as e:
        tqdm.write(f'  ERROR T6 auto score project {pid}: {e}')
        t6_auto = _ZERO_T6_AUTO.copy()

    try:
        t6_judge = score_task6_with_judge(
            proj, t2_details, t4_output, t3_output, t6_output, active_client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T6 judge project {pid}: {e}')
        t6_judge = {**_ZERO_T6_JUDGE, 'reasoning': str(e)}

    connected_results.append({
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,

        'task1': {
            'predicted_names': predicted_names,
            'gt_names':        list(gt_names),
            'matched':         list(t1_scores['matched_gt']),
            'precision':       round(t1_precision, 4),
            'recall':          round(t1_recall,    4),
            'f1':              round(t1_f1,        4),
            'latency':         t1_output['latency'],
        },

        'task2': {
            'gt_libs':        list(gt_canon)  if gt_canon  else [],
            'retrieved_libs': list(ret_canon) if ret_canon else [],
            'tp':             t2_tp,
            'precision':      round(t2_precision, 4) if t2_precision is not None else None,
            'recall':         round(t2_recall,    4) if t2_recall    is not None else None,
            'f1':             round(t2_f1,        4) if t2_f1        is not None else None,
            'details':        t2_details,
        },

        'task3': {
            'tasks':         t3_output.get('tasks', []),
            'conditions':    t3_output.get('conditions', []),
            'thresholds':    t3_output.get('thresholds', []),
            'inputs':        t3_output.get('inputs', []),
            'outputs':       t3_output.get('outputs', []),
            'logic_summary': t3_output.get('logic_summary', ''),
            'completeness':  t3_score['completeness'],
            'accuracy':      t3_score['accuracy'],
            'f1':            t3_score['f1'],
            'reasoning':     t3_score['reasoning'],
            'status':        t3_output.get('status', 'error'),
            'latency':       t3_output.get('latency', 0.0),
            'judge_latency': t3_score['latency'],
        },

        'task4': {
            'assignments':   t4_output.get('assignments', []),
            'conflicts':     t4_score['conflicts'],
            'feasibility':   t4_output.get('feasibility', 'unknown'),
            'pin_validity':  t4_score['pin_validity'],
            'conflict_free': t4_score['conflict_free'],
            'coverage':      t4_score['coverage'],
            'f1':            t4_score['f1'],
            'details':       t4_score['details'],
            'status':        t4_output.get('status', 'error'),
            'latency':       t4_output.get('latency', 0.0),
        },

        'task5': {
            'libraries':          t5_output['libraries'],
            'global_vars':        t5_output['global_vars'],
            'setup_logic':        t5_output['setup_logic'],
            'loop_logic':         t5_output['loop_logic'],
            'notes':              t5_output.get('notes', ''),
            'library_coverage':   t5_score['library_coverage'],
            'setup_completeness': t5_score['setup_completeness'],
            'loop_correctness':   t5_score['loop_correctness'],
            'f1':                 t5_score['f1'],
            'reasoning':          t5_score['reasoning'],
            'status':             t5_output.get('status', 'error'),
            'latency':            t5_output.get('latency', 0.0),
            'judge_latency':      t5_score['latency'],
        },

        'task6': {
            'sketch_path':        t6_output.get('sketch_path', ''),
            'libs_used':          t6_output.get('libs_used', []),
            'fqbn':               t6_output.get('fqbn', ''),
            'compilation_status': compile_result['status'],
            'compilation_success':compile_result['success'],
            'compilation_mode':   compile_result['mode'],
            'compile_error':      compile_result.get('error', ''),
            'missing_libs':       compile_result.get('missing_libs', []),
            'installed_libs':     compile_result.get('installed', []),
            'component_correct':  t6_auto['component_correct'],
            'library_precision':  t6_auto['library_precision'],
            'library_recall':     t6_auto['library_recall'],
            'library_f1':         t6_auto['library_f1'],
            'pin_valid':          t6_auto['pin_valid'],
            'logic_correctness':  t6_judge['logic_correctness'],
            'code_quality':       t6_judge['code_quality'],
            'reasoning':          t6_judge['reasoning'],
            'status':             t6_output.get('status', 'error'),
            'latency':            t6_output.get('latency', 0.0),
            'judge_latency':      t6_judge['latency'],
        },
    })

    if len(connected_results) % 5 == 0:
        with open(SAVE_PATH, 'w') as f:
            json.dump(connected_results, f, indent=2)
        tqdm.write(f'   Checkpoint — {len(connected_results)} saved')

with open(SAVE_PATH, 'w') as f:
    json.dump(connected_results, f, indent=2)

print(f'Done — saved to {SAVE_PATH}')
print(f'   Evaluated : {len(connected_results)} | Errors: {error_count}')

In [ ]:
# ── CELL 8: PRINT RESULTS ─────────────────────────────────────────────────────
t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t4_all = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']
t5_all = [r['task5'] for r in connected_results if r['task5']['status'] == 'ok']
t6_all = [r['task6'] for r in connected_results if r['task6']['status'] == 'ok']

print('=' * 65)
print(' Connected Pipeline — Tasks 1 + 2 + 3 + 4 + 5 + 6')
print('=' * 65)
print(f'  Model              : {active_client.model_key}')
print(f'  Projects evaluated : {len(connected_results)}')
print(f'  Errors             : {error_count}')
print()

print(f'  Task 1 — Component Extraction')
print(f'    Avg Precision    : {np.mean([r["precision"] for r in t1_all]):.3f}')
print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t1_all]):.3f}')
print(f'    Avg F1           : {np.mean([r["f1"]        for r in t1_all]):.3f}')
print(f'    Avg Latency      : {np.mean([r["latency"]   for r in t1_all]):.2f}s')
print()

print(f'  Task 2 — Library Retrieval')
if t2_all:
    print(f'    Avg Precision    : {np.mean([r["precision"] for r in t2_all]):.3f}')
    print(f'    Avg Recall       : {np.mean([r["recall"]    for r in t2_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]        for r in t2_all]):.3f}')
print(f'    Scored           : {len(t2_all)} / {len(connected_results)}')
print()

print(f'  Task 3 — Intent Understanding')
if t3_all:
    print(f'    Avg Completeness : {np.mean([r["completeness"] for r in t3_all]):.3f}')
    print(f'    Avg Accuracy     : {np.mean([r["accuracy"]     for r in t3_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]           for r in t3_all]):.3f}')
    print(f'    Avg Latency      : {np.mean([r["latency"]      for r in t3_all]):.2f}s')
print(f'    Scored           : {len(t3_all)} / {len(connected_results)}')
print()

print(f'  Task 4 — Hardware Pin Mapping')
if t4_all:
    print(f'    Avg Pin Validity : {np.mean([r["pin_validity"]  for r in t4_all]):.3f}')
    print(f'    Avg Conflict Free: {np.mean([r["conflict_free"] for r in t4_all]):.3f}')
    print(f'    Avg Coverage     : {np.mean([r["coverage"]      for r in t4_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]            for r in t4_all]):.3f}')
print(f'    Scored           : {len(t4_all)} / {len(connected_results)}')
print()

print(f'  Task 5 — Code Planning')
if t5_all:
    print(f'    Avg Lib Coverage : {np.mean([r["library_coverage"]   for r in t5_all]):.3f}')
    print(f'    Avg Setup Comp   : {np.mean([r["setup_completeness"] for r in t5_all]):.3f}')
    print(f'    Avg Loop Correct : {np.mean([r["loop_correctness"]   for r in t5_all]):.3f}')
    print(f'    Avg F1           : {np.mean([r["f1"]                 for r in t5_all]):.3f}')
    print(f'    Avg Latency      : {np.mean([r["latency"]            for r in t5_all]):.2f}s')
    print(f'    Parse failed     : {sum(1 for r in connected_results if r["task5"]["status"] != "ok")}')
print(f'    Scored           : {len(t5_all)} / {len(connected_results)}')
print()

print(f'  Task 6 — Code Synthesis')
if t6_all:
    compiled     = [r for r in t6_all if r['compilation_success']]
    failed       = [r for r in t6_all if r['compilation_status'] == 'failed']
    lib_missing  = [r for r in t6_all if r['compilation_status'] == 'library_missing']
    print(f'    Compilation Success  : {len(compiled)} / {len(t6_all)}  '
          f'({len(compiled)/len(t6_all)*100:.0f}%)')
    print(f'    Compile Failed       : {len(failed)}')
    print(f'    Library Missing      : {len(lib_missing)}')
    print(f'    Avg Component Correct: {np.mean([r["component_correct"] for r in t6_all]):.3f}')
    print(f'    Avg Library F1       : {np.mean([r["library_f1"]        for r in t6_all]):.3f}')
    print(f'    Avg Pin Valid        : {np.mean([r["pin_valid"]          for r in t6_all]):.3f}')
    print(f'    Avg Logic Correct    : {np.mean([r["logic_correctness"] for r in t6_all]):.2f} / 5')
    print(f'    Avg Code Quality     : {np.mean([r["code_quality"]      for r in t6_all]):.2f} / 5')
    print(f'    Avg Latency          : {np.mean([r["latency"]           for r in t6_all]):.2f}s')
print(f'    Scored               : {len(t6_all)} / {len(connected_results)}')
print('=' * 65)

# ── T6 by complexity ──────────────────────────────────────────────────────────
print('\n── Task 6 by Complexity ──')
for complexity in ['Low', 'Medium', 'High', 'Unknown']:
    subset = [r for r in connected_results
              if r['complexity'] == complexity
              and r['task6']['status'] == 'ok']
    if not subset:
        continue
    compiled_ct = sum(1 for r in subset if r['task6']['compilation_success'])
    print(f'  {complexity:<8} ({len(subset):>3}) — '
          f'Compiled:{compiled_ct}/{len(subset)}  '
          f'LibF1:{np.mean([r["task6"]["library_f1"]        for r in subset]):.2f}  '
          f'Logic:{np.mean([r["task6"]["logic_correctness"] for r in subset]):.1f}/5  '
          f'Quality:{np.mean([r["task6"]["code_quality"]    for r in subset]):.1f}/5')

# ── Per project sample ────────────────────────────────────────────────────────
print('\n── Per Project Sample (first 3) ──')
for r in connected_results[:3]:
    print(f"\n  Project {r['project_id']} [{r['complexity']}] | {r['board']}")
    print(f"  T1:{r['task1']['f1']:.2f}  "
          f"T2:{r['task2']['f1']}  "
          f"T3:{r['task3']['f1']:.2f}  "
          f"T4:{r['task4']['f1']:.2f}  "
          f"T5:{r['task5']['f1']:.2f}")
    print(f"  T6 Compiled   : {r['task6']['compilation_success']} "
          f"({r['task6']['compilation_status']})")
    print(f"  T6 Lib F1     : {r['task6']['library_f1']:.2f}")
    print(f"  T6 Pin Valid  : {r['task6']['pin_valid']:.2f}")
    print(f"  T6 Logic      : {r['task6']['logic_correctness']}/5")
    print(f"  T6 Quality    : {r['task6']['code_quality']}/5")
    print(f"  T6 Reasoning  : {r['task6']['reasoning']}")

In [ ]:
# ── CELL 9: TASK 6 — FAILURE ANALYSIS ────────────────────────────────────────
t6_all      = [r for r in connected_results if r['task6']['status'] == 'ok']
t6_failed   = [r for r in connected_results if r['task6']['status'] != 'ok']
t6_compiled = [r for r in t6_all if r['task6']['compilation_success']]
t6_no_comp  = [r for r in t6_all if not r['task6']['compilation_success']]

print('=' * 60)
print(' Task 6 — Failure Analysis')
print('=' * 60)
print(f'  Total evaluated      : {len(connected_results)}')
print(f'  Generation ok        : {len(t6_all)}')
print(f'  Generation failed    : {len(t6_failed)}')
print(f'  Compiled successfully: {len(t6_compiled)}')
print(f'  Compile failed       : {len([r for r in t6_all if r["task6"]["compilation_status"] == "failed"])}')
print(f'  Library missing      : {len([r for r in t6_all if r["task6"]["compilation_status"] == "library_missing"])}')
print('=' * 60)

# ── Score breakdown ───────────────────────────────────────────────────────────
if t6_all:
    lib_f1s  = [r['task6']['library_f1']        for r in t6_all]
    pin_vals = [r['task6']['pin_valid']          for r in t6_all]
    logics   = [r['task6']['logic_correctness']  for r in t6_all]
    quals    = [r['task6']['code_quality']       for r in t6_all]

    print(f'\n── Score Breakdown ──')
    print(f'  Library F1    — min:{min(lib_f1s):.2f}  max:{max(lib_f1s):.2f}  avg:{np.mean(lib_f1s):.2f}')
    print(f'  Pin Valid     — min:{min(pin_vals):.2f}  max:{max(pin_vals):.2f}  avg:{np.mean(pin_vals):.2f}')
    print(f'  Logic (0-5)   — min:{min(logics)}     max:{max(logics)}     avg:{np.mean(logics):.2f}')
    print(f'  Quality (0-5) — min:{min(quals)}     max:{max(quals)}     avg:{np.mean(quals):.2f}')

# ── Compilation failure reasons ───────────────────────────────────────────────
print(f'\n── Compilation Failures ──')
for r in t6_no_comp:
    print(f"  Project {r['project_id']} [{r['complexity']}] — "
          f"status: {r['task6']['compilation_status']}")
    if r['task6']['compile_error']:
        print(f"    Error   : {r['task6']['compile_error'][:120]}")
    if r['task6']['missing_libs']:
        print(f"    Missing : {r['task6']['missing_libs']}")

# ── Most common missing libraries ─────────────────────────────────────────────
all_missing = Counter()
for r in t6_all:
    for lib in r['task6']['missing_libs']:
        all_missing[lib] += 1

if all_missing:
    print(f'\n── Most Common Missing Libraries ──')
    for lib, count in all_missing.most_common(10):
        print(f'  {lib:<40} missing {count}x')

# ── Logic score distribution ──────────────────────────────────────────────────
print(f'\n── Logic Correctness Distribution ──')
for score in range(5, -1, -1):
    count = sum(1 for r in t6_all if r['task6']['logic_correctness'] == score)
    bar   = '█' * (count * 20 // max(len(t6_all), 1))
    print(f'  {score}/5 : {count:>4}  {bar}')

# ── Complexity breakdown ──────────────────────────────────────────────────────
print(f'\n── T6 by Complexity ──')
for complexity in ['Low', 'Medium', 'High', 'Unknown']:
    subset = [r for r in connected_results
              if r['complexity'] == complexity
              and r['task6']['status'] == 'ok']
    if not subset:
        continue
    compiled_ct = sum(1 for r in subset if r['task6']['compilation_success'])
    print(f'  {complexity:<8} ({len(subset):>3}) — '
          f'Compiled:{compiled_ct}/{len(subset)}  '
          f'Logic:{np.mean([r["task6"]["logic_correctness"] for r in subset]):.1f}/5  '
          f'Quality:{np.mean([r["task6"]["code_quality"] for r in subset]):.1f}/5  '
          f'LibF1:{np.mean([r["task6"]["library_f1"] for r in subset]):.2f}')

In [ ]:
# ── CELL 10: SAVE RESULTS ─────────────────────────────────────────────────────
SAVE_PATH = 'pipeline_t1_t2_t3_t4_t5_t6_final.json'

t1_all = [r['task1'] for r in connected_results]
t2_all = [r['task2'] for r in connected_results if r['task2']['f1'] is not None]
t3_all = [r['task3'] for r in connected_results if r['task3']['status'] == 'ok']
t4_all = [r['task4'] for r in connected_results if r['task4']['status'] == 'ok']
t5_all = [r['task5'] for r in connected_results if r['task5']['status'] == 'ok']
t6_all = [r['task6'] for r in connected_results if r['task6']['status'] == 'ok']

t6_failed = [r for r in connected_results if r['task6']['status'] != 'ok']


def _by_complexity(results, task_key, metrics):
    out = {}
    for c in ['Low', 'Medium', 'High', 'Unknown']:
        subset = [r[task_key] for r in results
                  if r['complexity'] == c
                  and r[task_key].get('status') == 'ok']
        if not subset:
            out[c] = {'count': 0}
            continue
        out[c] = {'count': len(subset)}
        for m in metrics:
            vals = [s[m] for s in subset if s.get(m) is not None]
            out[c][f'avg_{m}'] = round(float(np.mean(vals)), 4) if vals else None
    return out


summary = {
    'model':           active_client.model_key,
    'total_evaluated': len(connected_results),
    'errors':          error_count,

    'task1': {
        'avg_precision': round(np.mean([r['precision'] for r in t1_all]), 4),
        'avg_recall':    round(np.mean([r['recall']    for r in t1_all]), 4),
        'avg_f1':        round(np.mean([r['f1']        for r in t1_all]), 4),
        'avg_latency':   round(np.mean([r['latency']   for r in t1_all]), 3),
        'perfect_f1':    sum(1 for r in t1_all if r['f1'] == 1.0),
        'zero_f1':       sum(1 for r in t1_all if r['f1'] == 0.0),
    },

    'task2': {
        'avg_precision': round(np.mean([r['precision'] for r in t2_all]), 4) if t2_all else None,
        'avg_recall':    round(np.mean([r['recall']    for r in t2_all]), 4) if t2_all else None,
        'avg_f1':        round(np.mean([r['f1']        for r in t2_all]), 4) if t2_all else None,
        'scored':        len(t2_all),
    },

    'task3': {
        'avg_completeness': round(np.mean([r['completeness'] for r in t3_all]), 4) if t3_all else None,
        'avg_accuracy':     round(np.mean([r['accuracy']     for r in t3_all]), 4) if t3_all else None,
        'avg_f1':           round(np.mean([r['f1']           for r in t3_all]), 4) if t3_all else None,
        'scored':           len(t3_all),
        'by_complexity':    _by_complexity(connected_results, 'task3',
                                           ['completeness', 'accuracy', 'f1']),
    },

    'task4': {
        'avg_pin_validity':  round(np.mean([r['pin_validity']  for r in t4_all]), 4) if t4_all else None,
        'avg_conflict_free': round(np.mean([r['conflict_free'] for r in t4_all]), 4) if t4_all else None,
        'avg_coverage':      round(np.mean([r['coverage']      for r in t4_all]), 4) if t4_all else None,
        'avg_f1':            round(np.mean([r['f1']            for r in t4_all]), 4) if t4_all else None,
        'scored':            len(t4_all),
        'by_complexity':     _by_complexity(connected_results, 'task4',
                                            ['pin_validity', 'conflict_free', 'coverage', 'f1']),
    },

    'task5': {
        'avg_library_coverage':   round(np.mean([r['library_coverage']   for r in t5_all]), 4) if t5_all else None,
        'avg_setup_completeness': round(np.mean([r['setup_completeness'] for r in t5_all]), 4) if t5_all else None,
        'avg_loop_correctness':   round(np.mean([r['loop_correctness']   for r in t5_all]), 4) if t5_all else None,
        'avg_f1':                 round(np.mean([r['f1']                 for r in t5_all]), 4) if t5_all else None,
        'scored':                 len(t5_all),
        'parse_failed':           sum(1 for r in connected_results if r['task5']['status'] != 'ok'),
        'by_complexity':          _by_complexity(connected_results, 'task5',
                                                 ['library_coverage', 'setup_completeness',
                                                  'loop_correctness', 'f1']),
    },

    'task6': {
        'compilation_success_rate': round(
            sum(1 for r in t6_all if r['compilation_success']) / len(t6_all), 4
        ) if t6_all else None,
        'avg_library_f1':       round(np.mean([r['library_f1']        for r in t6_all]), 4) if t6_all else None,
        'avg_pin_valid':        round(np.mean([r['pin_valid']          for r in t6_all]), 4) if t6_all else None,
        'avg_component_correct':round(np.mean([r['component_correct']  for r in t6_all]), 4) if t6_all else None,
        'avg_logic_correctness':round(np.mean([r['logic_correctness']  for r in t6_all]), 2) if t6_all else None,
        'avg_code_quality':     round(np.mean([r['code_quality']       for r in t6_all]), 2) if t6_all else None,
        'avg_latency':          round(np.mean([r['latency']            for r in t6_all]), 3) if t6_all else None,
        'scored':               len(t6_all),
        'generation_failed':    len(t6_failed),
        'compiled':             sum(1 for r in t6_all if r['compilation_success']),
        'compile_failed':       sum(1 for r in t6_all if r['compilation_status'] == 'failed'),
        'library_missing':      sum(1 for r in t6_all if r['compilation_status'] == 'library_missing'),
        'by_complexity':        _by_complexity(connected_results, 'task6',
                                               ['compilation_success', 'library_f1',
                                                'pin_valid', 'logic_correctness', 'code_quality']),
    },

    'results': connected_results,
}

with open(SAVE_PATH, 'w') as f:
    json.dump(summary, f, indent=2)

print(f' Saved → {SAVE_PATH}')
print()
print('── Final Pipeline Summary ──')
print(f'  Task 1 — Avg F1                   : {summary["task1"]["avg_f1"]}')
print(f'  Task 2 — Avg F1                   : {summary["task2"]["avg_f1"]}')
print(f'  Task 3 — Avg F1                   : {summary["task3"]["avg_f1"]}')
print(f'  Task 4 — Avg F1                   : {summary["task4"]["avg_f1"]}')
print(f'  Task 5 — Avg F1                   : {summary["task5"]["avg_f1"]}')
print(f'  Task 6 — Compilation Success Rate : {summary["task6"]["compilation_success_rate"]}')
print(f'  Task 6 — Avg Logic Correctness    : {summary["task6"]["avg_logic_correctness"]} / 5')
print(f'  Task 6 — Avg Code Quality         : {summary["task6"]["avg_code_quality"]} / 5')
print(f'  Task 6 — Avg Library F1           : {summary["task6"]["avg_library_f1"]}')
print(f'  Task 6 — Avg Pin Valid            : {summary["task6"]["avg_pin_valid"]}')
print()
print('── Task 6 by Complexity ──')
for c, stats in summary['task6']['by_complexity'].items():
    if not stats.get('count'):
        continue
    print(f'  {c:<8} ({stats["count"]:>3}) — '
          f'Compiled:{stats.get("avg_compilation_success", 0):.0%}  '
          f'Logic:{stats.get("avg_logic_correctness", 0):.1f}/5  '
          f'Quality:{stats.get("avg_code_quality", 0):.1f}/5  '
          f'LibF1:{stats.get("avg_library_f1", 0):.2f}')

In [ ]:
# Run this quick diagnostic cell
for r in connected_results:
    if not r['task6']['compilation_success']:
        print(f"Project {r['project_id']} [{r['complexity']}]")
        print(f"  Libs used    : {r['task6']['libs_used']}")
        print(f"  Missing libs : {r['task6']['missing_libs']}")
        print(f"  Error        : {r['task6']['compile_error'][:200]}")
        print()

---
# 🏁 Task 7: End-to-End Generation
> **Measures:** Full pipeline performance across 3 architectural setups on the same projects.

### Three Setups
| Setup | Description |
|-------|-------------|
| **A — Baseline** | Single agent, no RAG — description → .ino directly |
| **B — Single Agent + RAG** | One agent with dictionary lookup → .ino |
| **C — Multi-Agent + RAG** | Full T1→T6 pipeline (already computed) |

### Comparison Metrics
- `compilation_success` — compiles with Arduino CLI (binary)
- `library_f1`          — precision/recall of #include statements
- `pin_valid`           — pin assignments correct (%)
- `logic_correctness`   — LLM judge 0–5
- `code_quality`        — LLM judge 0–5

### Results (llama3_3 — 10 projects)

| Metric | A: Baseline | B: Single+RAG | C: Multi+RAG |
|--------|-------------|---------------|--------------|
| Compilation % | 40% | 50% | 60% |
| Library F1 | 0.94 | 1.00 | 0.93 |
| Pin Valid | 0.68 | 1.00 | 0.93 |
| Logic (0-5) | 3.4 | 4.2 | 4.1 |
| Code Quality (0-5) | 3.1 | 4.0 | 4.0 |

### Key Findings
- **Multi-Agent+RAG achieves highest compilation rate (+20% over baseline)**
- **RAG dramatically improves pin validity** — 0.68 → 1.00 with dictionary lookup
- **Logic quality plateaus after Single+RAG** — marginal difference between B and C
- **Compilation failures are architecture-independent** — caused by library API
  hallucinations, not pipeline design

### Gains over Baseline
- B (Single+RAG) vs A: Compile +10%  LibF1 +0.060  Logic +0.800
- C (Multi+RAG)  vs A: Compile +20%  LibF1 -0.013  Logic +0.700

**Status:** ✅ Complete

In [ ]:
# ── CELL 1: TASK 7 — SETUP A: BASELINE (single agent, no RAG) ────────────────

SYSTEM_PROMPT_BASELINE = """You are an expert Arduino firmware engineer.

Given only a project description, generate a complete, compilable Arduino .ino sketch.

Rules:
- Output ONLY the raw .ino code — no markdown fences, no explanation, no preamble
- Include all necessary #include statements
- Define all required global variables
- Implement setup() with all component initializations
- Implement loop() with full project logic
- Use sensible default pin assignments
- Follow Arduino conventions
- Add brief inline comments
- If using Blynk library, always add these placeholders at the top:
  #define BLYNK_TEMPLATE_ID "YOUR_TEMPLATE_ID"
  #define BLYNK_TEMPLATE_NAME "YOUR_DEVICE_NAME"
  #define BLYNK_AUTH_TOKEN "YOUR_AUTH_TOKEN"
- For Arduino UNO R4 WiFi: use #include <WiFiS3.h> not WiFi.h
- For Arduino UNO R4 WiFi: use #include <Arduino_LED_Matrix.h> not arduino_led_matrix.h"""


USER_PROMPT_BASELINE_TEMPLATE = """Generate a complete Arduino .ino sketch for this project.

Project description: {description}
Board: {board}

Output ONLY the raw .ino code."""


T7_OUTPUT_DIR = Path('t7_output')
T7_OUTPUT_DIR.mkdir(exist_ok=True)


def run_baseline_agent(project: dict, client) -> dict:
    """
    Setup A — Single agent, no RAG.
    Input  : project description only
    Output : .ino code
    """
    pid   = project.get('project_id')
    board = project.get('board', 'Arduino Uno')
    fqbn  = FQBN_MAP.get(board, 'arduino:avr:uno')
    desc  = str(project.get('description', ''))

    user_prompt = USER_PROMPT_BASELINE_TEMPLATE.format(
        description = desc,
        board       = board,
    )

    raw, latency = client.generate(
        user_prompt,
        system_prompt = SYSTEM_PROMPT_BASELINE,
        max_tokens    = 2000,
    )

    code = raw.strip()
    code = re.sub(r'^```(?:cpp|arduino|ino|c\+\+)?\n?', '', code)
    code = re.sub(r'\n?```$', '', code).strip()

    if not code or len(code) < 50:
        return {
            'project_id':  pid,
            'status':      'generation_failed',
            'code':        '',
            'fqbn':        fqbn,
            'libs_used':   [],
            'sketch_path': None,
            'latency':     latency,
        }

    # Extract libs from generated #include statements
    libs_used = [
        m for m in re.findall(r'#include\s*[<"]([^>"]+)[>"]', code)
    ]
    libs_used = [l.removesuffix('.h').removesuffix('.H') for l in libs_used]

    sketch_dir  = T7_OUTPUT_DIR / 'baseline' / f'sketch_{pid}'
    sketch_dir.mkdir(parents=True, exist_ok=True)
    sketch_path = sketch_dir / f'sketch_{pid}.ino'
    sketch_path.write_text(code, encoding='utf-8')

    return {
        'project_id':  pid,
        'status':      'ok',
        'code':        code,
        'fqbn':        fqbn,
        'libs_used':   libs_used,
        'sketch_path': str(sketch_path),
        'latency':     latency,
    }


print(' run_baseline_agent defined')
print('   Setup A: single agent, no RAG, description → .ino only')

In [ ]:
# ── CELL 2: TASK 7 — SETUP B: SINGLE AGENT + RAG ─────────────────────────────

SYSTEM_PROMPT_SINGLE_RAG = """You are an expert Arduino firmware engineer.

Given a project description AND hardware dictionary context, generate a complete,
compilable Arduino .ino sketch.

Rules:
- Output ONLY the raw .ino code — no markdown fences, no explanation, no preamble
- Use ONLY the libraries listed in the dictionary context
- Use the exact pin types specified in the dictionary context
- Implement setup() with all component initializations
- Implement loop() with full project logic
- Follow Arduino conventions
- Add brief inline comments
- If using Blynk library, always add these placeholders at the top:
  #define BLYNK_TEMPLATE_ID "YOUR_TEMPLATE_ID"
  #define BLYNK_TEMPLATE_NAME "YOUR_DEVICE_NAME"
  #define BLYNK_AUTH_TOKEN "YOUR_AUTH_TOKEN"
- For Arduino UNO R4 WiFi: use #include <WiFiS3.h> not WiFi.h
- For Arduino UNO R4 WiFi: use #include <Arduino_LED_Matrix.h> not arduino_led_matrix.h"""


USER_PROMPT_SINGLE_RAG_TEMPLATE = """Generate a complete Arduino .ino sketch for this project.

Project description: {description}
Board: {board}

Hardware dictionary context:
{rag_context}

Output ONLY the raw .ino code."""


def build_rag_context(predicted_names: list, t2_details: list, t4_output: dict) -> str:
    """
    Build RAG context string from T1 components + T2 libraries + T4 pins.
    This is the dictionary lookup result passed to the single agent.
    """
    lines = []
    for r in t2_details:
        if not isinstance(r, dict):
            continue
        comp = r.get('component', '')
        lib  = r.get('library',   '')
        lines.append(f'Component: {comp}')
        if lib and normalise_lib(lib) not in NO_LIBRARY_NEEDED:
            lines.append(f'  Library : {lib}')
        else:
            lines.append(f'  Library : None (use native Arduino functions)')

    # Add pin assignments from T4
    if t4_output and t4_output.get('status') == 'ok':
        lines.append('\nPin Assignments:')
        for a in t4_output.get('assignments', []):
            comp     = a.get('component', '')
            pin_type = a.get('pin_type',  '')
            pins     = a.get('pins',      {})
            pin_str  = ', '.join(f'{k}:{v}' for k, v in pins.items())
            lines.append(f'  {comp} ({pin_type}) → {pin_str}')

    return '\n'.join(lines) if lines else 'No dictionary context available'


def run_single_rag_agent(project:    dict,
                          t2_details: list,
                          t4_output:  dict,
                          client) -> dict:
    """
    Setup B — Single agent with RAG dictionary context.
    Input  : description + T1 components + T2 libraries + T4 pins
    Output : .ino code
    """
    pid   = project.get('project_id')
    board = project.get('board', 'Arduino Uno')
    fqbn  = FQBN_MAP.get(board, 'arduino:avr:uno')
    desc  = str(project.get('description', ''))

    # Build RAG context from existing T1/T2/T4 results
    predicted_names = [r.get('component', '') for r in t2_details
                       if isinstance(r, dict)]
    rag_context = build_rag_context(predicted_names, t2_details, t4_output)

    user_prompt = USER_PROMPT_SINGLE_RAG_TEMPLATE.format(
        description = desc,
        board       = board,
        rag_context = rag_context,
    )

    if len(user_prompt) > MAX_PROMPT_CHARS:
        user_prompt = user_prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = client.generate(
        user_prompt,
        system_prompt = SYSTEM_PROMPT_SINGLE_RAG,
        max_tokens    = 2000,
    )

    code = raw.strip()
    code = re.sub(r'^```(?:cpp|arduino|ino|c\+\+)?\n?', '', code)
    code = re.sub(r'\n?```$', '', code).strip()

    if not code or len(code) < 50:
        return {
            'project_id':  pid,
            'status':      'generation_failed',
            'code':        '',
            'fqbn':        fqbn,
            'libs_used':   [],
            'sketch_path': None,
            'latency':     latency,
        }

    libs_used = [
        m.removesuffix('.h').removesuffix('.H')
        for m in re.findall(r'#include\s*[<"]([^>"]+)[>"]', code)
    ]

    sketch_dir  = T7_OUTPUT_DIR / 'single_rag' / f'sketch_{pid}'
    sketch_dir.mkdir(parents=True, exist_ok=True)
    sketch_path = sketch_dir / f'sketch_{pid}.ino'
    sketch_path.write_text(code, encoding='utf-8')

    return {
        'project_id':  pid,
        'status':      'ok',
        'code':        code,
        'fqbn':        fqbn,
        'libs_used':   libs_used,
        'sketch_path': str(sketch_path),
        'latency':     latency,
    }


print(' run_single_rag_agent defined')
print('   Setup B: single agent + RAG dictionary context → .ino')

In [ ]:
# ── CELL 3: TASK 7 — RUN ALL THREE SETUPS ────────────────────────────────────
T7_SAVE_PATH = 'task7_comparison.json'
task7_results = []

for proj in tqdm(projects[:10], desc='Task 7 — 3 setups'):
    pid   = proj['project_id']
    board = proj.get('board', 'Arduino Uno')

    # ── Get T1-T6 results from already computed connected_results ─────────────
    existing = next(
        (r for r in connected_results if r['project_id'] == pid), None
    )
    if not existing:
        tqdm.write(f'  SKIP project {pid} — not in connected_results')
        continue

    t2_details = existing['task2']['details']
    t4_output  = {
        'status':      existing['task4']['status'],
        'assignments': existing['task4']['assignments'],
        'conflicts':   existing['task4']['conflicts'],
        'feasibility': existing['task4']['feasibility'],
    }
    t3_output = {
        'status':        existing['task3']['status'],
        'tasks':         existing['task3']['tasks'],
        'conditions':    existing['task3']['conditions'],
        'thresholds':    existing['task3']['thresholds'],
        'logic_summary': existing['task3']['logic_summary'],
    }

    # ── Setup A — Baseline ────────────────────────────────────────────────────
    try:
        a_out     = run_baseline_agent(proj, active_client)
        a_compile = compile_sketch(a_out['sketch_path'], a_out['fqbn'], a_out['libs_used'])
        a_auto    = score_task6_metrics(proj, t2_details, t4_output, a_out, a_compile)
        _time.sleep(2.0)
        a_judge   = score_task6_with_judge(proj, t2_details, t4_output, t3_output,
                                            a_out, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR Setup A project {pid}: {e}')
        a_out     = {'status': 'error', 'code': '', 'fqbn': FQBN_MAP.get(board,'arduino:avr:uno'),
                     'libs_used': [], 'sketch_path': None, 'latency': 0.0}
        a_compile = {'status': 'skipped', 'success': False, 'error': str(e),
                     'missing_libs': [], 'installed': [], 'mode': 'skipped'}
        a_auto    = {'component_correct': 0, 'library_f1': 0.0,
                     'library_precision': 0.0, 'library_recall': 0.0, 'pin_valid': 0.0}
        a_judge   = {'logic_correctness': 0, 'code_quality': 0,
                     'reasoning': str(e), 'latency': 0.0}

    # ── Setup B — Single Agent + RAG ──────────────────────────────────────────
    try:
        b_out     = run_single_rag_agent(proj, t2_details, t4_output, active_client)
        b_compile = compile_sketch(b_out['sketch_path'], b_out['fqbn'], b_out['libs_used'])
        b_auto    = score_task6_metrics(proj, t2_details, t4_output, b_out, b_compile)
        _time.sleep(2.0)
        b_judge   = score_task6_with_judge(proj, t2_details, t4_output, t3_output,
                                            b_out, active_client)
    except Exception as e:
        tqdm.write(f'  ERROR Setup B project {pid}: {e}')
        b_out     = {'status': 'error', 'code': '', 'fqbn': FQBN_MAP.get(board,'arduino:avr:uno'),
                     'libs_used': [], 'sketch_path': None, 'latency': 0.0}
        b_compile = {'status': 'skipped', 'success': False, 'error': str(e),
                     'missing_libs': [], 'installed': [], 'mode': 'skipped'}
        b_auto    = {'component_correct': 0, 'library_f1': 0.0,
                     'library_precision': 0.0, 'library_recall': 0.0, 'pin_valid': 0.0}
        b_judge   = {'logic_correctness': 0, 'code_quality': 0,
                     'reasoning': str(e), 'latency': 0.0}

    # ── Setup C — Multi-Agent + RAG (already computed in T6) ─────────────────
    c_compile = {
        'success': existing['task6']['compilation_success'],
        'status':  existing['task6']['compilation_status'],
    }
    c_auto  = {
        'component_correct': existing['task6']['component_correct'],
        'library_f1':        existing['task6']['library_f1'],
        'library_precision': existing['task6']['library_precision'],
        'library_recall':    existing['task6']['library_recall'],
        'pin_valid':         existing['task6']['pin_valid'],
    }
    c_judge = {
        'logic_correctness': existing['task6']['logic_correctness'],
        'code_quality':      existing['task6']['code_quality'],
        'reasoning':         existing['task6']['reasoning'],
        'latency':           existing['task6']['judge_latency'],
    }

    task7_results.append({
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,

        'setup_a': {
            'name':               'Baseline (no RAG)',
            'status':             a_out['status'],
            'compilation_success':a_compile['success'],
            'compilation_status': a_compile['status'],
            'component_correct':  a_auto['component_correct'],
            'library_f1':         a_auto['library_f1'],
            'pin_valid':          a_auto['pin_valid'],
            'logic_correctness':  a_judge['logic_correctness'],
            'code_quality':       a_judge['code_quality'],
            'reasoning':          a_judge['reasoning'],
            'latency':            a_out['latency'],
        },

        'setup_b': {
            'name':               'Single Agent + RAG',
            'status':             b_out['status'],
            'compilation_success':b_compile['success'],
            'compilation_status': b_compile['status'],
            'component_correct':  b_auto['component_correct'],
            'library_f1':         b_auto['library_f1'],
            'pin_valid':          b_auto['pin_valid'],
            'logic_correctness':  b_judge['logic_correctness'],
            'code_quality':       b_judge['code_quality'],
            'reasoning':          b_judge['reasoning'],
            'latency':            b_out['latency'],
        },

        'setup_c': {
            'name':               'Multi-Agent + RAG',
            'status':             existing['task6']['status'],
            'compilation_success':c_compile['success'],
            'compilation_status': c_compile['status'],
            'component_correct':  c_auto['component_correct'],
            'library_f1':         c_auto['library_f1'],
            'pin_valid':          c_auto['pin_valid'],
            'logic_correctness':  c_judge['logic_correctness'],
            'code_quality':       c_judge['code_quality'],
            'reasoning':          c_judge['reasoning'],
            'latency':            existing['task6']['latency'],
        },
    })

    # Checkpoint every 5
    if len(task7_results) % 5 == 0:
        with open(T7_SAVE_PATH, 'w') as f:
            json.dump(task7_results, f, indent=2)
        tqdm.write(f'   Checkpoint — {len(task7_results)} saved')

with open(T7_SAVE_PATH, 'w') as f:
    json.dump(task7_results, f, indent=2)

print(f' Done — saved to {T7_SAVE_PATH}')
print(f'   Projects compared: {len(task7_results)}')

In [ ]:
# ── CELL 4: TASK 7 — PRINT COMPARISON ────────────────────────────────────────

def avg(results, setup_key, metric):
    vals = [r[setup_key][metric] for r in results
            if r[setup_key]['status'] == 'ok'
            and r[setup_key].get(metric) is not None]
    return round(np.mean(vals), 3) if vals else 0.0

def compile_rate(results, setup_key):
    total = [r for r in results if r[setup_key]['status'] == 'ok']
    if not total: return 0.0
    return round(sum(1 for r in total if r[setup_key]['compilation_success'])
                 / len(total), 3)

print('=' * 70)
print(' Task 7 — End-to-End Comparison')
print('=' * 70)
print(f'  Model    : {active_client.model_key}')
print(f'  Projects : {len(task7_results)}')
print()

headers = ['Metric', 'A: Baseline', 'B: Single+RAG', 'C: Multi+RAG', 'Best']
row_fmt = '  {:<22} {:<14} {:<14} {:<14} {}'

print(row_fmt.format(*headers))
print('  ' + '-' * 66)

metrics = [
    ('Compilation %',    'compilation_success'),
    ('Component Correct','component_correct'),
    ('Library F1',       'library_f1'),
    ('Pin Valid',        'pin_valid'),
    ('Logic (0-5)',      'logic_correctness'),
    ('Code Quality (0-5)','code_quality'),
]

for label, metric in metrics:
    if metric == 'compilation_success':
        a = compile_rate(task7_results, 'setup_a')
        b = compile_rate(task7_results, 'setup_b')
        c = compile_rate(task7_results, 'setup_c')
    else:
        a = avg(task7_results, 'setup_a', metric)
        b = avg(task7_results, 'setup_b', metric)
        c = avg(task7_results, 'setup_c', metric)
    best = ['A', 'B', 'C'][[a, b, c].index(max(a, b, c))]
    print(row_fmt.format(label, a, b, c, best))

print('=' * 70)

# ── Improvement over baseline ─────────────────────────────────────────────────
print('\n── Improvement over Baseline (Setup A) ──')
for label, metric in metrics:
    if metric == 'compilation_success':
        a = compile_rate(task7_results, 'setup_a')
        b = compile_rate(task7_results, 'setup_b')
        c = compile_rate(task7_results, 'setup_c')
    else:
        a = avg(task7_results, 'setup_a', metric)
        b = avg(task7_results, 'setup_b', metric)
        c = avg(task7_results, 'setup_c', metric)
    b_gain = round(b - a, 3)
    c_gain = round(c - a, 3)
    print(f'  {label:<22} B vs A: {b_gain:+.3f}   C vs A: {c_gain:+.3f}')

# ── Per project breakdown ─────────────────────────────────────────────────────
print('\n── Per Project ──')
print(f'  {"PID":<6} {"Complexity":<10} '
      f'{"A Compile":<12} {"B Compile":<12} {"C Compile":<12} '
      f'{"A Logic":<10} {"B Logic":<10} {"C Logic"}')
print('  ' + '-' * 78)
for r in task7_results:
    print(f'  {r["project_id"]:<6} {r["complexity"]:<10} '
          f'{str(r["setup_a"]["compilation_success"]):<12} '
          f'{str(r["setup_b"]["compilation_success"]):<12} '
          f'{str(r["setup_c"]["compilation_success"]):<12} '
          f'{r["setup_a"]["logic_correctness"]}/5       '
          f'{r["setup_b"]["logic_correctness"]}/5       '
          f'{r["setup_c"]["logic_correctness"]}/5')

# ── By complexity ─────────────────────────────────────────────────────────────
print('\n── By Complexity ──')
for complexity in ['Low', 'Medium', 'High']:
    subset = [r for r in task7_results if r['complexity'] == complexity]
    if not subset: continue
    print(f'\n  {complexity} ({len(subset)} projects):')
    for setup_key, label in [('setup_a','Baseline'),
                               ('setup_b','Single+RAG'),
                               ('setup_c','Multi+RAG')]:
        comp = sum(1 for r in subset if r[setup_key]['compilation_success'])
        logic = np.mean([r[setup_key]['logic_correctness'] for r in subset])
        lib   = np.mean([r[setup_key]['library_f1']        for r in subset])
        print(f'    {label:<15} Compiled:{comp}/{len(subset)}  '
              f'Logic:{logic:.1f}/5  LibF1:{lib:.2f}')

In [ ]:
# ── CELL 5: TASK 7 — SAVE SUMMARY ────────────────────────────────────────────
T7_FINAL_PATH = 'task7_final.json'

summary_t7 = {
    'model':    active_client.model_key,
    'projects': len(task7_results),

    'setup_a': {
        'name':                   'Baseline (single agent, no RAG)',
        'compilation_success_rate': compile_rate(task7_results, 'setup_a'),
        'avg_component_correct':    avg(task7_results, 'setup_a', 'component_correct'),
        'avg_library_f1':           avg(task7_results, 'setup_a', 'library_f1'),
        'avg_pin_valid':            avg(task7_results, 'setup_a', 'pin_valid'),
        'avg_logic_correctness':    avg(task7_results, 'setup_a', 'logic_correctness'),
        'avg_code_quality':         avg(task7_results, 'setup_a', 'code_quality'),
    },

    'setup_b': {
        'name':                   'Single Agent + RAG',
        'compilation_success_rate': compile_rate(task7_results, 'setup_b'),
        'avg_component_correct':    avg(task7_results, 'setup_b', 'component_correct'),
        'avg_library_f1':           avg(task7_results, 'setup_b', 'library_f1'),
        'avg_pin_valid':            avg(task7_results, 'setup_b', 'pin_valid'),
        'avg_logic_correctness':    avg(task7_results, 'setup_b', 'logic_correctness'),
        'avg_code_quality':         avg(task7_results, 'setup_b', 'code_quality'),
    },

    'setup_c': {
        'name':                   'Multi-Agent + RAG',
        'compilation_success_rate': compile_rate(task7_results, 'setup_c'),
        'avg_component_correct':    avg(task7_results, 'setup_c', 'component_correct'),
        'avg_library_f1':           avg(task7_results, 'setup_c', 'library_f1'),
        'avg_pin_valid':            avg(task7_results, 'setup_c', 'pin_valid'),
        'avg_logic_correctness':    avg(task7_results, 'setup_c', 'logic_correctness'),
        'avg_code_quality':         avg(task7_results, 'setup_c', 'code_quality'),
    },

    'gains_b_over_a': {
        'compilation': round(compile_rate(task7_results,'setup_b') -
                             compile_rate(task7_results,'setup_a'), 3),
        'library_f1':  round(avg(task7_results,'setup_b','library_f1') -
                             avg(task7_results,'setup_a','library_f1'), 3),
        'logic':       round(avg(task7_results,'setup_b','logic_correctness') -
                             avg(task7_results,'setup_a','logic_correctness'), 3),
    },

    'gains_c_over_a': {
        'compilation': round(compile_rate(task7_results,'setup_c') -
                             compile_rate(task7_results,'setup_a'), 3),
        'library_f1':  round(avg(task7_results,'setup_c','library_f1') -
                             avg(task7_results,'setup_a','library_f1'), 3),
        'logic':       round(avg(task7_results,'setup_c','logic_correctness') -
                             avg(task7_results,'setup_a','logic_correctness'), 3),
    },

    'results': task7_results,
}

with open(T7_FINAL_PATH, 'w') as f:
    json.dump(summary_t7, f, indent=2)

print(f' Saved → {T7_FINAL_PATH}')
print()
print('── Task 7 Final Summary ──')
print(f'  {"Metric":<25} {"A: Baseline":<15} {"B: Single+RAG":<15} {"C: Multi+RAG"}')
print('  ' + '-' * 65)
for key, label in [
    ('compilation_success_rate', 'Compilation %'),
    ('avg_library_f1',           'Library F1'),
    ('avg_pin_valid',            'Pin Valid'),
    ('avg_logic_correctness',    'Logic (0-5)'),
    ('avg_code_quality',         'Code Quality (0-5)'),
]:
    a = summary_t7['setup_a'][key]
    b = summary_t7['setup_b'][key]
    c = summary_t7['setup_c'][key]
    print(f'  {label:<25} {a:<15} {b:<15} {c}')
print()
print('── Gains over Baseline ──')
print(f'  B (Single+RAG) vs A : '
      f'Compile {summary_t7["gains_b_over_a"]["compilation"]:+.3f}  '
      f'LibF1 {summary_t7["gains_b_over_a"]["library_f1"]:+.3f}  '
      f'Logic {summary_t7["gains_b_over_a"]["logic"]:+.3f}')
print(f'  C (Multi+RAG)  vs A : '
      f'Compile {summary_t7["gains_c_over_a"]["compilation"]:+.3f}  '
      f'LibF1 {summary_t7["gains_c_over_a"]["library_f1"]:+.3f}  '
      f'Logic {summary_t7["gains_c_over_a"]["logic"]:+.3f}')

---
# 🤖 Multi-Model Comparison
> **Measures:** Performance of 5 LLMs across Tasks 1–6 on the same 10 projects.

### Models
| # | Model | Provider | Type | Status |
|---|-------|----------|------|--------|
| 1 | llama3_3 | NVIDIA | Open source | ✅ Complete |
| 2 | mistral | NVIDIA | Open source | ✅ Complete |
| 3 | gemma_27b | NVIDIA | Open source | ✅ Complete |
| 4 | gpt4o_mini | OpenAI | Commercial | ✅ Complete |
| 5 | claude_sonnet | Anthropic | Commercial | ✅ Complete |

> **Note:** phi_medium dropped — failed to produce structured JSON output for
> component extraction (T1 F1 = 0.000), indicating insufficient instruction-following
> capability for multi-agent hardware code generation pipelines.
> gemini dropped — API version incompatibility at time of evaluation.

### Reused Results
- llama3_3 results loaded from `pipeline_t1_t2_t3_t4_t5_t6_final.json`

### Output
- One results file per model: `llm_outputs/model_results_<model_key>.json`
- Combined comparison: `llm_outputs/model_comparison_final.json`

### Results Summary

| Model | T1 F1 | T2 F1 | T3 F1 | T4 F1 | T5 F1 | T6 Compile | T6 Logic |
|-------|-------|-------|-------|-------|-------|------------|----------|
| llama3_3 | 0.915 | — | 0.925 | 0.892 | 0.812 | 6/10 | 4.1/5 |
| mistral | 0.875 | — | — | — | 0.935 | 6/10 | 3.6/5 |
| gemma_27b | 0.839 | — | — | — | 0.810 | 4/10 | 2.8/5 |
| gpt4o_mini | 0.854 | — | — | — | 0.853 | 3/10 | 3.5/5 |
| claude | 0.915 | — | — | — | 0.862 | 7/10 | 3.5/5 |

### Key Findings
- **Claude achieved highest compilation rate** — 7/10 (70%)
- **llama3_3 achieved highest logic score** — 4.1/5
- **mistral achieved highest T5 planning score** — 0.935
- **llama3_3 and claude tied on T1** — both 0.915

**Status:** ✅ Complete

In [ ]:
# ── CELL 1: MULTI-MODEL COMPARISON — SETUP ───────────────────────────────────
from pathlib import Path

# ── Output directory ──────────────────────────────────────────────────────────
LLM_OUTPUT_DIR = Path('llm_outputs')
LLM_OUTPUT_DIR.mkdir(exist_ok=True)

print(f' llm_outputs dir: {LLM_OUTPUT_DIR.resolve()}')

In [ ]:
# ── CELL 2: MULTI-PROVIDER LLM CLIENT ────────────────────────────────────────
import os
import re
import json
import time as _time
import random
from typing import Tuple, Optional
from openai    import OpenAI
import anthropic
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path('.env'))

# ── Model registry ────────────────────────────────────────────────────────────
MODELS = {
    # NVIDIA models
    'llama3_3':   ('nvidia',    'meta/llama-3.3-70b-instruct',                  'NVIDIA_API_KEY_LLAMA33'),
    'mistral':    ('nvidia',    'mistralai/mistral-large-3-675b-instruct-2512', 'NVIDIA_API_KEY_MISTRAL'),
    'phi_medium': ('nvidia',    'microsoft/phi-3-medium-128k-instruct',         'NVIDIA_API_KEY_PHI3MEDIUM'),
    'gemma_27b':  ('nvidia',    'google/gemma-2-27b-it',                        'NVIDIA_API_KEY_GEMMA'),
    # OpenAI
    'gpt4o_mini': ('openai',    'gpt-4o-mini',                                  'OPENAI_API_KEY'),
    # Anthropic
    'claude':     ('anthropic', 'claude-sonnet-4-5',                            'CLAUDE_API_KEY'),
}

NVIDIA_BASE_URL  = 'https://integrate.api.nvidia.com/v1'
API_CALL_DELAY   = 1.0
API_TIMEOUT      = 150
MAX_RETRIES      = 5
MAX_PROMPT_CHARS = 32000


class LLMClient:
    def __init__(self, model_key: str, temperature: float = 0):
        if model_key not in MODELS:
            raise ValueError(f'Unknown model_key: {model_key!r}. Valid: {list(MODELS.keys())}')

        self.model_key   = model_key
        self.provider, self.model_name, api_key_name = MODELS[model_key]
        self.temperature = temperature
        self.call_log    = []

        api_key = os.getenv(api_key_name)
        if not api_key:
            raise ValueError(f'Missing API key — set {api_key_name} in your .env file')

        # ── Initialise provider-specific client ───────────────────────────────
        if self.provider == 'nvidia':
            self.client = OpenAI(
                base_url = NVIDIA_BASE_URL,
                api_key  = api_key
            )

        elif self.provider == 'openai':
            self.client = OpenAI(api_key=api_key)

        elif self.provider == 'anthropic':
            self.client = anthropic.Anthropic(api_key=api_key)

        elif self.provider == 'google':
            genai.configure(api_key=api_key)
            self.client = genai.GenerativeModel(self.model_name)

        print(f' LLMClient ready: {model_key} ({self.model_name}) [{self.provider}]')


    def generate(self, prompt: str, system_prompt: str,
                 max_tokens: int = 512) -> Tuple[str, float]:
        """
        Unified generate() across all providers.
        Returns (text, latency).
        """
        last_exc = None

        for attempt in range(MAX_RETRIES):
            try:
                _time.sleep(API_CALL_DELAY)
                t0 = _time.time()

                # ── NVIDIA / OpenAI ───────────────────────────────────────────
                if self.provider in ('nvidia', 'openai'):
                    messages = (
                        [{'role': 'user', 'content': system_prompt + '\n\n' + prompt}]
                        if self.model_key == 'gemma_27b' else
                        [
                            {'role': 'system', 'content': system_prompt},
                            {'role': 'user',   'content': prompt},
                        ]
                    )
                    response = self.client.chat.completions.create(
                        model       = self.model_name,
                        messages    = messages,
                        temperature = self.temperature,
                        max_tokens  = max_tokens,
                        timeout     = API_TIMEOUT,
                    )
                    text = response.choices[0].message.content.strip()

                # ── Anthropic ─────────────────────────────────────────────────
                elif self.provider == 'anthropic':
                    response = self.client.messages.create(
                        model      = self.model_name,
                        max_tokens = max_tokens,
                        system     = system_prompt,
                        messages   = [{'role': 'user', 'content': prompt}],
                    )
                    text = response.content[0].text.strip()

                # ── Google Gemini ─────────────────────────────────────────────
                elif self.provider == 'google':
                    # Gemini combines system + user in one prompt
                    full_prompt = f'{system_prompt}\n\n{prompt}'
                    response    = self.client.generate_content(
                        full_prompt,
                        generation_config=genai.types.GenerationConfig(
                            temperature = self.temperature,
                            max_output_tokens = max_tokens,
                        )
                    )
                    text = response.text.strip()

                latency = _time.time() - t0
                self.call_log.append({
                    'latency': round(latency, 3),
                    'attempt': attempt + 1,
                    'success': True,
                })
                return text, latency

            except Exception as exc:
                last_exc = exc
                if attempt < MAX_RETRIES - 1:
                    # Longer wait for rate limit errors
                    is_rate_limit = any(
                        x in str(exc)
                        for x in ['429', 'rate', 'quota', 'Resource exhausted']
                    )
                    wait = (
                        15.0 * (2 ** attempt) + random.uniform(1, 3)
                        if is_rate_limit else
                        0.5 * (attempt + 1)
                    )
                    print(f'  Attempt {attempt+1}/{MAX_RETRIES} failed: {exc} — retrying in {wait:.1f}s')
                    _time.sleep(wait)

        self.call_log.append({
            'latency': 0.0,
            'success': False,
            'error':   str(last_exc)[:100],
        })
        print(f'LLM call FAILED after {MAX_RETRIES} retries: {last_exc}')
        return '', 0.0


def call_llm_json(prompt: str, model, system_prompt: str,
                  max_tokens: int = 512) -> Tuple[any, float]:
    """
    Generic LLM call → parsed JSON + latency.
    Works across all providers via unified LLMClient.generate().
    """
    if model == 'stub':
        return {}, 0.0
    if not hasattr(model, 'generate'):
        raise TypeError('model must be "stub" or LLMClient instance')
    if len(prompt) > MAX_PROMPT_CHARS:
        prompt = prompt[:MAX_PROMPT_CHARS - 200] + '\n\n[TRUNCATED]'

    raw, latency = model.generate(
        prompt,
        system_prompt = system_prompt,
        max_tokens    = max_tokens,
    )
    if not raw:
        return {}, latency

    # Strip markdown fences
    text = raw.strip()
    text = re.sub(r'^```(?:json)?\n?', '', text)
    text = re.sub(r'\n?```$',          '', text).strip()

    # Fix unquoted Arduino analog pin constants
    text = re.sub(r':\s*(A[0-9])\b', r': "\1"', text)

    # ── 1) Full parse ─────────────────────────────────────────────────────────
    try:
        return json.loads(text), latency
    except Exception:
        pass

    # ── 2) Depth-tracking outermost JSON extractor ────────────────────────────
    def extract_outermost(s: str) -> Optional[str]:
        for start, ch in enumerate(s):
            if ch not in ('{', '['):
                continue
            close = '}' if ch == '{' else ']'
            depth, in_str, esc = 0, False, False
            for i in range(start, len(s)):
                c = s[i]
                if esc:
                    esc = False; continue
                if c == '\\' and in_str:
                    esc = True;  continue
                if c == '"':
                    in_str = not in_str; continue
                if in_str:
                    continue
                if c in ('{', '['):
                    depth += 1
                elif c in ('}', ']'):
                    depth -= 1
                    if depth == 0 and c == close:
                        return s[start:i + 1]
        return None

    fragment = extract_outermost(text)
    if fragment:
        try:
            return json.loads(fragment), latency
        except Exception:
            pass

    # ── 3) Truncation repair ──────────────────────────────────────────────────
    open_b  = text.count('{') - text.count('}')
    open_sq = text.count('[') - text.count(']')
    if 0 < open_b + open_sq <= 6:
        patched = text + (']' * max(0, open_sq)) + ('}' * max(0, open_b))
        try:
            return json.loads(patched), latency
        except Exception:
            pass

    logger.warning('Could not parse LLM response:\n%s', text[:400])
    return {}, latency


# ── API Key check ─────────────────────────────────────────────────────────────
print('\n── API Key Check ──')
for key, (provider, model_name, env_var) in MODELS.items():
    status = 'Present' if os.getenv(env_var) else ' MISSING'
    print(f'  {key:<12} {provider:<10} {env_var:<35} {status}')
print()
print('Ready — initialise with: active_client = LLMClient("gpt4o_mini")')

In [ ]:
# ── CELL 3: LOAD EXISTING RESULTS + INITIALISE ALL CLIENTS ───────────────────

# ── Load llama3_3 results (already computed) ──────────────────────────────────
LLAMA_RESULTS_PATH = Path('pipeline_t1_t2_t3_t4_t5_t6_final.json')

with open(LLAMA_RESULTS_PATH) as f:
    llama_saved = json.load(f)

llama_results = llama_saved['results']
print(f' Loaded llama3_3 results: {len(llama_results)} projects')

# ── All 7 models in order ─────────────────────────────────────────────────────
ALL_MODELS = [
    'llama3_3',    # ← already done, will load from file
    'mistral',
    'gemma_27b',
    'gpt4o_mini',
    'claude',
]

# ── Models to actually run (skip llama3_3) ────────────────────────────────────
MODELS_TO_RUN = [m for m in ALL_MODELS if m != 'llama3_3']

# ── Initialise clients for models to run ─────────────────────────────────────
clients = {}
for model_key in MODELS_TO_RUN:
    try:
        clients[model_key] = LLMClient(model_key, temperature=0)
    except Exception as e:
        print(f' Failed to init {model_key}: {e}')

print(f'\nClients ready  : {list(clients.keys())}')
print(f' Output dir     : {LLM_OUTPUT_DIR.resolve()}')
print(f' Projects       : {len(projects[:10])}')
print(f' Tasks          : T1 → T6')
print(f'\n── Run order ──')
for i, m in enumerate(ALL_MODELS, 1):
    status = 'done' if m == 'llama3_3' else 'pending'
    print(f'  {i}. {m:<15} {status}')

In [ ]:
# ── CELL 4: FULL PIPELINE FUNCTION ───────────────────────────────────────────

_EMPTY_T3 = {
    'project_id': None, 'status': 'error',
    'tasks': [], 'conditions': [], 'thresholds': [],
    'inputs': [], 'outputs': [], 'logic_summary': '', 'latency': 0.0, 'raw': {},
}
_EMPTY_T4 = {
    'project_id': None, 'status': 'error',
    'assignments': [], 'conflicts': [], 'feasibility': 'unknown',
    'spec_list': [], 'latency': 0.0,
}
_EMPTY_T5 = {
    'project_id': None, 'status': 'error',
    'libraries': [], 'global_vars': [], 'setup_logic': [], 'loop_logic': [],
    'notes': '', 'latency': 0.0,
}
_EMPTY_T6 = {
    'project_id': None, 'status': 'error',
    'code': '', 'fqbn': '', 'libs_used': [],
    'sketch_path': None, 'latency': 0.0,
}
_ZERO_T5_SCORE = {
    'library_coverage': 0.0, 'setup_completeness': 0.0,
    'loop_correctness': 0.0, 'f1': 0.0,
    'reasoning': 'error', 'latency': 0.0,
}
_ZERO_T6_AUTO = {
    'component_correct': 0, 'library_precision': 0.0,
    'library_recall': 0.0, 'library_f1': 0.0, 'pin_valid': 0.0,
}
_ZERO_T6_JUDGE = {
    'logic_correctness': 0, 'code_quality': 0,
    'reasoning': 'error', 'latency': 0.0,
}
_ZERO_COMPILE = {
    'status': 'skipped', 'success': False, 'error': '',
    'missing_libs': [], 'installed': [], 'mode': 'skipped',
}


def run_full_pipeline(proj:       dict,
                        client,
                      sketch_dir: Path = None) -> dict:
    """
    Run Tasks 1-6 for a single project with a single model.
    sketch_dir: where to save .ino files (defaults to T6_OUTPUT_DIR)
    Returns complete result dict or None on T1 failure.
    """
    pid   = proj['project_id']
    desc  = proj.get('description', '')
    board = proj.get('board', 'Arduino Uno')
    gt_libs = set(proj.get('ground_truth_libraries', []))

    # ── Task 1 ───────────────────────────────────────────────────────────────
    try:
        t1_output       = run_task1_agent(proj, client, use_rag=True)
        predicted_names = t1_output['predicted_names']
    except Exception as e:
        print(f'  ERROR T1 project {pid}: {e}')
        return None

    gt_names = get_gt_names(proj)
    if not gt_names:
        return None

    t1_scores    = match_predicted_to_gt(predicted_names, gt_names)
    t1_tp        = t1_scores['tp']
    t1_fp        = t1_scores['fp']
    t1_fn        = t1_scores['fn']
    t1_precision = t1_tp / (t1_tp + t1_fp) if (t1_tp + t1_fp) > 0 else 0
    t1_recall    = t1_tp / (t1_tp + t1_fn) if (t1_tp + t1_fn) > 0 else 0
    t1_f1        = (2 * t1_precision * t1_recall / (t1_precision + t1_recall)
                    if (t1_precision + t1_recall) > 0 else 0)

    # ── Task 2 ───────────────────────────────────────────────────────────────
    try:
        t2_details = run_task2_agent(
            predicted_names, desc, client, board=board
        )
    except Exception as e:
        t2_details = [
            {'component': n, 'library': None, 'status': 'error',
             'confidence': 'low', 'reasoning': str(e), 'latency': 0.0}
            for n in predicted_names
        ]

    retrieved_libs = {r['library'] for r in t2_details if r.get('library')}
    gt_canon       = canonicalize_set(gt_libs)
    ret_canon      = canonicalize_set(retrieved_libs)
    gt_canon       = {l for l in gt_canon  if l and l not in PLATFORM_LIBS}
    ret_canon      = {l for l in ret_canon if l and l not in PLATFORM_LIBS}

    if gt_canon:
        t2_tp        = match_with_equivalence(gt_canon, ret_canon)
        t2_precision = t2_tp / len(ret_canon) if ret_canon else 0
        t2_recall    = t2_tp / len(gt_canon)  if gt_canon  else 0
        t2_f1        = (2 * t2_precision * t2_recall / (t2_precision + t2_recall)
                        if (t2_precision + t2_recall) > 0 else 0)
    else:
        t2_tp = t2_precision = t2_recall = t2_f1 = None

    # ── Task 3 ───────────────────────────────────────────────────────────────
    try:
        t3_output = run_task3_agent(proj, predicted_names, client)
    except Exception as e:
        t3_output = {**_EMPTY_T3, 'project_id': pid}

    try:
        t3_score = score_intent_with_judge(proj, t3_output, client)
    except Exception as e:
        t3_score = {'completeness': 0.0, 'accuracy': 0.0,
                    'f1': 0.0, 'reasoning': str(e), 'latency': 0.0}

    # ── Task 4 ───────────────────────────────────────────────────────────────
    try:
        t4_output = run_task4_agent(proj, predicted_names, client, df_comp)
    except Exception as e:
        t4_output = {**_EMPTY_T4, 'project_id': pid}

    try:
        t4_score = score_task4(t4_output, board)
    except Exception as e:
        t4_score = {'pin_validity': 0.0, 'conflict_free': 0.0,
                    'coverage': 0.0, 'f1': 0.0,
                    'conflicts': [], 'details': []}

    # ── Task 5 ───────────────────────────────────────────────────────────────
    try:
        t5_output = run_task5_agent(
            proj, t3_output, t4_output, t2_details, client
        )
    except Exception as e:
        t5_output = {**_EMPTY_T5, 'project_id': pid}

    try:
        t5_score = score_task5_with_judge(
            proj, t2_details, t3_output, t4_output, t5_output, client
        )
    except Exception as e:
        t5_score = {**_ZERO_T5_SCORE, 'reasoning': str(e)}

    # ── Task 6 ───────────────────────────────────────────────────────────────
    try:
        t6_output = run_task6_agent(
            proj, t2_details, t5_output, client,
            output_dir=sketch_dir,
        )
    except Exception as e:
        t6_output = {**_EMPTY_T6, 'project_id': pid}

    try:
        compile_result = compile_sketch(
            t6_output['sketch_path'],
            t6_output['fqbn'],
            t6_output['libs_used'],
        )
    except Exception as e:
        compile_result = _ZERO_COMPILE.copy()

    try:
        t6_auto = score_task6_metrics(
            proj, t2_details, t4_output, t6_output, compile_result
        )
    except Exception as e:
        t6_auto = _ZERO_T6_AUTO.copy()

    try:
        t6_judge = score_task6_with_judge(
            proj, t2_details, t4_output, t3_output, t6_output, client
        )
    except Exception as e:
        t6_judge = {**_ZERO_T6_JUDGE, 'reasoning': str(e)}

    return {
        'project_id': pid,
        'complexity': proj.get('complexity'),
        'board':      board,

        'task1': {
            'predicted_names': predicted_names,
            'gt_names':        list(gt_names),
            'matched':         list(t1_scores['matched_gt']),
            'precision':       round(t1_precision, 4),
            'recall':          round(t1_recall,    4),
            'f1':              round(t1_f1,        4),
            'latency':         t1_output['latency'],
        },

        'task2': {
            'gt_libs':        list(gt_canon)  if gt_canon  else [],
            'retrieved_libs': list(ret_canon) if ret_canon else [],
            'tp':             t2_tp,
            'precision':      round(t2_precision, 4) if t2_precision is not None else None,
            'recall':         round(t2_recall,    4) if t2_recall    is not None else None,
            'f1':             round(t2_f1,        4) if t2_f1        is not None else None,
            'details':        t2_details,
        },

        'task3': {
            'tasks':         t3_output.get('tasks', []),
            'conditions':    t3_output.get('conditions', []),
            'thresholds':    t3_output.get('thresholds', []),
            'inputs':        t3_output.get('inputs', []),
            'outputs':       t3_output.get('outputs', []),
            'logic_summary': t3_output.get('logic_summary', ''),
            'completeness':  t3_score['completeness'],
            'accuracy':      t3_score['accuracy'],
            'f1':            t3_score['f1'],
            'reasoning':     t3_score['reasoning'],
            'status':        t3_output.get('status', 'error'),
            'latency':       t3_output.get('latency', 0.0),
            'judge_latency': t3_score['latency'],
        },

        'task4': {
            'assignments':   t4_output.get('assignments', []),
            'conflicts':     t4_score['conflicts'],
            'feasibility':   t4_output.get('feasibility', 'unknown'),
            'pin_validity':  t4_score['pin_validity'],
            'conflict_free': t4_score['conflict_free'],
            'coverage':      t4_score['coverage'],
            'f1':            t4_score['f1'],
            'details':       t4_score['details'],
            'status':        t4_output.get('status', 'error'),
            'latency':       t4_output.get('latency', 0.0),
        },

        'task5': {
            'libraries':          t5_output['libraries'],
            'global_vars':        t5_output['global_vars'],
            'setup_logic':        t5_output['setup_logic'],
            'loop_logic':         t5_output['loop_logic'],
            'notes':              t5_output.get('notes', ''),
            'library_coverage':   t5_score['library_coverage'],
            'setup_completeness': t5_score['setup_completeness'],
            'loop_correctness':   t5_score['loop_correctness'],
            'f1':                 t5_score['f1'],
            'reasoning':          t5_score['reasoning'],
            'status':             t5_output.get('status', 'error'),
            'latency':            t5_output.get('latency', 0.0),
            'judge_latency':      t5_score['latency'],
        },

        'task6': {
            'sketch_path':         t6_output.get('sketch_path', ''),
            'libs_used':           t6_output.get('libs_used', []),
            'fqbn':                t6_output.get('fqbn', ''),
            'compilation_status':  compile_result['status'],
            'compilation_success': compile_result['success'],
            'compilation_mode':    compile_result['mode'],
            'compile_error':       compile_result.get('error', ''),
            'missing_libs':        compile_result.get('missing_libs', []),
            'installed_libs':      compile_result.get('installed', []),
            'component_correct':   t6_auto['component_correct'],
            'library_precision':   t6_auto['library_precision'],
            'library_recall':      t6_auto['library_recall'],
            'library_f1':          t6_auto['library_f1'],
            'pin_valid':           t6_auto['pin_valid'],
            'logic_correctness':   t6_judge['logic_correctness'],
            'code_quality':        t6_judge['code_quality'],
            'reasoning':           t6_judge['reasoning'],
            'status':              t6_output.get('status', 'error'),
            'latency':             t6_output.get('latency', 0.0),
            'judge_latency':       t6_judge['latency'],
        },
    }


print(' run_full_pipeline defined')
print(' Accepts sketch_dir param for model-specific output folders')

In [ ]:
# ── CELL 5: RUN ALL MODELS T1→T6 ─────────────────────────────────────────────
all_model_results = {}

# ── Load llama3_3 from saved file ─────────────────────────────────────────────
all_model_results['llama3_3'] = llama_results
print(f' llama3_3 loaded — {len(llama_results)} projects')
print()

# ── Run remaining 6 models sequentially ──────────────────────────────────────
for model_key in MODELS_TO_RUN:
    client    = clients[model_key]
    results   = []
    errors    = 0
    save_path = LLM_OUTPUT_DIR / f'model_results_{model_key}.json'

    # ── Resume from checkpoint if exists ─────────────────────────────────────
    if save_path.exists():
        with open(save_path) as f:
            results = json.load(f)
        completed_ids = {r['project_id'] for r in results}
        print(f'▶ {model_key} — resuming from {len(results)} completed')
    else:
        completed_ids = set()

    print(f'\n{"=" * 55}')
    print(f'  Running : {model_key} ({client.model_name})')
    print(f'  Provider: {client.provider}')
    print(f'{"=" * 55}')

    # ── Model-specific sketch output folder ───────────────────────────────────
    model_sketch_dir = LLM_OUTPUT_DIR / model_key / 'sketches'
    model_sketch_dir.mkdir(parents=True, exist_ok=True)

    for proj in tqdm(projects[:10], desc=model_key):
        pid = proj['project_id']

        if pid in completed_ids:
            continue

        result = run_full_pipeline(proj, client, sketch_dir=model_sketch_dir)

        if result is None:
            errors += 1
            tqdm.write(f'  SKIP project {pid} — pipeline returned None')
            continue

        results.append(result)

        # ── Checkpoint every 5 projects ───────────────────────────────────────
        if len(results) % 5 == 0:
            with open(save_path, 'w') as f:
                json.dump(results, f, indent=2)
            tqdm.write(f'  Checkpoint — {len(results)} saved')

    # ── Final save ────────────────────────────────────────────────────────────
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)

    all_model_results[model_key] = results

    # ── Per model summary ─────────────────────────────────────────────────────
    t1_f1s      = [r['task1']['f1']                    for r in results]
    t6_compiled = [r for r in results if r['task6']['compilation_success']]
    t6_logic    = [r['task6']['logic_correctness']      for r in results]
    t5_f1s      = [r['task5']['f1'] for r in results   if r['task5']['status'] == 'ok']

    print(f'\n   {model_key} complete')
    print(f'     Projects   : {len(results)} | Errors: {errors}')
    print(f'     T1 Avg F1  : {np.mean(t1_f1s):.3f}')
    print(f'     T5 Avg F1  : {np.mean(t5_f1s):.3f}' if t5_f1s else '     T5 Avg F1  : N/A')
    print(f'     T6 Compiled: {len(t6_compiled)}/{len(results)}')
    print(f'     T6 Logic   : {np.mean(t6_logic):.1f}/5')
    print()

# ── Final summary across all models ──────────────────────────────────────────
print()
print('=' * 65)
print(' All Models Complete — Quick Summary')
print('=' * 65)
print(f'  {"Model":<15} {"T1 F1":<10} {"T5 F1":<10} {"T6 Compile":<12} {"T6 Logic"}')
print('  ' + '-' * 55)

for model_key in ALL_MODELS:
    r = all_model_results.get(model_key, [])
    if not r:
        print(f'  {model_key:<15} no results')
        continue
    t1  = np.mean([x['task1']['f1'] for x in r])
    t5s = [x['task5']['f1'] for x in r if x['task5']['status'] == 'ok']
    t5  = np.mean(t5s) if t5s else 0.0
    t6c = sum(1 for x in r if x['task6']['compilation_success'])
    t6l = np.mean([x['task6']['logic_correctness'] for x in r])
    print(f'  {model_key:<15} {t1:<10.3f} {t5:<10.3f} '
          f'{t6c}/{len(r):<9} {t6l:.1f}/5')

print('=' * 65)

In [ ]:
# ── RUN T6 ONLY — Re-run T6 for all non-llama models using saved T1-T5 ────────

MODELS_TO_RERUN_T6 = ['mistral', 'gemma_27b', 'gpt4o_mini', 'claude']

for model_key in MODELS_TO_RERUN_T6:
    client    = clients[model_key]
    save_path = LLM_OUTPUT_DIR / f'model_results_{model_key}.json'

    if not save_path.exists():
        print(f' No saved results for {model_key} — skipping')
        continue

    with open(save_path) as f:
        results = json.load(f)

    print(f'\n{"=" * 55}')
    print(f'  Re-running T6: {model_key} ({len(results)} projects)')
    print(f'{"=" * 55}')

    model_sketch_dir = LLM_OUTPUT_DIR / model_key / 'sketches'
    model_sketch_dir.mkdir(parents=True, exist_ok=True)

    for r in tqdm(results, desc=f'T6 {model_key}'):
        pid  = r['project_id']
        proj = next((p for p in projects if p['project_id'] == pid), None)
        if not proj:
            continue

        # ── Reconstruct T5 and T2 from saved results ──────────────────────────
        t2_details = r['task2']['details']
        t5_output  = {
            'status':      r['task5']['status'],
            'libraries':   r['task5'].get('libraries',   []),
            'global_vars': r['task5'].get('global_vars', []),
            'setup_logic': r['task5'].get('setup_logic', []),
            'loop_logic':  r['task5'].get('loop_logic',  []),
            'notes':       r['task5'].get('notes',       ''),
        }
        t3_output = {
            'status':        r['task3']['status'],
            'tasks':         r['task3'].get('tasks',         []),
            'conditions':    r['task3'].get('conditions',    []),
            'thresholds':    r['task3'].get('thresholds',    []),
            'logic_summary': r['task3'].get('logic_summary', ''),
        }
        t4_output = {
            'status':      r['task4']['status'],
            'assignments': r['task4'].get('assignments', []),
            'conflicts':   r['task4'].get('conflicts',   []),
            'feasibility': r['task4'].get('feasibility', 'unknown'),
        }

        # ── Run T6 generation ─────────────────────────────────────────────────
        try:
            t6_output = run_task6_agent(
                proj, t2_details, t5_output, client,
                output_dir=model_sketch_dir,
            )
        except Exception as e:
            tqdm.write(f'  ERROR T6 gen project {pid}: {e}')
            t6_output = {
                'project_id': pid, 'status': 'error',
                'code': '', 'fqbn': FQBN_MAP.get(proj.get('board','Arduino Uno'),'arduino:avr:uno'),
                'libs_used': [], 'sketch_path': None, 'latency': 0.0,
            }

        # ── Compile ───────────────────────────────────────────────────────────
        try:
            compile_result = compile_sketch(
                t6_output['sketch_path'],
                t6_output['fqbn'],
                t6_output['libs_used'],
            )
        except Exception as e:
            tqdm.write(f'  ERROR compile project {pid}: {e}')
            compile_result = {
                'status': 'error', 'success': False, 'error': str(e),
                'missing_libs': [], 'installed': [], 'mode': 'real',
            }

        # ── Auto metrics ──────────────────────────────────────────────────────
        try:
            t6_auto = score_task6_metrics(
                proj, t2_details, t4_output, t6_output, compile_result
            )
        except Exception as e:
            t6_auto = {
                'component_correct': 0, 'library_precision': 0.0,
                'library_recall': 0.0, 'library_f1': 0.0, 'pin_valid': 0.0,
            }

        # ── Judge ─────────────────────────────────────────────────────────────
        try:
            t6_judge = score_task6_with_judge(
                proj, t2_details, t4_output, t3_output, t6_output, client
            )
        except Exception as e:
            tqdm.write(f'  ERROR T6 judge project {pid}: {e}')
            t6_judge = {
                'logic_correctness': 0, 'code_quality': 0,
                'reasoning': str(e), 'latency': 0.0,
            }

        # ── Update saved result ───────────────────────────────────────────────
        r['task6'] = {
            'sketch_path':         t6_output.get('sketch_path', ''),
            'libs_used':           t6_output.get('libs_used', []),
            'fqbn':                t6_output.get('fqbn', ''),
            'compilation_status':  compile_result['status'],
            'compilation_success': compile_result['success'],
            'compilation_mode':    compile_result['mode'],
            'compile_error':       compile_result.get('error', ''),
            'missing_libs':        compile_result.get('missing_libs', []),
            'installed_libs':      compile_result.get('installed', []),
            'component_correct':   t6_auto['component_correct'],
            'library_precision':   t6_auto['library_precision'],
            'library_recall':      t6_auto['library_recall'],
            'library_f1':          t6_auto['library_f1'],
            'pin_valid':           t6_auto['pin_valid'],
            'logic_correctness':   t6_judge['logic_correctness'],
            'code_quality':        t6_judge['code_quality'],
            'reasoning':           t6_judge['reasoning'],
            'status':              t6_output.get('status', 'error'),
            'latency':             t6_output.get('latency', 0.0),
            'judge_latency':       t6_judge['latency'],
        }

    # ── Save updated results ──────────────────────────────────────────────────
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)

    all_model_results[model_key] = results

    # ── Summary ───────────────────────────────────────────────────────────────
    t6_all      = [r for r in results if r['task6']['status'] == 'ok']
    t6_compiled = [r for r in t6_all   if r['task6']['compilation_success']]
    t6_logic    = [r['task6']['logic_correctness'] for r in t6_all]

    print(f'\n   {model_key} T6 complete')
    print(f'     Generation ok : {len(t6_all)}/{len(results)}')
    print(f'     Compiled      : {len(t6_compiled)}/{len(t6_all)}')
    print(f'     Avg Logic     : {np.mean(t6_logic):.1f}/5' if t6_logic else '     Avg Logic     : N/A')
    print()

print(' T6 re-run complete for all models')

In [ ]:
# ── CELL 7: FIX CLAUDE T5 JUDGE ──────────────────────────────────────────────
# Claude's verbose reasoning was truncating at max_tokens=256
# Re-score T5 judge only for claude with max_tokens=512

claude_path = LLM_OUTPUT_DIR / 'model_results_claude.json'
with open(claude_path) as f:
    claude_results = json.load(f)

client = clients['claude']
print(f'── Re-scoring Claude T5 judge ({len(claude_results)} projects) ──')

for r in tqdm(claude_results, desc='Claude T5 fix'):
    pid  = r['project_id']
    proj = next((p for p in projects if p['project_id'] == pid), None)
    if not proj:
        continue

    # Only re-score if T5 failed or scored zero
    t5_needs_rescore = (
        r['task5']['status'] != 'ok' or
        r['task5']['f1'] == 0.0 or
        r['task5']['library_coverage'] == 0.0
    )
    if not t5_needs_rescore:
        continue

    t2_details = r['task2']['details']
    t3_output  = {**r['task3']}
    t4_output  = {**r['task4']}
    t5_output  = {
        'status':      r['task5']['status'],
        'libraries':   r['task5'].get('libraries',   []),
        'global_vars': r['task5'].get('global_vars', []),
        'setup_logic': r['task5'].get('setup_logic', []),
        'loop_logic':  r['task5'].get('loop_logic',  []),
        'notes':       r['task5'].get('notes',       ''),
    }

    # Skip if T5 generation itself failed
    if not any([t5_output['setup_logic'], t5_output['loop_logic']]):
        tqdm.write(f'  SKIP project {pid} — T5 output empty')
        continue

    try:
        # Re-score with higher max_tokens
        _time.sleep(2.0)
        t5_score = score_task5_with_judge(
            proj, t2_details, t3_output, t4_output, t5_output, client
        )
        r['task5']['library_coverage']   = t5_score['library_coverage']
        r['task5']['setup_completeness'] = t5_score['setup_completeness']
        r['task5']['loop_correctness']   = t5_score['loop_correctness']
        r['task5']['f1']                 = t5_score['f1']
        r['task5']['reasoning']          = t5_score['reasoning']
        r['task5']['status']             = 'ok'
        tqdm.write(f'  Project {pid} → F1: {t5_score["f1"]:.3f}')
    except Exception as e:
        tqdm.write(f'  ERROR project {pid}: {e}')

# Save updated results
with open(claude_path, 'w') as f:
    json.dump(claude_results, f, indent=2)

all_model_results['claude'] = claude_results

# Summary
t5_ok  = [r for r in claude_results if r['task5']['status'] == 'ok']
t5_f1s = [r['task5']['f1'] for r in t5_ok]
print(f'\n Claude T5 fix complete')
print(f'   T5 ok      : {len(t5_ok)}/{len(claude_results)}')
print(f'   Avg T5 F1  : {np.mean(t5_f1s):.3f}' if t5_f1s else '   Avg T5 F1  : N/A')

In [ ]:
# ── CELL 8: FIX CLAUDE T5 GENERATION (4 failed projects) ────────────────────

claude_path = LLM_OUTPUT_DIR / 'model_results_claude.json'
with open(claude_path) as f:
    claude_results = json.load(f)

client = clients['claude']

# Find projects where T5 generation failed
t5_failed = [
    r for r in claude_results
    if not any([
        r['task5'].get('setup_logic'),
        r['task5'].get('loop_logic'),
    ])
]

print(f'── Re-generating Claude T5 for {len(t5_failed)} failed projects ──')
for r in t5_failed:
    print(f'  Project {r["project_id"]} — T5 status: {r["task5"]["status"]}')

print()

for r in tqdm(t5_failed, desc='Claude T5 regen'):
    pid  = r['project_id']
    proj = next((p for p in projects if p['project_id'] == pid), None)
    if not proj:
        continue

    t2_details = r['task2']['details']
    t3_output  = {**r['task3']}
    t4_output  = {**r['task4']}

    # ── Re-generate T5 ────────────────────────────────────────────────────────
    try:
        t5_output = run_task5_agent(
            proj, t3_output, t4_output, t2_details, client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T5 gen project {pid}: {e}')
        continue

    if t5_output['status'] != 'ok':
        tqdm.write(f'  T5 still failed project {pid}: {t5_output["status"]}')
        continue

    # ── Re-score T5 judge ─────────────────────────────────────────────────────
    try:
        _time.sleep(2.0)
        t5_score = score_task5_with_judge(
            proj, t2_details, t3_output, t4_output, t5_output, client
        )
    except Exception as e:
        tqdm.write(f'  ERROR T5 judge project {pid}: {e}')
        t5_score = {
            'library_coverage': 0.0, 'setup_completeness': 0.0,
            'loop_correctness': 0.0, 'f1': 0.0,
            'reasoning': str(e), 'latency': 0.0,
        }

    # ── Update saved result ───────────────────────────────────────────────────
    r['task5']['libraries']          = t5_output['libraries']
    r['task5']['global_vars']        = t5_output['global_vars']
    r['task5']['setup_logic']        = t5_output['setup_logic']
    r['task5']['loop_logic']         = t5_output['loop_logic']
    r['task5']['notes']              = t5_output.get('notes', '')
    r['task5']['library_coverage']   = t5_score['library_coverage']
    r['task5']['setup_completeness'] = t5_score['setup_completeness']
    r['task5']['loop_correctness']   = t5_score['loop_correctness']
    r['task5']['f1']                 = t5_score['f1']
    r['task5']['reasoning']          = t5_score['reasoning']
    r['task5']['status']             = 'ok'
    tqdm.write(f'  Project {pid} → T5 F1: {t5_score["f1"]:.3f}')

    # ── Also re-run T6 for this project ──────────────────────────────────────
    model_sketch_dir = LLM_OUTPUT_DIR / 'claude' / 'sketches'
    try:
        t6_output = run_task6_agent(
            proj, t2_details, t5_output, client,
            output_dir=model_sketch_dir,
        )
        compile_result = compile_sketch(
            t6_output['sketch_path'], t6_output['fqbn'], t6_output['libs_used']
        )
        t6_auto  = score_task6_metrics(proj, t2_details, t4_output, t6_output, compile_result)
        t6_judge = score_task6_with_judge(proj, t2_details, t4_output, t3_output, t6_output, client)

        r['task6'] = {
            'sketch_path':         t6_output.get('sketch_path', ''),
            'libs_used':           t6_output.get('libs_used', []),
            'fqbn':                t6_output.get('fqbn', ''),
            'compilation_status':  compile_result['status'],
            'compilation_success': compile_result['success'],
            'compilation_mode':    compile_result['mode'],
            'compile_error':       compile_result.get('error', ''),
            'missing_libs':        compile_result.get('missing_libs', []),
            'installed_libs':      compile_result.get('installed', []),
            'component_correct':   t6_auto['component_correct'],
            'library_precision':   t6_auto['library_precision'],
            'library_recall':      t6_auto['library_recall'],
            'library_f1':          t6_auto['library_f1'],
            'pin_valid':           t6_auto['pin_valid'],
            'logic_correctness':   t6_judge['logic_correctness'],
            'code_quality':        t6_judge['code_quality'],
            'reasoning':           t6_judge['reasoning'],
            'status':              t6_output.get('status', 'error'),
            'latency':             t6_output.get('latency', 0.0),
            'judge_latency':       t6_judge['latency'],
        }
        tqdm.write(f'  Project {pid} → T6 compiled: {compile_result["success"]}')
    except Exception as e:
        tqdm.write(f'  ERROR T6 project {pid}: {e}')

# ── Save ──────────────────────────────────────────────────────────────────────
with open(claude_path, 'w') as f:
    json.dump(claude_results, f, indent=2)

all_model_results['claude'] = claude_results

# ── Final Claude summary ──────────────────────────────────────────────────────
t5_ok       = [r for r in claude_results if r['task5']['status'] == 'ok']
t5_f1s      = [r['task5']['f1'] for r in t5_ok]
t6_ok       = [r for r in claude_results if r['task6']['status'] == 'ok']
t6_compiled = [r for r in t6_ok if r['task6']['compilation_success']]
t6_logic    = [r['task6']['logic_correctness'] for r in t6_ok]

print(f'\n Claude fully fixed')
print(f'   T5 ok      : {len(t5_ok)}/{len(claude_results)}')
print(f'   T5 Avg F1  : {np.mean(t5_f1s):.3f}' if t5_f1s else '   T5 Avg F1  : N/A')
print(f'   T6 compiled: {len(t6_compiled)}/{len(t6_ok)}')
print(f'   T6 Avg Logic: {np.mean(t6_logic):.1f}/5' if t6_logic else '   T6 Avg Logic: N/A')

In [ ]:
# ── Quick diagnosis for persistently failing projects ─────────────────────────
for pid in [5, 6, 11]:
    proj = next((p for p in projects if p['project_id'] == pid), None)
    r    = next((r for r in claude_results if r['project_id'] == pid), None)
    if not proj or not r:
        continue
    print(f'Project {pid} [{proj.get("complexity")}] | {proj.get("board")}')
    print(f'  Description : {proj.get("description", "")[:100]}')
    print(f'  Components  : {r["task1"]["predicted_names"]}')
    print(f'  T3 status   : {r["task3"]["status"]}')
    print(f'  T4 status   : {r["task4"]["status"]}')
    print(f'  T5 status   : {r["task5"]["status"]}')
    print()

In [ ]:
# ── CELL 9: FIX CLAUDE T5 FOR COMPLEX PROJECTS ───────────────────────────────
# Projects 5, 6, 11 — non-standard boards + complex components
# Need higher max_tokens for T5 generation

claude_path = LLM_OUTPUT_DIR / 'model_results_claude.json'
with open(claude_path) as f:
    claude_results = json.load(f)

client   = clients['claude']
FAIL_IDS = [5, 6, 11]

print(f'── Re-generating T5 for projects {FAIL_IDS} with max_tokens=2500 ──')

# Temporarily patch T5_MAX_TOKENS
original_t5_tokens = T5_MAX_TOKENS

for pid in FAIL_IDS:
    r    = next((r for r in claude_results if r['project_id'] == pid), None)
    proj = next((p for p in projects      if p['project_id'] == pid), None)
    if not r or not proj:
        continue

    print(f'\n  Project {pid} [{proj.get("complexity")}] | {proj.get("board")}')

    t2_details = r['task2']['details']
    t3_output  = {**r['task3']}
    t4_output  = {**r['task4']}

    # ── Re-generate with higher tokens ───────────────────────────────────────
    try:
        # Build T5 prompt manually with higher max_tokens
        board         = proj.get('board', 'Arduino Uno')
        t3_ok         = t3_output.get('status') == 'ok'
        tasks         = t3_output.get('tasks',         []) if t3_ok else []
        conditions    = t3_output.get('conditions',    []) if t3_ok else []
        thresholds    = t3_output.get('thresholds',    []) if t3_ok else []
        inputs        = t3_output.get('inputs',        []) if t3_ok else []
        outputs       = t3_output.get('outputs',       []) if t3_ok else []
        logic_summary = t3_output.get('logic_summary', '') if t3_ok else ''

        pin_assignments_str = format_pin_assignments(t4_output)
        libraries_str       = format_libraries_from_t2(t2_details)

        user_prompt = USER_PROMPT_MODULE3C_TEMPLATE.format(
            description     = str(proj.get('description', '')),
            pin_assignments = pin_assignments_str,
            tasks           = json.dumps(tasks),
            conditions      = json.dumps(conditions),
            thresholds      = json.dumps(thresholds),
            inputs          = json.dumps(inputs),
            outputs         = json.dumps(outputs),
            logic_summary   = logic_summary,
            libraries       = libraries_str,
            board           = board,
        )

        result, latency = call_llm_json(
            user_prompt,
            model         = client,
            system_prompt = SYSTEM_PROMPT_MODULE3C,
            max_tokens    = 2500,              # ← bumped from 1500
        )

        if not isinstance(result, dict) or not any(result.get(k) for k in
                ['libraries', 'global_vars', 'setup_logic', 'loop_logic']):
            print(f'    Still parse_failed — skipping')
            continue

        # Fill missing keys
        for key in ['libraries', 'global_vars', 'setup_logic', 'loop_logic']:
            if key not in result:
                result[key] = []

        t5_output = {
            'status':      'ok',
            'libraries':   result.get('libraries',   []),
            'global_vars': result.get('global_vars', []),
            'setup_logic': result.get('setup_logic', []),
            'loop_logic':  result.get('loop_logic',  []),
            'notes':       result.get('notes',       ''),
            'latency':     latency,
        }

        # ── Score T5 ─────────────────────────────────────────────────────────
        _time.sleep(2.0)
        t5_score = score_task5_with_judge(
            proj, t2_details, t3_output, t4_output, t5_output, client
        )

        r['task5'].update({
            'libraries':          t5_output['libraries'],
            'global_vars':        t5_output['global_vars'],
            'setup_logic':        t5_output['setup_logic'],
            'loop_logic':         t5_output['loop_logic'],
            'notes':              t5_output['notes'],
            'library_coverage':   t5_score['library_coverage'],
            'setup_completeness': t5_score['setup_completeness'],
            'loop_correctness':   t5_score['loop_correctness'],
            'f1':                 t5_score['f1'],
            'reasoning':          t5_score['reasoning'],
            'status':             'ok',
        })
        print(f'     T5 F1: {t5_score["f1"]:.3f}')

        # ── Re-run T6 ─────────────────────────────────────────────────────────
        model_sketch_dir = LLM_OUTPUT_DIR / 'claude' / 'sketches'
        t6_output      = run_task6_agent(proj, t2_details, t5_output, client,
                                          output_dir=model_sketch_dir)
        compile_result = compile_sketch(t6_output['sketch_path'],
                                         t6_output['fqbn'], t6_output['libs_used'])
        t6_auto        = score_task6_metrics(proj, t2_details, t4_output,
                                              t6_output, compile_result)
        t6_judge       = score_task6_with_judge(proj, t2_details, t4_output,
                                                 t3_output, t6_output, client)
        r['task6'].update({
            'sketch_path':         t6_output.get('sketch_path', ''),
            'libs_used':           t6_output.get('libs_used', []),
            'fqbn':                t6_output.get('fqbn', ''),
            'compilation_status':  compile_result['status'],
            'compilation_success': compile_result['success'],
            'compilation_mode':    compile_result['mode'],
            'compile_error':       compile_result.get('error', ''),
            'missing_libs':        compile_result.get('missing_libs', []),
            'installed_libs':      compile_result.get('installed', []),
            'component_correct':   t6_auto['component_correct'],
            'library_precision':   t6_auto['library_precision'],
            'library_recall':      t6_auto['library_recall'],
            'library_f1':          t6_auto['library_f1'],
            'pin_valid':           t6_auto['pin_valid'],
            'logic_correctness':   t6_judge['logic_correctness'],
            'code_quality':        t6_judge['code_quality'],
            'reasoning':           t6_judge['reasoning'],
            'status':              t6_output.get('status', 'error'),
            'latency':             t6_output.get('latency', 0.0),
            'judge_latency':       t6_judge['latency'],
        })
        print(f'     T6 compiled: {compile_result["success"]}')

    except Exception as e:
        print(f'    ERROR: {e}')

# ── Save ──────────────────────────────────────────────────────────────────────
with open(claude_path, 'w') as f:
    json.dump(claude_results, f, indent=2)

all_model_results['claude'] = claude_results

# ── Final summary ─────────────────────────────────────────────────────────────
t5_ok       = [r for r in claude_results if r['task5']['status'] == 'ok']
t5_f1s      = [r['task5']['f1'] for r in t5_ok]
t6_ok       = [r for r in claude_results if r['task6']['status'] == 'ok']
t6_compiled = [r for r in t6_ok if r['task6']['compilation_success']]
t6_logic    = [r['task6']['logic_correctness'] for r in t6_ok]

print(f'\n Claude final state')
print(f'   T5 ok       : {len(t5_ok)}/10')
print(f'   T5 Avg F1   : {np.mean(t5_f1s):.3f}' if t5_f1s else '   T5 Avg F1   : N/A')
print(f'   T6 compiled : {len(t6_compiled)}/{len(t6_ok)}')
print(f'   T6 Avg Logic: {np.mean(t6_logic):.1f}/5' if t6_logic else '   T6 Avg Logic: N/A')

In [ ]:
# ── CELL 10: FULL MODEL COMPARISON ───────────────────────────────────────────
import numpy as np

print('=' * 75)
print(' Multi-Model Comparison — Tasks 1 to 6')
print('=' * 75)
print(f'  {"Model":<15} {"T1 F1":<10} {"T2 F1":<10} {"T3 F1":<10} {"T4 F1":<10} {"T5 F1":<10} {"T6 Compile":<12} {"T6 Logic"}')
print('  ' + '-' * 70)

model_summary = {}

for model_key in ALL_MODELS:
    results = all_model_results.get(model_key, [])
    if not results:
        print(f'  {model_key:<15} no results')
        continue

    # ── T1 ────────────────────────────────────────────────────────────────────
    t1_f1 = np.mean([r['task1']['f1'] for r in results])

    # ── T2 ────────────────────────────────────────────────────────────────────
    t2_all = [r['task2']['f1'] for r in results if r['task2']['f1'] is not None]
    t2_f1  = np.mean(t2_all) if t2_all else 0.0

    # ── T3 ────────────────────────────────────────────────────────────────────
    t3_all = [r['task3']['f1'] for r in results if r['task3']['status'] == 'ok']
    t3_f1  = np.mean(t3_all) if t3_all else 0.0

    # ── T4 ────────────────────────────────────────────────────────────────────
    t4_all = [r['task4']['f1'] for r in results if r['task4']['status'] == 'ok']
    t4_f1  = np.mean(t4_all) if t4_all else 0.0

    # ── T5 ────────────────────────────────────────────────────────────────────
    t5_all = [r['task5']['f1'] for r in results if r['task5']['status'] == 'ok']
    t5_f1  = np.mean(t5_all) if t5_all else 0.0

    # ── T6 ────────────────────────────────────────────────────────────────────
    t6_ok       = [r for r in results if r['task6']['status'] == 'ok']
    t6_compiled = sum(1 for r in t6_ok if r['task6']['compilation_success'])
    t6_logic    = [r['task6']['logic_correctness'] for r in t6_ok]
    t6_quality  = [r['task6']['code_quality']      for r in t6_ok]
    t6_lib_f1   = [r['task6']['library_f1']        for r in t6_ok]
    t6_pin      = [r['task6']['pin_valid']          for r in t6_ok]

    compile_rate = t6_compiled / len(t6_ok) if t6_ok else 0.0
    avg_logic    = np.mean(t6_logic)   if t6_logic   else 0.0
    avg_quality  = np.mean(t6_quality) if t6_quality else 0.0
    avg_lib_f1   = np.mean(t6_lib_f1)  if t6_lib_f1  else 0.0
    avg_pin      = np.mean(t6_pin)     if t6_pin      else 0.0

    model_summary[model_key] = {
        't1_f1':        round(t1_f1,       3),
        't2_f1':        round(t2_f1,       3),
        't3_f1':        round(t3_f1,       3),
        't4_f1':        round(t4_f1,       3),
        't5_f1':        round(t5_f1,       3),
        'compile_rate': round(compile_rate,3),
        'compiled':     t6_compiled,
        't6_total':     len(t6_ok),
        'avg_logic':    round(avg_logic,   2),
        'avg_quality':  round(avg_quality, 2),
        'avg_lib_f1':   round(avg_lib_f1,  3),
        'avg_pin':      round(avg_pin,     3),
    }

    print(f'  {model_key:<15} '
          f'{t1_f1:<10.3f} '
          f'{t2_f1:<10.3f} '
          f'{t3_f1:<10.3f} '
          f'{t4_f1:<10.3f} '
          f'{t5_f1:<10.3f} '
          f'{t6_compiled}/{len(t6_ok):<9} '
          f'{avg_logic:.1f}/5')

print('=' * 75)

# ── T6 detailed breakdown ─────────────────────────────────────────────────────
print('\n── Task 6 Detailed Breakdown ──')
print(f'  {"Model":<15} {"Compile%":<12} {"Lib F1":<10} {"Pin Valid":<12} {"Logic":<10} {"Quality"}')
print('  ' + '-' * 60)
for model_key, s in model_summary.items():
    print(f'  {model_key:<15} '
          f'{s["compile_rate"]*100:<12.0f} '
          f'{s["avg_lib_f1"]:<10.3f} '
          f'{s["avg_pin"]:<12.3f} '
          f'{s["avg_logic"]:<10.1f} '
          f'{s["avg_quality"]:.1f}')

# ── Winner per metric ─────────────────────────────────────────────────────────
print('\n── Winner per Metric ──')
metrics = [
    ('T1 Component Extraction', 't1_f1'),
    ('T2 Library Retrieval',    't2_f1'),
    ('T3 Intent Understanding', 't3_f1'),
    ('T4 Pin Mapping',          't4_f1'),
    ('T5 Code Planning',        't5_f1'),
    ('T6 Compilation Rate',     'compile_rate'),
    ('T6 Logic Correctness',    'avg_logic'),
    ('T6 Code Quality',         'avg_quality'),
    ('T6 Library F1',           'avg_lib_f1'),
    ('T6 Pin Validity',         'avg_pin'),
]

for label, key in metrics:
    scores = {m: s[key] for m, s in model_summary.items()}
    winner = max(scores, key=scores.get)
    best   = scores[winner]
    print(f'  {label:<30} → {winner:<15} ({best:.3f})')

print('=' * 75)

In [ ]:
# ── CELL 11: SAVE FINAL COMPARISON ───────────────────────────────────────────
COMPARISON_PATH = LLM_OUTPUT_DIR / 'model_comparison_final.json'

final_comparison = {
    'models':   ALL_MODELS,
    'projects': 10,
    'summary':  model_summary,
    'results':  {
        model_key: all_model_results.get(model_key, [])
        for model_key in ALL_MODELS
    },
}

with open(COMPARISON_PATH, 'w') as f:
    json.dump(final_comparison, f, indent=2)

print(f'✅ Saved → {COMPARISON_PATH}')
print()
print('── Files saved ──')
for model_key in ALL_MODELS:
    path = LLM_OUTPUT_DIR / f'model_results_{model_key}.json'
    exists = '✅' if path.exists() else '❌'
    print(f'  {exists} {path}')
print(f'  ✅ {COMPARISON_PATH}')

In [ ]:
import shutil

# Copy llama3_3 results to llm_outputs folder
src = Path('pipeline_t1_t2_t3_t4_t5_t6_final.json')
dst = LLM_OUTPUT_DIR / 'model_results_llama3_3.json'

with open(src) as f:
    llama_saved = json.load(f)

# Save just the results list (same format as other models)
with open(dst, 'w') as f:
    json.dump(llama_saved['results'], f, indent=2)

print(f'✅ Copied llama3_3 results → {dst}')

In [ ]:
# ── CELL 12: RANKINGS ─────────────────────────────────────────────────────────

print('=' * 65)
print(' Final Model Rankings')
print('=' * 65)

# ── Overall score — weighted average of all metrics ──────────────────────────
# Weights reflect importance in hardware code generation
WEIGHTS = {
    't1_f1':        0.10,   # component extraction
    't2_f1':        0.10,   # library retrieval
    't3_f1':        0.10,   # intent understanding
    't4_f1':        0.10,   # pin mapping
    't5_f1':        0.15,   # code planning
    'compile_rate': 0.25,   # compilation — most objective metric
    'avg_logic':    0.10,   # logic correctness (normalised /5)
    'avg_quality':  0.10,   # code quality (normalised /5)
}

overall_scores = {}
for model_key, s in model_summary.items():
    score = (
        s['t1_f1']        * WEIGHTS['t1_f1']        +
        s['t2_f1']        * WEIGHTS['t2_f1']        +
        s['t3_f1']        * WEIGHTS['t3_f1']        +
        s['t4_f1']        * WEIGHTS['t4_f1']        +
        s['t5_f1']        * WEIGHTS['t5_f1']        +
        s['compile_rate'] * WEIGHTS['compile_rate'] +
        (s['avg_logic']  / 5) * WEIGHTS['avg_logic']   +
        (s['avg_quality']/ 5) * WEIGHTS['avg_quality']
    )
    overall_scores[model_key] = round(score, 4)

# Sort by overall score
ranked = sorted(overall_scores.items(), key=lambda x: x[1], reverse=True)

print(f'\n  {"Rank":<6} {"Model":<15} {"Overall Score":<15} {"Compile%":<12} {"Logic":<10} {"T5 F1"}')
print('  ' + '-' * 60)
for rank, (model_key, score) in enumerate(ranked, 1):
    s = model_summary[model_key]
    medal = ['🥇', '🥈', '🥉', '  ', '  '][rank - 1]
    print(f'  {medal} {rank:<4} {model_key:<15} {score:<15.4f} '
          f'{s["compile_rate"]*100:<12.0f} '
          f'{s["avg_logic"]:<10.1f} '
          f'{s["t5_f1"]:.3f}')

print()
print(f'  🏆 Overall winner: {ranked[0][0]} (score: {ranked[0][1]:.4f})')
print()

# ── Per category winners ──────────────────────────────────────────────────────
print('── Category Winners ──')
categories = [
    ('Best Component Extraction (T1)', 't1_f1'),
    ('Best Library Retrieval (T2)',     't2_f1'),
    ('Best Intent Understanding (T3)',  't3_f1'),
    ('Best Pin Mapping (T4)',           't4_f1'),
    ('Best Code Planning (T5)',         't5_f1'),
    ('Best Compilation Rate (T6)',      'compile_rate'),
    ('Best Logic Correctness (T6)',     'avg_logic'),
    ('Best Code Quality (T6)',          'avg_quality'),
]

for label, key in categories:
    scores = {m: model_summary[m][key] for m in ALL_MODELS}
    winner = max(scores, key=scores.get)
    print(f'  {label:<35} → {winner} ({scores[winner]:.3f})')

print('=' * 65)